# 🚐 ระบบจัดสรรรถตู้ตามคำขอจอง (Van Allocation, MILP) — v6

โมเดลตัดสินใจแบบ Mixed-Integer Linear Programming (pulp) สำหรับ ICT / SCI / SEEN
เวอร์ชัน v6: นำเข้าข้อมูลจริงของคณะวิทยาศาสตร์ (SCI) จาก sci_driversN.csv และ sci_bookingN.csv


## สิ่งที่เปลี่ยนจาก v4 → v5
1. **ทริปหลายวัน**: คำนวณใหม่เป็นรอบ 24 ชม. นับจากเวลาเริ่มทริป (ทำงาน 14 ชม. + พัก 10 ชม.) และเชื่อม `check_multi_day_trip` (จุดตัด 08:00 น.) เข้า solver จริง เลิกใช้ `daily_duty_hours = workload_hours`
2. **งานนอกปฏิทิน (walk-in / ธุระด่วนในคณะ)**: ข้อมูลคนขับ 70% ไม่มี booking รองรับ จึงสร้าง *busy blocks* คงที่ (walk-in + ช่วงเวลาจองที่อนุมัติแล้ว) ให้ solver เคารพเสมอ
3. **คำขอด่วนข้ามหน่วยงาน**: ไม่ผ่านระบบ — ผู้ขอโทรหาผู้ดูแลรถตู้ ถ้าอนุมัติจะบันทึกเป็น block คงที่ (`add_manager_approved_block`)
4. **เพิ่ม Mathematical Formulations + Logic Breakdown** (markdown ด้านล่าง) และเอกสาร Word แยกต่างหาก
5. **EDA และ Data Preparation ยังอยู่ครบ** (เพิ่ม EDA งาน booking vs walk-in)
6. **โหมดเสริมที่ปิดไว้ (comment)**: urgent priority, ETA เวลาเดินทาง, ลำดับผู้ให้ยืม — ดูเซลล์ "โหมดเสริม" ท้ายส่วน test
7. Test cases ปรับเป็น 69 ข้อ (ตัดกรณีที่ผูกกับโหมดเสริมออกไปไว้ในเซลล์คอมเมนต์)

In [1]:
# นำเข้าไลบรารีที่จำเป็น
import os  # ใช้จัดการเส้นทางของไฟล์ CSV
import pandas as pd  # ใช้จัดการข้อมูลแบบตาราง DataFrame
import numpy as np  # ใช้คำนวณและสุ่มค่าทางคณิตศาสตร์
import matplotlib.pyplot as plt  # ใช้วาดกราฟ EDA
import warnings  # ใช้จัดการข้อความเตือน
import pulp
from datetime import timedelta
from itertools import combinations

warnings.filterwarnings('ignore', category=FutureWarning)  # ซ่อนเฉพาะคำเตือน FutureWarning
pd.set_option('display.max_columns', 100)  # กำหนดให้แสดงคอลัมน์ได้เพียงพอสำหรับ EDA
RANDOM_SEED = 42  # กำหนดเมล็ดสุ่มให้ผลการสาธิตทำซ้ำได้
np.random.seed(RANDOM_SEED)  # เริ่มต้นตัวสุ่มด้วยเมล็ดที่กำหนด
print('✅ Libraries imported successfully!')  # ยืนยันว่าการนำเข้าไลบรารีสำเร็จ

✅ Libraries imported successfully!


In [2]:
# การดึงข้อมูลจากโฟลเดอร์ raw_data (v6: รองรับ sci_driversN และ sci_bookingN)
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# กำหนดให้ใช้เฉพาะข้อมูลใน raw_data เท่านั้น
DATA_DIR = BASE_DIR / "raw_data"
search_dirs = [DATA_DIR]

file_candidates = {
    'ict_drivers': ['ict_drivers.csv', '02-ict_drivers.csv'],
    'sci_drivers': ['sci_driversN.csv', 'sci_drivers.csv', '04-sci_drivers.csv'],
    'seen_drivers': ['seen_drivers.csv', '06-seen_drivers.csv'],
    'ict_booking': ['ict_booking.csv', '01-ict_booking.csv'],
    'sci_booking': ['sci_bookingN.csv', 'sci_booking.csv', '03-sci_booking.csv'],
    'seen_booking': ['seen_booking.csv', '05-seen_booking.csv'],
}

resolved = {}
for key, names in file_candidates.items():
    found = next((d / name for d in search_dirs for name in names if (d / name).exists()), None)
    resolved[key] = found

if all(v is None for v in resolved.values()):
    raise FileNotFoundError('ไม่พบไฟล์ข้อมูลเลยใน raw_data')
missing_files = [k for k, v in resolved.items() if v is None]
if missing_files:
    print('⚠️ ไม่พบไฟล์:', ', '.join(missing_files), '→ ข้ามคณะนั้นในส่วนข้อมูลจริง (test case ยังใช้ข้อมูลจำลองครบ 3 คณะ)')

def read_input_csv(p):
    if p is None:
        return pd.DataFrame()
    df = pd.read_csv(p)
    # หากแถวแรกเป็นชื่อคอลัมน์ภาษาอังกฤษ (มี header 2 ชั้น ไทย+อังกฤษ) ให้ใช้ header=1
    if not df.empty and ('driver_log_id' in df.iloc[0].values or 'booking_id' in df.iloc[0].values):
        df = pd.read_csv(p, header=1)
    # ตัดแถวว่างท้ายไฟล์ที่ติดมาจาก Excel ออก
    df = df.dropna(how='all').reset_index(drop=True)
    # ปรับมาตรฐานสถานะการจอง (sci_bookingN ใช้ approve / delete)
    if 'booking_status_raw' in df.columns:
        df['booking_status_raw'] = df['booking_status_raw'].replace({'approve': 'approved', 'delete': 'cancel'})
    # ป้องกันรหัส driver_log_id หรือ booking_id ชนกันข้ามคณะ
    if 'driver_log_id' in df.columns and 'source_faculty_code' in df.columns:
        mask_sci = (df['source_faculty_code'] == 'SCI') & (~df['driver_log_id'].astype(str).str.startswith('SCI_'))
        df.loc[mask_sci, 'driver_log_id'] = 'SCI_' + df.loc[mask_sci, 'driver_log_id'].astype(str)
    if 'booking_id' in df.columns and 'source_faculty_code' in df.columns:
        mask_b_sci = (df['source_faculty_code'] == 'SCI') & (~df['booking_id'].astype(str).str.startswith('SCI_'))
        df.loc[mask_b_sci, 'booking_id'] = 'SCI_' + df.loc[mask_b_sci, 'booking_id'].astype(str)
    return df

frames = {key: read_input_csv(path) for key, path in resolved.items()}
ict_drivers, sci_drivers, seen_drivers = frames['ict_drivers'], frames['sci_drivers'], frames['seen_drivers']
ict_booking, sci_booking, seen_booking = frames['ict_booking'], frames['sci_booking'], frames['seen_booking']
df_drivers_raw = pd.concat([f for f in (seen_drivers, sci_drivers, ict_drivers) if not f.empty], ignore_index=True)
df_bookings_raw = pd.concat([f for f in (seen_booking, sci_booking, ict_booking) if not f.empty], ignore_index=True)
print('อ่านไฟล์ข้อมูลสำเร็จ:', ', '.join(f"{k}={v.name}" for k, v in resolved.items() if v is not None))
print(f"จำนวนข้อมูลประวัติคนขับทั้งหมด: {len(df_drivers_raw):,} รายการ")
print(f"จำนวนข้อมูลการจองทั้งหมด: {len(df_bookings_raw):,} รายการ")


อ่านไฟล์ข้อมูลสำเร็จ: ict_drivers=ict_drivers.csv, sci_drivers=sci_driversN.csv, seen_drivers=seen_drivers.csv, ict_booking=ict_booking.csv, sci_booking=sci_bookingN.csv, seen_booking=seen_booking.csv
จำนวนข้อมูลประวัติคนขับทั้งหมด: 2,798 รายการ
จำนวนข้อมูลการจองทั้งหมด: 650 รายการ


In [3]:
import pandas as pd

# สร้าง DataFrame สรุปจำนวนแถวของแต่ละตารางของทั้ง 3 คณะ
row_counts = pd.DataFrame({
    'Faculty': ['ICT', 'SCI', 'SEEN'],
    'Booking Rows': [len(ict_booking), len(sci_booking), len(seen_booking)],
    'Driver Rows': [len(ict_drivers), len(sci_drivers), len(seen_drivers)]
})

print("📊 จำนวนแถวของข้อมูลแต่ละตารางแยกตามคณะ:")
display(row_counts)


📊 จำนวนแถวของข้อมูลแต่ละตารางแยกตามคณะ:


  Faculty  Booking Rows  Driver Rows
0     ICT           247          598
1     SCI           156         1602
2    SEEN           247          598

## Exploratory Data Analysis

In [4]:
# การสำรวจข้อมูล (Exploratory Data Analysis - EDA) - ข้อมูลคนขับ
# มาดูหน้าตาของข้อมูลคนขับกันก่อน 5 บรรทัดแรก
display(df_drivers_raw.head())

# ดูข้อมูลสรุป (จำนวนแถว, ชนิดข้อมูล, ค่าว่าง)
print("\n--- ข้อมูลสรุปของตารางคนขับ ---")
df_drivers_raw.info()


--- ข้อมูลสรุปของตารางคนขับ ---
<class 'pandas.DataFrame'>
RangeIndex: 2798 entries, 0 to 2797
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   driver_log_id             2798 non-null   str   
 1   source_faculty_code       2798 non-null   str   
 2   driver_name               2798 non-null   str   
 3   vehicle_plate             2798 non-null   str   
 4   actual_start_date         2798 non-null   str   
 5   actual_start_time         2796 non-null   str   
 6   actual_end_date           2798 non-null   str   
 7   actual_end_time           2792 non-null   str   
 8   vehicle_user_name         2793 non-null   str   
 9   destination_raw           2797 non-null   str   
 10  destination_province_raw  2796 non-null   str   
 11  odometer_start            2798 non-null   int64 
 12  odometer_end              2798 non-null   int64 
 13  recorded_distance_km      2748 non-null   object
dtypes:

  driver_log_id source_faculty_code      driver_name vehicle_plate actual_start_date actual_start_time actual_end_date actual_end_time      vehicle_user_name               destination_raw destination_province_raw  odometer_start  odometer_end recorded_distance_km
0  SEEN_DL_0001                SEEN  SEEN_DRIVER_OLD   SIM-SEEN-01        2024-09-30          07:00:00      2024-09-30        08:06:00         ภารกิจเร่งด่วน  มหาวิทยาลัยในจังหวัดเชียงราย                 เชียงราย          145000        145016                   16
1  SEEN_DL_0002                SEEN  SEEN_DRIVER_OLD   SIM-SEEN-01        2024-10-01          19:37:00      2024-10-01        21:25:00           งานธุรการคณะ      อาคารสำนักงานมหาวิทยาลัย                เชียงใหม่          145016        145039                   23
2  SEEN_DL_0003                SEEN  SEEN_DRIVER_OLD   SIM-SEEN-01        2024-10-03          04:35:00      2024-10-03        06:58:00           งานธุรการคณะ     มหาวิทยาลัยในจังหวัดลำพูน                    ล

In [5]:
# การสำรวจข้อมูล (EDA) - ข้อมูลการจอง
# มาดูหน้าตาของข้อมูลการจอง 5 บรรทัดแรก
display(df_bookings_raw.head())

# ดูข้อมูลสรุปของตารางการจอง
print("\n--- ข้อมูลสรุปของตารางการจอง ---")
df_bookings_raw.info()


--- ข้อมูลสรุปของตารางการจอง ---
<class 'pandas.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   booking_id                     650 non-null    str    
 1   source_faculty_code            650 non-null    str    
 2   request_date                   650 non-null    str    
 3   requester_person_id            494 non-null    str    
 4   requester_department           494 non-null    str    
 5   requester_faculty_raw          650 non-null    str    
 6   trip_scope_raw                 650 non-null    str    
 7   purpose_raw                    649 non-null    str    
 8   passenger_count                643 non-null    float64
 9   planned_start_date             650 non-null    str    
 10  planned_start_time             650 non-null    str    
 11  planned_end_date               649 non-null    str    
 12  planned_end_time           

     booking_id source_faculty_code request_date requester_person_id requester_department    requester_faculty_raw trip_scope_raw          purpose_raw  passenger_count planned_start_date planned_start_time planned_end_date planned_end_time               destination_raw destination_province_raw pickup_location_raw dropoff_location_raw receiver_person_id coordinator_person_id  funding_source_1_flag  funding_source_2_flag  funding_source_3_flag  funding_source_4_flag funding_source_4_payment_date booking_status_raw requester_name receiver_name coordinator_name
0  SEEN_BK_0001                SEEN   2025-01-14       REQ_SEEN_0011             งานวิจัย  หน่วยงานกลางมหาวิทยาลัย  ภายนอกจังหวัด      ติดต่อประสานงาน             10.0         2025-02-01           13:17:00       2025-02-01         19:12:00          มหาวิทยาลัยในจ.ลำพูน                    ลำพูน             หน้าคณะ     อาคารสำนักงานคณะ      RCV_SEEN_0072         COO_SEEN_0054                    0.0                    0.0              

In [6]:
# EDA - ตรวจคุณภาพข้อมูลดิบก่อนทำความสะอาด
for table_name, table_df, id_column in [
    ('log คนขับ', df_drivers_raw, 'driver_log_id'),  # กำหนดตาราง log คนขับและรหัสหลัก
    ('คำขอจอง', df_bookings_raw, 'booking_id')  # กำหนดตารางคำขอจองและรหัสหลัก
]:  # วนตรวจคุณภาพข้อมูลทั้ง 2 ตาราง

    print(f'\n===== EDA: {table_name} =====')  # แสดงชื่อรายงานของตารางปัจจุบัน
    print(f'ขนาด: {table_df.shape[0]:,} แถว × {table_df.shape[1]:,} คอลัมน์')  # แสดงจำนวนแถวและคอลัมน์
    print(f'แถวซ้ำทั้งแถว: {table_df.duplicated().sum():,} แถว')  # นับแถวที่ซ้ำทุกคอลัมน์
    print(f'รหัส {id_column} ซ้ำ: {table_df[id_column].duplicated().sum():,} แถว')  # นับรหัสหลักที่ซ้ำ

    missing_report = table_df.isna().sum().to_frame('missing_count')  # นับค่าว่างของแต่ละคอลัมน์
    missing_report['missing_pct'] = missing_report['missing_count'] / len(table_df) * 100  # คำนวณร้อยละค่าว่าง

    display(
        missing_report[missing_report['missing_count'].gt(0)]
        .sort_values('missing_count', ascending=False)
    )  # แสดงเฉพาะคอลัมน์ที่มีค่าว่าง เรียงจากมากไปน้อย

# สรุปจำนวนคำขอและจำนวนผู้โดยสารแยกตามคณะ
booking_summary = (
    df_bookings_raw
    .groupby('source_faculty_code')
    .agg(
        booking_count=('booking_id', 'count'),  # นับจำนวนคำขอจอง
        average_passengers=('passenger_count', 'mean'),  # คำนวณจำนวนผู้โดยสารเฉลี่ย
        median_passengers=('passenger_count', 'median')  # คำนวณค่ากลางจำนวนผู้โดยสาร
    )
    .reset_index()
)

# สร้างตารางสรุปสถานะการจองแยกตามคณะ
status_summary = pd.crosstab(
    df_bookings_raw['source_faculty_code'],  # กำหนดแถวเป็นคณะเจ้าของข้อมูล
    df_bookings_raw['booking_status_raw']  # กำหนดคอลัมน์เป็นสถานะการจอง
)

# สร้างตารางสรุปขอบเขตการเดินทางแยกตามคณะ
scope_summary = pd.crosstab(
    df_bookings_raw['source_faculty_code'],  # กำหนดแถวเป็นคณะเจ้าของข้อมูล
    df_bookings_raw['trip_scope_raw']  # กำหนดคอลัมน์เป็นภายใน/ภายนอกจังหวัด
)

print('--- สรุปจำนวนคำขอและผู้โดยสาร ---')  # แสดงหัวข้อตารางแรก
display(booking_summary)  # แสดงสรุปคำขอและผู้โดยสาร

print('--- สถานะการจอง ---')  # แสดงหัวข้อตารางที่สอง
display(status_summary)  # แสดงสถานะการจอง

print('--- ขอบเขตการเดินทาง ---')  # แสดงหัวข้อตารางที่สาม
display(scope_summary)  # แสดงขอบเขตการเดินทาง

# สร้างพื้นที่กราฟ 2 รูปในแถวเดียว
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# วาดกราฟแท่งจำนวนคำขอจองแยกตามคณะ
booking_summary.plot.bar(
    x='source_faculty_code',  # ใช้คณะเป็นแกนนอน
    y='booking_count',  # ใช้จำนวนคำขอเป็นแกนตั้ง
    ax=axes[0],  # วาดลงในกราฟด้านซ้าย
    legend=False,  # ไม่แสดงคำอธิบายกราฟ
    color='#6b8e23'  # กำหนดสีเขียวขี้ม้า
)

axes[0].set_title('Number of booking requests by faculty')  # จำนวนคำขอจองตามคณะ
axes[0].set_xlabel('Faculty')  # คณะ
axes[0].set_ylabel('Number of requests')  # จำนวนคำขอ

# วาด Boxplot เพื่อตรวจการกระจายและค่าผิดปกติของจำนวนผู้โดยสาร
df_bookings_raw.boxplot(
    column='passenger_count',  # เลือกตัวแปรจำนวนผู้โดยสาร
    by='source_faculty_code',  # แยกกล่องตามคณะ
    ax=axes[1]  # วาดลงในกราฟด้านขวา
)

axes[1].set_title('We will contact you as soon as possible')  # การกระจายจำนวนผู้โดยสาร
axes[1].set_xlabel('Faculty')  # คณะ
axes[1].set_ylabel('Latest News')  # จำนวนผู้โดยสาร

plt.suptitle('')  # ลบหัวข้ออัตโนมัติของ Boxplot
plt.tight_layout()  # ปรับระยะห่างไม่ให้ข้อความทับกัน
plt.show()  # แสดงกราฟ


===== EDA: log คนขับ =====
ขนาด: 2,798 แถว × 14 คอลัมน์
แถวซ้ำทั้งแถว: 0 แถว
รหัส driver_log_id ซ้ำ: 0 แถว

===== EDA: คำขอจอง =====
ขนาด: 650 แถว × 28 คอลัมน์
แถวซ้ำทั้งแถว: 0 แถว
รหัส booking_id ซ้ำ: 0 แถว
--- สรุปจำนวนคำขอและผู้โดยสาร ---
--- สถานะการจอง ---
--- ขอบเขตการเดินทาง ---


                          missing_count  missing_pct
recorded_distance_km                 50     1.786991
actual_end_time                       6     0.214439
vehicle_user_name                     5     0.178699
actual_start_time                     2     0.071480
destination_province_raw              2     0.071480
destination_raw                       1     0.035740

                               missing_count  missing_pct
coordinator_name                         502    77.230769
receiver_name                            499    76.769231
requester_name                           494    76.000000
funding_source_4_payment_date            456    70.153846
funding_source_2_flag                    156    24.000000
funding_source_1_flag                    156    24.000000
requester_department                     156    24.000000
requester_person_id                      156    24.000000
receiver_person_id                       156    24.000000
coordinator_person_id                    156    24.000000
funding_source_3_flag                    156    24.000000
funding_source_4_flag                    156    24.000000
planned_end_time                          11     1.692308
dropoff_location_raw                      10     1.538462
destination_raw                            7     1.076923
passenger_count                            7     1.076923
pickup_locatio

  source_faculty_code  booking_count  average_passengers  median_passengers
0                 ICT            247            4.295547                3.0
1                 SCI            156            3.798658                2.0
2                SEEN            247            6.000000                6.0

booking_status_raw   approved  cancel  pending
source_faculty_code                           
ICT                       218      28        1
SCI                       146      10        0
SEEN                      168      49       30

trip_scope_raw       ภายนอกจังหวัด  ภายในจังหวัด
source_faculty_code                             
ICT                            118           129
SCI                             70            86
SEEN                           133           114

## Data Preparation

In [7]:
# Data Preparation - ทำความสะอาดและสร้างชุดข้อมูลพร้อมใช้
# สร้างสำเนาข้อมูลดิบ เพื่อเก็บข้อมูลต้นฉบับไว้ตรวจสอบย้อนหลัง
df_bookings = df_bookings_raw.copy()  # สำเนาข้อมูลคำขอจอง
df_drivers = df_drivers_raw.copy()  # สำเนาข้อมูล log คนขับ

# ทำความสะอาดข้อความในทั้ง 2 ตาราง
for table_df in [df_bookings, df_drivers]:  # วนทำงานกับข้อมูลจองและข้อมูลคนขับ

    text_columns = table_df.select_dtypes(include='object').columns  # เลือกเฉพาะคอลัมน์ข้อความ

    for column in text_columns:  # วนทำความสะอาดทีละคอลัมน์ข้อความ
        table_df[column] = table_df[column].astype('string').str.strip()  # ตัดช่องว่างหน้าและหลัง
        table_df[column] = table_df[column].replace({
            '': pd.NA,  # เปลี่ยนข้อความว่างเป็นค่าว่างมาตรฐาน
            '-': pd.NA,  # เปลี่ยนเครื่องหมาย - เป็นค่าว่างมาตรฐาน
            'null': pd.NA,  # เปลี่ยนข้อความ null เป็นค่าว่างมาตรฐาน
            'None': pd.NA  # เปลี่ยนข้อความ None เป็นค่าว่างมาตรฐาน
        })

# ตัดเฉพาะแถวที่ซ้ำทุกคอลัมน์ออก
df_bookings = df_bookings.drop_duplicates().copy()  # ตัดแถวจองซ้ำ
df_drivers = df_drivers.drop_duplicates().copy()  # ตัด log คนขับซ้ำ

import re as _re

def _preclean_be_date_str(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip()
    # รูปแบบ D/M/YYYY (พ.ศ. เช่น 6/6/2566 หรือคำผิด 2577 -> 2567)
    m = _re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if y == 2577:
            y = 2567  # แก้คำผิดจากการพิมพ์ปี พ.ศ. 2567 เป็น 2577 ใน sci_driversN
        if y > 2400:
            y -= 543
        return f'{y:04d}-{mo:02d}-{d:02d}'
    # รูปแบบ YYYY-MM-DD
    m2 = _re.match(r'^(\d{4})-(\d{1,2})-(\d{1,2})$', s)
    if m2:
        y, mo, d = int(m2.group(1)), int(m2.group(2)), int(m2.group(3))
        if y == 2577:
            y = 2567
        if y > 2400:
            y -= 543
        return f'{y:04d}-{mo:02d}-{d:02d}'
    return s

def parse_flexible_date(series):
    cleaned = series.map(_preclean_be_date_str)
    dt1 = pd.to_datetime(cleaned, format='%Y-%m-%d', errors='coerce')
    dt2 = pd.to_datetime(cleaned, dayfirst=True, errors='coerce')
    return dt1.fillna(dt2)

def normalize_time_str(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('.', ':')  # รองรับเวลาที่คั่นด้วยจุด เช่น 10.00 -> 10:00
    parts = s.split(':')
    if len(parts) >= 2:
        try:
            h = int(parts[0])
            m = int(parts[1])
            sec = int(parts[2]) if len(parts) >= 3 else 0
            if 0 <= h < 24 and 0 <= m < 60 and 0 <= sec < 60:
                return f'{h:02d}:{m:02d}:{sec:02d}'
        except ValueError:
            return pd.NA
    return pd.NA

# กรองแถวที่ไม่มีรหัสหลักออก
df_bookings = df_bookings[df_bookings['booking_id'].notna()].copy()
df_drivers = df_drivers[df_drivers['driver_log_id'].notna()].copy()

# แปลงคอลัมน์วันที่ของคำขอจอง
for column in ['request_date', 'planned_start_date', 'planned_end_date']:
    df_bookings[column] = parse_flexible_date(df_bookings[column])

# แปลงคอลัมน์เวลาของคำขอจองให้อยู่ในรูปแบบเดียวกัน
for column in ['planned_start_time', 'planned_end_time']:
    df_bookings[column] = df_bookings[column].apply(normalize_time_str)

# แปลงจำนวนผู้โดยสารเป็นตัวเลข
df_bookings['passenger_count'] = pd.to_numeric(
    df_bookings['passenger_count'],  # เลือกข้อมูลจำนวนผู้โดยสาร
    errors='coerce'  # ให้ค่าที่ไม่ใช่ตัวเลขเป็น NaN
)

# รวมวันและเวลาเริ่มเป็น datetime
df_bookings['planned_start_dt'] = pd.to_datetime(
    df_bookings['planned_start_date'].dt.strftime('%Y-%m-%d')  # แปลงวันเริ่มเป็นข้อความมาตรฐาน
    + ' '  # เติมช่องว่างคั่นวันและเวลา
    + df_bookings['planned_start_time'].astype('string'),  # ต่อด้วยเวลาเริ่ม
    errors='coerce'  # ให้ค่าที่แปลงไม่ได้เป็น NaT
)

# รวมวันและเวลาสิ้นสุดเป็น datetime
df_bookings['planned_end_dt'] = pd.to_datetime(
    df_bookings['planned_end_date'].dt.strftime('%Y-%m-%d')  # แปลงวันสิ้นสุดเป็นข้อความมาตรฐาน
    + ' '  # เติมช่องว่างคั่นวันและเวลา
    + df_bookings['planned_end_time'].astype('string'),  # ต่อด้วยเวลาสิ้นสุด
    errors='coerce'  # ให้ค่าที่แปลงไม่ได้เป็น NaT
)

# คำนวณระยะเวลาที่ขอใช้รถเป็นชั่วโมง
df_bookings['planned_duration_hours'] = (
    df_bookings['planned_end_dt'] - df_bookings['planned_start_dt']
).dt.total_seconds() / 3600  # แปลงผลต่างเวลาเป็นชั่วโมง

# สร้างตัวแปรตรวจสอบความถูกต้องของคำขอจอง
df_bookings['is_valid_booking_time'] = df_bookings['planned_duration_hours'].gt(0)  # เวลาสิ้นสุดต้องหลังเวลาเริ่ม
df_bookings['is_valid_passenger_count'] = df_bookings['passenger_count'].gt(0)  # ผู้โดยสารต้องมากกว่า 0
df_bookings['booking_status_clean'] = df_bookings['booking_status_raw'].str.lower()  # ทำสถานะเป็นตัวพิมพ์เล็ก

# แปลงคอลัมน์วันที่จริงของ log คนขับ
for column in ['actual_start_date', 'actual_end_date']:
    df_drivers[column] = parse_flexible_date(df_drivers[column])

# แปลงคอลัมน์เวลาจริงของ log คนขับ
for column in ['actual_start_time', 'actual_end_time']:
    df_drivers[column] = df_drivers[column].apply(normalize_time_str)

# แปลงตัวแปรระยะทางและเลขไมล์ให้เป็นตัวเลข
for column in ['odometer_start', 'odometer_end', 'recorded_distance_km']:  # ระบุคอลัมน์ตัวเลข
    df_drivers[column] = pd.to_numeric(
        df_drivers[column],  # นำคอลัมน์มาแปลง
        errors='coerce'  # ให้ค่าที่ผิดเป็น NaN
    )

# รวมวันและเวลาเริ่มจริงของคนขับ
df_drivers['actual_start_dt'] = pd.to_datetime(
    df_drivers['actual_start_date'].dt.strftime('%Y-%m-%d')
    + ' '
    + df_drivers['actual_start_time'].astype('string'),
    errors='coerce'
)

# รวมวันและเวลาสิ้นสุดจริงของคนขับ
df_drivers['actual_end_dt'] = pd.to_datetime(
    df_drivers['actual_end_date'].dt.strftime('%Y-%m-%d')
    + ' '
    + df_drivers['actual_end_time'].astype('string'),
    errors='coerce'
)

# คำนวณระยะเวลาปฏิบัติงานจริงเป็นชั่วโมง
df_drivers['actual_duration_hours'] = (
    df_drivers['actual_end_dt'] - df_drivers['actual_start_dt']
).dt.total_seconds() / 3600

# คำนวณระยะทางจากเลขไมล์ต้นทางและปลายทาง
df_drivers['odometer_distance_km'] = (
    df_drivers['odometer_end'] - df_drivers['odometer_start']
)

# คำนวณส่วนต่างของระยะทางที่บันทึกกับระยะทางจากเลขไมล์
df_drivers['distance_difference_km'] = (
    df_drivers['recorded_distance_km'] - df_drivers['odometer_distance_km']
).abs()

# กำหนดค่าเริ่มต้นให้ OUTLIER_DRIVER_LOG_IDS เป็นรายการว่าง หากยังไม่มีการกำหนดค่าไว้ที่อื่น
OUTLIER_DRIVER_LOG_IDS = []

# แยก Outlier เก็บไว้ตรวจสอบย้อนหลัง
driver_log_outliers = df_drivers[
    df_drivers['driver_log_id'].isin(OUTLIER_DRIVER_LOG_IDS)
].copy()

# ตัด Outlier ออกจากข้อมูลที่จะนำไปคำนวณ Hard Constraints
df_drivers = df_drivers[
    ~df_drivers['driver_log_id'].isin(OUTLIER_DRIVER_LOG_IDS)
].copy()

# สร้างตัวแปรตรวจสอบคุณภาพของ log คนขับ
df_drivers['is_valid_driver_time'] = df_drivers['actual_duration_hours'].gt(0)  # เวลาจบต้องหลังเวลาเริ่ม
df_drivers['is_odometer_consistent'] = df_drivers['distance_difference_km'].le(1)  # ส่วนต่างระยะทางต้องไม่เกิน 1 กม.

# คำนวณเกณฑ์ Outlier ของระยะทางด้วย IQR
q1_distance = df_drivers['recorded_distance_km'].quantile(0.25)  # หาควอไทล์ที่ 1
q3_distance = df_drivers['recorded_distance_km'].quantile(0.75)  # หาควอไทล์ที่ 3
iqr_distance = q3_distance - q1_distance  # คำนวณช่วง IQR
upper_distance_bound = q3_distance + 1.5 * iqr_distance  # คำนวณขอบบนของ Outlier

# ทำเครื่องหมาย log ที่มีระยะทางสูงผิดปกติ แต่ยังไม่ลบทิ้ง
df_drivers['is_distance_outlier'] = (
    df_drivers['recorded_distance_km'].gt(upper_distance_bound)
)

# แยก log คนขับที่มีปัญหาไว้ให้ตรวจสอบ
driver_logs_issue = df_drivers[
    ~df_drivers['is_valid_driver_time']
    | ~df_drivers['is_odometer_consistent']
].copy()

# สร้างชุดคำขอสะอาดสำหรับใช้ในขั้นตอนถัดไป
df_bookings_clean = df_bookings[
    df_bookings['is_valid_booking_time']
    & df_bookings['is_valid_passenger_count']
].copy()

# สร้างชุด log คนขับสะอาดสำหรับใช้ในขั้นตอนถัดไป
df_drivers_clean = df_drivers[
    df_drivers['is_valid_driver_time']
].copy()

# กำหนด bookings_issue ที่นี่
bookings_issue = df_bookings[
    ~df_bookings['is_valid_booking_time']
    | ~df_bookings['is_valid_passenger_count']
].copy()

# สร้างตารางสรุปผลคุณภาพข้อมูลหลังเตรียม
quality_summary = pd.DataFrame({
    'ตัวชี้วัด': [
        'คำขอสะอาด',
        'log คนขับสะอาด',
        'outlier ระยะทาง',
        'คำขอที่มีปัญหา',
        'log คนขับที่มีปัญหา'
    ],
    'จำนวนแถว': [
        len(df_bookings_clean),
        len(df_drivers_clean),
        df_drivers['is_distance_outlier'].sum(),
        len(bookings_issue),
        len(driver_logs_issue)
    ]
})

print(f'log คนขับที่มีปัญหา: {len(driver_logs_issue):,} แถว')  # รายงานจำนวน log ที่ไม่ผ่านตรวจสอบ
print(f'ขอบบน Outlier ระยะทาง: {upper_distance_bound:,.2f} กิโลเมตร')  # รายงานเกณฑ์ Outlier
display(quality_summary)  # แสดงตารางสรุปคุณภาพข้อมูล


log คนขับที่มีปัญหา: 75 แถว
ขอบบน Outlier ระยะทาง: 144.00 กิโลเมตร


             ตัวชี้วัด  จำนวนแถว
0            คำขอสะอาด       630
1       log คนขับสะอาด      2767
2      outlier ระยะทาง       400
3       คำขอที่มีปัญหา        20
4  log คนขับที่มีปัญหา        75

In [8]:
# เปลี่ยนชื่อคนขับใน DataFrame หลัก (df_drivers และ df_drivers_clean) จาก _NEW ให้เป็น _NOW
for df_target in [df_drivers, df_drivers_clean]:
    if "driver_name" in df_target.columns:
        df_target["driver_name"] = df_target["driver_name"].str.replace("_NEW", "_NOW", regex=False)

# แสดงชื่อคนขับทั้งหมดที่มีอยู่ในข้อมูลหลังจากการแปลงเพื่อตรวจสอบความถูกต้อง
print("รายชื่อคนขับทั้งหมดใน df_drivers_clean หลังเปลี่ยนเป็น _NOW:")
print(sorted(df_drivers_clean["driver_name"].unique()))

รายชื่อคนขับทั้งหมดใน df_drivers_clean หลังเปลี่ยนเป็น _NOW:
['ICT_DRIVER_NOW', 'ICT_DRIVER_OLD', 'SCI_DRIVER_NOW', 'SEEN_DRIVER_NOW', 'SEEN_DRIVER_OLD']


In [9]:
# กรองข้อมูลที่เป็น Outlier ระยะทาง (มากกว่า 442.88 กม.)
distance_outliers = df_drivers[df_drivers['is_distance_outlier'] == True].copy()

print(f"📊 สรุปข้อมูลระยะทาง Outlier (พบทั้งหมด {len(distance_outliers)} รายการ):")
print(f"- ระยะทางต่ำสุดในกลุ่ม Outlier: {distance_outliers['recorded_distance_km'].min():,.2f} กม.")
print(f"- ระยะทางสูงสุดในกลุ่ม Outlier: {distance_outliers['recorded_distance_km'].max():,.2f} กม.")
print(f"- ระยะทางเฉลี่ยของกลุ่ม Outlier: {distance_outliers['recorded_distance_km'].mean():,.2f} กม.\n")

# แสดงจำนวนข้อมูล Outlier แยกตามคนขับ
print("🚗 จำนวนรายการ Outlier แยกตามคนขับ:")
display(distance_outliers['driver_name'].value_counts().to_frame('จำนวนครั้งที่พบ'))

# แสดงตัวอย่างรายการที่เป็น Outlier
print("\n📌 ตัวอย่างข้อมูลบันทึกที่เป็น Outlier ระยะทาง (20 แถวแรก):")
columns_to_show = ['driver_name', 'vehicle_plate', 'destination_raw', 'destination_province_raw', 'recorded_distance_km', 'actual_duration_hours']
display(distance_outliers[columns_to_show].sort_values(by='recorded_distance_km', ascending=False).head(20))

📊 สรุปข้อมูลระยะทาง Outlier (พบทั้งหมด 400 รายการ):
- ระยะทางต่ำสุดในกลุ่ม Outlier: 145.00 กม.
- ระยะทางสูงสุดในกลุ่ม Outlier: 2,398.00 กม.
- ระยะทางเฉลี่ยของกลุ่ม Outlier: 377.27 กม.

🚗 จำนวนรายการ Outlier แยกตามคนขับ:

📌 ตัวอย่างข้อมูลบันทึกที่เป็น Outlier ระยะทาง (20 แถวแรก):


                 จำนวนครั้งที่พบ
driver_name                     
ICT_DRIVER_OLD               229
SCI_DRIVER_NOW                80
ICT_DRIVER_NOW                75
SEEN_DRIVER_OLD               11
SEEN_DRIVER_NOW                5

         driver_name            vehicle_plate             destination_raw destination_province_raw  recorded_distance_km  actual_duration_hours
2595  ICT_DRIVER_OLD  1 นช 3009 กรุงเทพมหานคร              รร.จปร นครนายก                  นครนายก                  2398                  160.0
2360  ICT_DRIVER_OLD  1 นช 3009 กรุงเทพมหานคร             เดินทางไปชลบุรี                   ชลบุรี                  2054                  160.0
836   ICT_DRIVER_OLD            นข 3948 พะเยา                    ศรีสะเกษ                 ศรีสะเกษ                  1871                   13.0
708   ICT_DRIVER_OLD            นข 3948 พะเยา                      ชลบุรี                   ชลบุรี                  1739                   64.0
1147  ICT_DRIVER_OLD            นข 3948 พะเยา                     กรุงเทพ                  กรุงเทพ                  1676                  157.0
2038  SCI_DRIVER_NOW            นข 3948 พะเยา                     นครนายก                  นครนายก                  1618                

In [10]:
# แก้ไขข้อมูลเฉพาะรายการ (ทำเฉพาะเมื่อพบ log นี้ในข้อมูลชุดที่โหลดมา)
_fix_mask = df_drivers['driver_log_id'] == 'DL000171'
if _fix_mask.any():
    df_drivers.loc[_fix_mask, 'destination_province_raw'] = 'อุตรดิตถ์'
    print("ตรวจสอบการเปลี่ยนแปลงสำหรับ driver_log_id 'DL000171':")
    display(df_drivers[_fix_mask][['driver_log_id', 'destination_province_raw']])
else:
    print("ℹ️ ไม่พบ driver_log_id 'DL000171' ในข้อมูลชุดนี้ (ข้ามการแก้ไข)")


ตรวจสอบการเปลี่ยนแปลงสำหรับ driver_log_id 'DL000171':


     driver_log_id destination_province_raw
2404      DL000171                อุตรดิตถ์

In [11]:
# แสดงคำขอที่มีปัญหา (เช่น เวลาไม่ถูกต้อง หรือจำนวนผู้โดยสารเป็น 0)
print(f"❌ พบคำขอที่มีปัญหาทั้งหมด {len(bookings_issue)} รายการ:")
display(bookings_issue)

❌ พบคำขอที่มีปัญหาทั้งหมด 20 รายการ:


       booking_id source_faculty_code request_date requester_person_id requester_department requester_faculty_raw trip_scope_raw                                                                                                             purpose_raw  passenger_count planned_start_date planned_start_time planned_end_date planned_end_time                           destination_raw destination_province_raw                       pickup_location_raw                       dropoff_location_raw receiver_person_id coordinator_person_id  funding_source_1_flag  funding_source_2_flag  funding_source_3_flag  funding_source_4_flag funding_source_4_payment_date booking_status_raw requester_name receiver_name coordinator_name    planned_start_dt      planned_end_dt  planned_duration_hours  is_valid_booking_time  is_valid_passenger_count booking_status_clean
264  SCI_BK000140                 SCI   2023-12-27                <NA>                 <NA>                   SCI   ภายในจังหวัด                    

In [12]:
# ลบแถวที่มี booking_id เป็น 'BK000294' ออกจาก df_bookings_clean (ทำเฉพาะเมื่อพบรายการนี้)
before_count = len(df_bookings_clean)
if (df_bookings_clean['booking_id'] == 'BK000294').any():
    df_bookings_clean = df_bookings_clean[df_bookings_clean['booking_id'] != 'BK000294'].copy()
    print(f"✅ ลบรายการ BK000294 สำเร็จ (จำนวนแถวลดลงจาก {before_count} เหลือ {len(df_bookings_clean)})")
else:
    print("ℹ️ ไม่พบ booking_id 'BK000294' ในข้อมูลชุดนี้ (ข้ามการลบ)")


ℹ️ ไม่พบ booking_id 'BK000294' ในข้อมูลชุดนี้ (ข้ามการลบ)


## EDA เพิ่มเติม: งานในปฏิทิน (booking) เทียบกับงานนอกปฏิทิน (walk-in)
ใน log คนขับ มีทั้งงานที่มาจากคำขอจอง (`vehicle_user_name` = `ผู้ใช้บริการ_...` และเวลาทับกับ booking ที่อนุมัติ) และงานที่ผู้ดูแลสั่งตรงในคณะ (เช่น `ภารกิจเร่งด่วน`, `รับส่งเอกสาร`) ซึ่งไม่มีใน booking เลย
งานกลุ่มหลังเลี่ยงไม่ได้ ระบบจึงต้องมองเป็น *ช่วงเวลาที่ไม่ว่างแบบคงที่* ก่อนจัดสรรคำขอใหม่

In [13]:
# EDA: แยก log ที่มี booking รองรับ ออกจาก walk-in และดูความต่างของระยะเวลา
BOOKED_USER_PREFIX = "ผู้ใช้บริการ_"
_eda = {}
for _fac in sorted(df_drivers_clean['source_faculty_code'].unique()):
    _d = df_drivers_clean[df_drivers_clean['source_faculty_code'] == _fac].copy()
    _b = df_bookings_clean[df_bookings_clean['source_faculty_code'] == _fac].copy()
    _d['s'] = pd.to_datetime(_d['actual_start_date'].astype(str) + ' ' + _d['actual_start_time'].astype(str))
    _d['e'] = pd.to_datetime(_d['actual_end_date'].astype(str) + ' ' + _d['actual_end_time'].astype(str))
    _b['s'] = pd.to_datetime(_b['planned_start_date'].astype(str) + ' ' + _b['planned_start_time'].astype(str))
    _b['e'] = pd.to_datetime(_b['planned_end_date'].astype(str) + ' ' + _b['planned_end_time'].astype(str))
    _d['hours'] = (_d['e'] - _d['s']).dt.total_seconds() / 3600
    _b['hours'] = (_b['e'] - _b['s']).dt.total_seconds() / 3600
    _d['is_booked'] = _d['vehicle_user_name'].astype(str).str.startswith(BOOKED_USER_PREFIX)
    _appr = _b[_b['booking_status_raw'].astype(str).str.lower() == 'approved']
    _eda[_fac] = (_d, _b, _appr)
    print(f"===== {_fac} =====")
    print(f"log ทั้งหมด {len(_d):,} | มี booking รองรับ {_d['is_booked'].sum():,} | walk-in {(~_d['is_booked']).sum():,} ({(~_d['is_booked']).mean():.1%})")
    print(f"booking อนุมัติ {len(_appr):,} รายการ; ช่วงเวลาจองเฉลี่ย {_appr['hours'].mean():.1f} ชม. แต่ log จริงของงานเดียวกันเฉลี่ย {_d.loc[_d['is_booked'],'hours'].mean():.1f} ชม.")
    print(f"booking ที่ช่วงเวลาจอง > 14 ชม. (ทริปหลายวัน): {(_b['hours'] > 14).sum()} รายการ, ยาวสุด {_b['hours'].max():.1f} ชม.")
    _tmp = _d.sort_values('s')
    _gap_viol = 0
    for _drv, _g in _tmp.groupby('driver_name'):
        _gap = (_g['s'].shift(-1) - _g['e']).dt.total_seconds() / 3600
        _gap_viol += int((_gap < MIN_REST_HOURS_EDA).sum()) if 'MIN_REST_HOURS_EDA' in globals() else int((_gap < 10).sum())
    print(f"ช่วงว่างระหว่าง log ติดกัน (ต่อคนขับ) < 10 ชม.: {_gap_viol} ครั้ง จาก {len(_d)-_d['driver_name'].nunique()} ช่วง")
    print("walk-in แยกตามวัตถุประสงค์ (บันทึกใน log):")
    display(_d.loc[~_d['is_booked'], 'vehicle_user_name'].value_counts().to_frame('จำนวน'))
    print("การกระจายชั่วโมงเริ่มงานใน log (ถ้ากระจายเกือบเท่ากันทุกชั่วโมง แปลว่า 'เข้างาน 08:00' ไม่ปรากฏในข้อมูล):")
    display(_d['s'].dt.hour.value_counts().sort_index().to_frame('จำนวน').T)


===== ICT =====
log ทั้งหมด 598 | มี booking รองรับ 0 | walk-in 598 (100.0%)
booking อนุมัติ 217 รายการ; ช่วงเวลาจองเฉลี่ย 15.5 ชม. แต่ log จริงของงานเดียวกันเฉลี่ย nan ชม.
booking ที่ช่วงเวลาจอง > 14 ชม. (ทริปหลายวัน): 41 รายการ, ยาวสุด 206.0 ชม.
ช่วงว่างระหว่าง log ติดกัน (ต่อคนขับ) < 10 ชม.: 171 ครั้ง จาก 596 ช่วง
walk-in แยกตามวัตถุประสงค์ (บันทึกใน log):
การกระจายชั่วโมงเริ่มงานใน log (ถ้ากระจายเกือบเท่ากันทุกชั่วโมง แปลว่า 'เข้างาน 08:00' ไม่ปรากฏในข้อมูล):
===== SCI =====
log ทั้งหมด 1,571 | มี booking รองรับ 0 | walk-in 1,571 (100.0%)
booking อนุมัติ 135 รายการ; ช่วงเวลาจองเฉลี่ย 11.4 ชม. แต่ log จริงของงานเดียวกันเฉลี่ย nan ชม.
booking ที่ช่วงเวลาจอง > 14 ชม. (ทริปหลายวัน): 18 รายการ, ยาวสุด 157.0 ชม.
ช่วงว่างระหว่าง log ติดกัน (ต่อคนขับ) < 10 ชม.: 1009 ครั้ง จาก 1569 ช่วง
walk-in แยกตามวัตถุประสงค์ (บันทึกใน log):
การกระจายชั่วโมงเริ่มงานใน log (ถ้ากระจายเกือบเท่ากันทุกชั่วโมง แปลว่า 'เข้างาน 08:00' ไม่ปรากฏในข้อมูล):
===== SEEN =====
log ทั้งหมด 598 | มี booking รองรับ 168 |

                   จำนวน
vehicle_user_name       
ICT                  563
กองวิจัย               4
รร.สาธิต ม.พะเยา       3
เภสัช                  3
กองบริการศึกษา         2
LAW                    2
BCA                    2
คณะวิทย์               2
ศูนย์ศิลปะ             2
สถาบันถ่ายทอด          1
ศูนศิลปะ               1
กองกฎหมาย              1
คระบริหาร              1
ICT,กองวิจัย           1
คณะเกษตร               1
คณะวิทย์แพทย์          1
วิทย์แพทย์             1
สภามอ                  1
รร.สาธิต               1
กองแผน                 1
คณะทันตะ               1
คณะBCA                 1
สำนักงานอธิการ         1
คณะสถาปัตยกรรม         1

s      1   2   3   4   5   6   7    8   9   10  11  12  13  14  15  16  17  18
จำนวน   1   2   3   5  12  24  35  162  72  85  38  29  48  38  26  11   6   1

                              จำนวน
vehicle_user_name                  
ป้านก                           466
พี่อ้อม                         190
เบียร์                           64
พี่หนู                           59
อ.ติ                             54
พี่เผ้า                          54
พี่จักรี                         53
พี่เย้า                          47
พี่อึ่ง                          38
น้องกุ๊ก                         35
ขวัญชาย                          24
พี่ขวัญหญิง                      18
น้องเก๋                          17
พี่ขวัญ                          16
พี่                              12
ป้านก + พี่หนู                   12
พนักงานรถตู้                     11
พี่อ้อ                           10
นัท                               9
คณะบดี                            7
พี่อิง                            7
กองบริการ                         7
พี่อึ๋ง                           6
อ.ยา                              6
สถาบัน                            6
สาธิต                       

s      1   3   4   5   6   7    8    9    10   11  12  13   14   15  16  17  19  20
จำนวน   1   1   1  10  44  36  122  117  358  136  59  95  339  155  89   5   1   2

                       จำนวน
vehicle_user_name           
รับส่งบุคลากรระยะสั้น     84
ภารกิจเร่งด่วน            80
รับส่งเอกสาร              76
รับส่งพัสดุ               70
ติดต่อหน่วยงาน            63
งานธุรการคณะ              57

s      0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23
จำนวน  38  21  33  29  28  27  15  28  26  29  24  27  20  17  21  19  30  22  26  25  28  22  21  22

# Test Case

In [14]:
# กำหนดข้อจำกัดชั่วโมง (Hard constraints)
MAX_DAILY_HOURS = 14.0     # ชั่วโมงทำงานสูงสุดต่อ "วันปฏิบัติงาน"
MAX_WEEKLY_HOURS = 72.0    # ชั่วโมงทำงานสะสมสูงสุดในช่วง 7 วันปฏิบัติงานที่เลื่อนต่อเนื่อง (rolling)
MIN_REST_HOURS = 10.0      # เวลาพักขั้นต่ำระหว่างทริป

# รถตู้เข้างาน 08:00 น. เมื่อไม่มีงานในปฏิทิน -> ใช้เป็นจุดเริ่มของ "วันปฏิบัติงาน" (08:00 -> 08:00 วันถัดไป)
DUTY_DAY_START_HOUR = 8

# ทริปหลายวัน: ทุกรอบ 24 ชม. นับจากเวลาเริ่มทริป = ทำงาน 14 ชม. + พัก 10 ชม.
MULTIDAY_WORK_HOURS = 14.0
MULTIDAY_REST_HOURS = 10.0

FACULTIES = ["ICT", "SCI", "SEEN"]

# ช่วงเวลาที่ใช้ตรวจสถานะคนขับปัจจุบันในข้อมูลตัวอย่าง
AS_OF = pd.Timestamp("2026-09-28 08:00:00")


# Mathematical Formulations

## 1. เซตและพารามิเตอร์
| สัญลักษณ์ | ความหมาย |
|---|---|
| $I$ | เซตทรัพยากร (คู่ รถตู้–คนขับ) ปกติ $\lvert I\rvert=3$ (ICT, SCI, SEEN) |
| $J$ | เซตคำขอจอง |
| $D$ | เซต "วันปฏิบัติงาน" (วัน $d$ = ช่วง $[d+08{:}00,\ d+1\text{ วัน}+08{:}00)$) |
| $o_i$, $f_j$ | คณะเจ้าของรถ $i$ / คณะผู้ขอ $j$ |
| $n_j$ | จำนวนรถที่คำขอ $j$ ต้องการ |
| $[s_j, e_j]$ | ช่วงเวลาจอง (ใช้เป็นเวลาทำงาน เพราะไม่มีเวลาขับจริง) |
| $w_{jd}$ | **ชั่วโมงทำงานของคำขอ $j$ ในวันปฏิบัติงาน $d$** (คำนวณล่วงหน้าจาก Logic Breakdown) |
| $H_{id}$ | ชั่วโมงทำงานเดิมของ $i$ ในวัน $d$ (จาก busy blocks: walk-in + booking ที่อนุมัติแล้ว) |
| $a_i$, $\text{cap}_i$, $p_j$ | รถพร้อมใช้ (0/1), ความจุ (11), จำนวนผู้โดยสาร |
| $R=10$, $L_d=14$, $L_w=72$ | เวลาพักขั้นต่ำ, ชม./วัน สูงสุด, ชม./สัปดาห์ สูงสุด |
| $e_{ij}\in\{0,1\}$ | สิทธิ์ใช้ได้ (ดูข้อ 4.2) |

## 2. ตัวแปรตัดสินใจ
$$x_{ij}\in\{0,1\}\quad\text{(1 ถ้าจัดทรัพยากร } i \text{ ให้คำขอ } j\text{)},\qquad m_j\in\{0,1,\dots,n_j\}\quad\text{(จำนวนรถที่จัดไม่ได้)}$$

## 3. ฟังก์ชันวัตถุประสงค์ (Lexicographic, แก้ทีละขั้น แล้วล็อกค่าที่ได้)
$$\begin{aligned}
Z_1&=\sum_{j\in J} m_j &&\text{(1) จัดให้ได้มากที่สุด}\\
Z_2&=\sum_{i,j:\ o_i\neq f_j} x_{ij} &&\text{(2) ใช้รถข้ามคณะให้น้อยที่สุด = รถคณะใดให้คณะนั้นก่อน}\\
Z_3&=\sum_{i,j}\pi(f_j,o_i)\,x_{ij} &&\text{(3) ตัดสินเสมอ: } \pi=1 \text{ รถตัวเอง}, \pi=2 \text{ รถคณะอื่น}
\end{aligned}$$
รูปแบบเทียบเท่า (`objective_method="weighted"`): $\min\ M_1Z_1+M_2Z_2+Z_3$ โดย $M_1\gg M_2\gg 1$

## 4. เงื่อนไข (Constraints)
**4.1 ครบจำนวนรถหรือบันทึกว่าขาด**
$$\sum_{i\in I}x_{ij}+m_j=n_j\qquad\forall j\in J \tag{C1}$$

**4.2 สิทธิ์ใช้ได้** $\;x_{ij}\le e_{ij}$ โดย
$$e_{ij}=a_i\cdot\mathbb 1[\text{cap}_i\ge p_j]\cdot\mathbb 1[\text{ไม่ชน/พักไม่ครบกับ busy block ใด ๆ}]\cdot\mathbb 1[M_j=1\ \lor\ e_j-s_j\le L_d] \tag{C2}$$
(พจน์สุดท้าย: ทริปวันเดียวต้องไม่ยาวเกิน 14 ชม. ต่อเนื่อง)

**4.3 ไม่ชนเวลา และพักอย่างน้อย 10 ชม. ระหว่างทริปของทรัพยากรเดียวกัน**
$$x_{ij}+x_{ik}\le1\qquad \forall i,\ \forall j<k:\ \ [s_j,e_j)\cap[s_k,e_k)\neq\varnothing\ \ \lor\ \ 0\le s_k-e_j<R\ \ \lor\ \ 0\le s_j-e_k<R \tag{C3}$$

**4.4 ไม่เกิน 14 ชั่วโมงต่อวันปฏิบัติงาน**
$$\sum_{j\in J}w_{jd}\,x_{ij}\;\le\;L_d-H_{id}\qquad\forall i,\ \forall d\in D \tag{C4}$$

**4.5 ไม่เกิน 72 ชั่วโมงในหน้าต่าง 7 วันปฏิบัติงาน (rolling)**
$$\sum_{d'=d-6}^{d}\ \sum_{j\in J}w_{jd'}\,x_{ij}\;\le\;L_w-\sum_{d'=d-6}^{d}H_{id'}\qquad\forall i,\ \forall d \tag{C5}$$

(ถ้า $H_{id}$ เกินเพดานอยู่แล้ว ระบบบังคับ $x_{ij}=0$ ให้ทุกคำขอที่แตะวันนั้น)

# Logic Breakdown: ทริปข้ามวัน = ทำงาน 14 ชม. / พัก 10 ชม. → สมการที่ Solver ประมวลผลได้

**แนวคิด**: solver เป็น MILP จึงรับได้เฉพาะ *ค่าคงที่* ที่คูณกับตัวแปร $x_{ij}$ เราจึงแปลงกติกาทั้งหมดเป็นตาราง $w_{jd}$ (ชั่วโมงทำงานของคำขอ $j$ ในวัน $d$) ล่วงหน้าด้วยโค้ด แล้วให้ solver ตรวจแค่ผลรวมเชิงเส้น (C4), (C5)

### ขั้น 1 — วันปฏิบัติงาน (จุดตัด 08:00 น.)
รถตู้เข้างาน 08:00 น. เมื่อไม่มีงานในปฏิทิน จึงนับวันปฏิบัติงาน $d$ เป็นช่วง $[d+8\text{ชม.},\ d+32\text{ชม.})$ (ไม่ใช่ 00:00–24:00) ทำให้งานกลางคืนที่ข้ามเที่ยงคืน (เช่น 22:00–02:00) ไม่ถูกตัดครึ่ง

### ขั้น 2 — จำแนกทริป (`check_multi_day_trip`)
$$M_j=\mathbb 1\big[\,e_j>\lfloor s_j\rfloor_{\text{วัน}}+1\text{ วัน}+08{:}00\,\big]$$
- $M_j=0$ **ทริปวันเดียว**: จบก่อนรถเข้างานรอบถัดไป → ชั่วโมงทำงาน = เวลาจองจริง $D_j=e_j-s_j$ และต้อง $D_j\le14$ (ไม่เช่นนั้นคนขับคนเดียวทำไม่ได้ → ไม่มีรถผ่าน)
- $M_j=1$ **ทริปหลายวัน**: ใช้ขั้น 3

### ขั้น 3 — ทริปหลายวัน: รอบ 24 ชม. นับจากเวลาเริ่มทริป
รอบที่ $c=0,1,2,\dots$ มีช่วงทำงาน $B_{jc}$ และช่วงพัก:
$$B_{jc}=\big[\,s_j+24c,\ \min(e_j,\ s_j+24c+14)\,\big),\qquad \text{พัก}=[\,s_j+24c+14,\ s_j+24(c+1)\,)\ \ (10\text{ ชม.})$$
ชั่วโมงทำงานรวม $W_j=14\lfloor D_j/24\rfloor+\min\!\big(14,\ D_j \bmod 24\big)$
**คุณสมบัติ**: รูปแบบนี้เป็นคาบ 24 ชม. ดังนั้น *ทุกหน้าต่าง 24 ชม.* มีงานไม่เกิน 14 ชม. และมีพักต่อเนื่อง 10 ชม. ทุกรอบ โดยไม่ขึ้นกับว่าเริ่มทริปตอนกี่โมง

### ขั้น 4 — แปลงเป็นค่าคงที่ $w_{jd}$
$$w_{jd}=\sum_{c}\big|\,B_{jc}\cap[\,d+8\text{ชม.},\ d+32\text{ชม.})\,\big|$$
(ทริปวันเดียวใช้ $B_{j0}=[s_j,e_j)$ ก้อนเดียว) busy block ในอดีตก็ใช้สูตรเดียวกันเพื่อได้ $H_{id}$ → ได้ (C4), (C5) ที่เป็นเชิงเส้นล้วน

### ขั้น 5 — ตัวอย่างจากข้อมูลจริง (SCI_BK_0025: 5 มี.ค. 11:16 → 6 มี.ค. 22:41, 35.4 ชม.)
$M_j=1$ เพราะจบหลัง 08:00 ของ 6 มี.ค. | $B_{j0}=$ 5 มี.ค. 11:16→6 มี.ค. 01:16 (14 ชม.) | $B_{j1}=$ 6 มี.ค. 11:16→22:41 (11.4 ชม.)
$w_{j,\,5\text{ มี.ค.}}=14$, $w_{j,\,6\text{ มี.ค.}}=11.4$ รวม 25.4 ชม. (ไม่ใช่ 35.4 และไม่ใช่ 28)

### ขั้น 6 — ผลต่อเพดาน 72 ชม./สัปดาห์
ทริปต่อเนื่องที่ยาวที่สุดที่คนขับคนเดียวทำได้ ≈ 5 วัน 2 ชม. (122 ชม. → $W=70+2=72$) ทริป 6 วัน $=14\times5+12=82>72$ จึงถูกปฏิเสธโดย (C5)

### ข้อสมมติที่ควรยืนยันกับผู้ดูแล
1. ทริปหลายวันถือว่าคนขับได้พักค้างคืนจริงตามรอบ 14/10 (ตามกติกาที่กำหนด) — ทริปที่ยาว 14–24 ชม. และข้ามจุดตัด 08:00 จะถูกจัดเป็นหลายวันและนับทำงาน 14 ชม.
2. ใช้เวลาจอง = เวลาทำงาน (ไม่มีเวลาขับจริง) ซึ่งสูงกว่าเวลาใน log จริงราว 3 เท่า → โมเดลจึงระมัดระวังเกินจริงเล็กน้อย (ปลอดภัยไว้ก่อน)
3. คำขอด่วนข้ามหน่วยงานไม่เข้า solver: ผู้ขอโทรหาผู้ดูแลรถตู้ ถ้าอนุมัติจะบันทึกเป็น busy block ของรถนั้น

In [15]:
# ---- แบบจำลองเวลาแบบเชิงเส้น: วันปฏิบัติงาน, การจำแนกทริป, ตารางชั่วโมงทำงาน w[j,d] ----
import pandas as pd

DUTY_DAY_START = pd.Timedelta(hours=DUTY_DAY_START_HOUR)

def duty_day_of(ts):
    """คืนรหัสวันปฏิบัติงานของเวลา ts (วัน d = [d 08:00, d+1 08:00))"""
    return (pd.Timestamp(ts) - DUTY_DAY_START).normalize()

def duty_day_end(d):
    return d + pd.Timedelta(days=1) + DUTY_DAY_START

def check_multi_day_trip(start_dt, end_dt):
    """
    ตรวจว่าการจองเป็นทริปหลายวันหรือไม่ โดยใช้จุดตัด 08:00 น. (รถตู้เข้างานปกติ)
    หากเวลาสิ้นสุดเกิน 08:00 น. ของวันถัดไปจากวันเริ่มงาน ถือว่าเกิน 1 วันปฏิบัติงาน
    """
    start, end = pd.Timestamp(start_dt), pd.Timestamp(end_dt)
    threshold_next_day = start.normalize() + pd.Timedelta(days=1) + DUTY_DAY_START
    if end > threshold_next_day:
        days_count = ((end - start.normalize()).days) + (1 if end.time() > (pd.Timestamp("00:00") + DUTY_DAY_START).time() else 0)
        return True, f"เดินทางหลายวัน ({days_count} วันปฏิบัติงาน) เนื่องจากสิ้นสุดหลังเวลาเข้างานวันถัดไป ({threshold_next_day})"
    return False, "เดินทางวันเดียว (สิ้นสุดก่อนจุดตัดเข้างานของวันถัดไป 08:00 น.)"

def work_blocks(start, end):
    """คืน (is_multi_day, [(b0, b1), ...]) = ช่วงเวลาที่นับเป็นชั่วโมงทำงาน
    - ทริปวันเดียว: ก้อนเดียว = เวลาจอง
    - ทริปหลายวัน: ทุกรอบ 24 ชม. นับจากเวลาเริ่ม ทำงาน MULTIDAY_WORK_HOURS แล้วพักที่เหลือ (10 ชม.)"""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    multi, _ = check_multi_day_trip(s, e)
    if not multi:
        return False, [(s, e)]
    cycle = MULTIDAY_WORK_HOURS + MULTIDAY_REST_HOURS
    blocks, c = [], 0
    while True:
        w0 = s + pd.Timedelta(hours=cycle * c)
        if w0 >= e:
            break
        blocks.append((w0, min(e, w0 + pd.Timedelta(hours=MULTIDAY_WORK_HOURS))))
        c += 1
    return True, blocks

def daily_profile(start, end):
    """คืน (is_multi_day, {วันปฏิบัติงาน: ชั่วโมงทำงาน}) = w[j,d]"""
    multi, blocks = work_blocks(start, end)
    prof = {}
    for b0, b1 in blocks:
        cur = b0
        while cur < b1:
            d = duty_day_of(cur)
            nxt = min(b1, duty_day_end(d))
            prof[d] = prof.get(d, 0.0) + (nxt - cur).total_seconds() / 3600
            cur = nxt
    return multi, prof

def _hist_profile(logs):
    """รวมชั่วโมงทำงานเดิม H[i,d] จาก busy blocks (ใช้กติกาเดียวกับคำขอ)"""
    prof = {}
    if logs is None or len(logs) == 0:
        return prof
    for r in logs.itertuples(index=False):
        if pd.isna(r.start_dt) or pd.isna(r.end_dt) or r.end_dt <= r.start_dt:
            continue
        _, p = daily_profile(r.start_dt, r.end_dt)
        for d, h in p.items():
            prof[d] = prof.get(d, 0.0) + h
    return prof

def _window_sum(prof, end_day, days=7):
    lo = end_day - pd.Timedelta(days=days - 1)
    return float(sum(h for d, h in prof.items() if lo <= d <= end_day))

# ---- ตัวอย่างและการตรวจความถูกต้องของตรรกะ ----
_examples = [
    ("วันเดียว 08:00-18:00", "2026-09-29 08:00", "2026-09-29 18:00"),
    ("วันเดียว 15 ชม. ต่อเนื่อง", "2026-09-29 08:00", "2026-09-29 23:00"),
    ("กลางคืนข้ามเที่ยงคืน 22:00-04:00", "2026-09-29 22:00", "2026-09-30 04:00"),
    ("หลายวัน 3 วัน 08:00 -> 20:00 วันที่ 3", "2026-09-29 08:00", "2026-10-01 20:00"),
    ("SCI_BK_0025 จากข้อมูลจริง (35.4 ชม.)", "2025-03-05 11:16", "2025-03-06 22:41"),
    ("หลายวัน 6 วัน", "2026-09-29 08:00", "2026-10-04 20:00"),
]
_rows = []
for _n, _s, _e in _examples:
    _m, _p = daily_profile(_s, _e)
    _rows.append({"กรณี": _n, "ชม.จองจริง": round((pd.Timestamp(_e) - pd.Timestamp(_s)).total_seconds()/3600, 2),
                  "หลายวัน?": _m, "ชม.ทำงานที่นับ": round(sum(_p.values()), 2),
                  "แยกตามวันปฏิบัติงาน": {k.strftime("%d/%m"): round(v, 2) for k, v in _p.items()}})
display(pd.DataFrame(_rows))
assert abs(sum(daily_profile("2026-09-29 08:00", "2026-10-01 20:00")[1].values()) - 40) < 1e-9      # 14+14+12
assert abs(sum(daily_profile("2026-09-29 08:00", "2026-10-04 20:00")[1].values()) - 82) < 1e-9      # 14*5+12 > 72
assert check_multi_day_trip("2026-09-29 22:00", "2026-09-30 08:00")[0] is False                       # จบพอดี 08:00 = วันเดียว
assert check_multi_day_trip("2026-09-29 22:00", "2026-09-30 08:01")[0] is True
print("✅ ตรรกะทริปหลายวันผ่านการตรวจเบื้องต้น")


✅ ตรรกะทริปหลายวันผ่านการตรวจเบื้องต้น


                                    กรณี  ชม.จองจริง  หลายวัน?  ชม.ทำงานที่นับ                                                                         แยกตามวันปฏิบัติงาน
0                   วันเดียว 08:00-18:00       10.00     False           10.00                                                                             {'29/09': 10.0}
1              วันเดียว 15 ชม. ต่อเนื่อง       15.00     False           15.00                                                                             {'29/09': 15.0}
2       กลางคืนข้ามเที่ยงคืน 22:00-04:00        6.00     False            6.00                                                                              {'29/09': 6.0}
3  หลายวัน 3 วัน 08:00 -> 20:00 วันที่ 3       60.00      True           40.00                                               {'29/09': 14.0, '30/09': 14.0, '01/10': 12.0}
4   SCI_BK_0025 จากข้อมูลจริง (35.4 ชม.)       35.42      True           25.42                                                             {'05/0

In [16]:
# เตรียมและทำความสะอาด driver log
def prepare_driver_logs(df):
    out = df.copy()

    required = [
        "driver_name",  # ชื่คนขับ
        "vehicle_plate", # ป้ายทะเบียนรถ
        "actual_start_date",  # วันที่เริ่มทริป
        "actual_start_time", # เวลาที่เริ่มทริป
        "actual_end_date",  # วันที่จบทริป
        "actual_end_time", # เวลาที่จบทริป
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"ขาดคอลัมน์ driver log: {missing}")

    out["start_dt"] = pd.to_datetime(
        out["actual_start_date"].astype(str) + " " +
        out["actual_start_time"].astype(str),
        errors="coerce"
    )
    out["end_dt"] = pd.to_datetime(
        out["actual_end_date"].astype(str) + " " +
        out["actual_end_time"].astype(str),
        errors="coerce"
    )

    out = out.dropna(subset=["driver_name", "start_dt", "end_dt"]).copy()
    out = out[out["end_dt"] >= out["start_dt"]].copy()
    out["trip_hours"] = (
        (out["end_dt"] - out["start_dt"]).dt.total_seconds() / 3600
    )

    return out.sort_values(["driver_name", "start_dt"]).reset_index(drop=True)

# Group df_drivers_clean by 'source_faculty_code' to get DataFrames per faculty
df_drivers_grouped_by_faculty = df_drivers_clean.groupby('source_faculty_code')

driver_logs = {
    faculty: prepare_driver_logs(df)
    for faculty, df in df_drivers_grouped_by_faculty
}

In [17]:
# Drivers
NOW_DRIVER_KEYWORDS = {
    "ICT": ["NOW"],
    "SCI": ["NOW"],
    "SEEN": ["NOW"],
}

def is_new_driver(name, faculty):
    name = str(name).upper()
    return any(
        keyword.upper() in name
        for keyword in NOW_DRIVER_KEYWORDS[faculty]
    )

is_now_driver = is_new_driver

now_driver_logs = {}

for faculty, df in driver_logs.items():
    now_driver_logs[faculty] = df[
        df["driver_name"].apply(lambda x: is_new_driver(x, faculty))
    ].copy()

for faculty, df in now_driver_logs.items():
    print(f"{faculty}: คนขับใหม่ในข้อมูล =", sorted(df["driver_name"].unique()))

ICT: คนขับใหม่ในข้อมูล = ['ICT_DRIVER_NOW']
SCI: คนขับใหม่ในข้อมูล = ['SCI_DRIVER_NOW']
SEEN: คนขับใหม่ในข้อมูล = ['SEEN_DRIVER_NOW']


In [18]:
# สร้างทะเบียนคนขับใหม่ประจำคณะ
driver_master_rows = []

for faculty, df in now_driver_logs.items():
    for driver_name, group in df.groupby("driver_name"):
        first_log = group["start_dt"].min()
        last_log = group["end_dt"].max()

        plates = group["vehicle_plate"].dropna().astype(str).unique()
        plate = plates[0] if len(plates) else None

        driver_master_rows.append({
            "faculty": faculty,
            "driver_name": driver_name,
            "vehicle_plate": plate,
            "plate_suffix": (
                plate.split()[-1] if isinstance(plate, str) else None
            ),
            "first_recorded_work_date": first_log,
            "last_recorded_trip_end": last_log,
            "employment_status": (
                "ยังทำงานอยู่ (อนุมานจากมีประวัติในข้อมูล)"
                if last_log >= AS_OF - pd.Timedelta(days=90)
                else "ต้องตรวจสอบสถานะกับหน่วยงาน"
            ),
            "log_count": len(group),
        })

driver_master = pd.DataFrame(driver_master_rows)
display(driver_master)

  faculty      driver_name            vehicle_plate   plate_suffix first_recorded_work_date last_recorded_trip_end                          employment_status  log_count
0     ICT   ICT_DRIVER_NOW  1 นช 3009 กรุงเทพมหานคร  กรุงเทพมหานคร      2025-11-30 17:30:00    2026-10-10 18:00:00  ยังทำงานอยู่ (อนุมานจากมีประวัติในข้อมูล)        309
1     SCI   SCI_DRIVER_NOW            นข 3948 พะเยา          พะเยา      2024-10-01 10:00:00    2025-09-30 16:30:00                ต้องตรวจสอบสถานะกับหน่วยงาน        666
2    SEEN  SEEN_DRIVER_NOW              SIM-SEEN-01    SIM-SEEN-01      2025-11-15 12:45:00    2026-07-31 14:39:00  ยังทำงานอยู่ (อนุมานจากมีประวัติในข้อมูล)        229

In [19]:
# สร้าง "busy blocks" = ช่วงเวลาที่คนขับ/รถไม่ว่างแบบคงที่ จากข้อมูลจริง
#  - walk-in : log ที่ไม่มี booking รองรับ (ธุระด่วนในคณะ ฯลฯ) ใช้เวลาตาม log
#  - booked  : booking ที่อนุมัติแล้ว ใช้ "ช่วงเวลาจอง" เป็นเวลาทำงาน (ตามกติกา: ไม่มีเวลาขับจริง)
BOOKED_USER_PREFIX = "ผู้ใช้บริการ_"

def prepare_booking_times(df):
    out = df.copy()
    out["start_dt"] = pd.to_datetime(out["planned_start_date"].astype(str) + " " + out["planned_start_time"].astype(str), errors="coerce")
    out["end_dt"] = pd.to_datetime(out["planned_end_date"].astype(str) + " " + out["planned_end_time"].astype(str), errors="coerce")
    return out.dropna(subset=["start_dt", "end_dt"]).copy()

bookings_prepared = {fac: prepare_booking_times(df) for fac, df in df_bookings_clean.groupby("source_faculty_code")}

def build_busy_blocks(logs, bookings):
    """คืน (blocks_df, สรุปการจับคู่)"""
    logs = logs.copy().reset_index(drop=True)
    appr = bookings[bookings["booking_status_raw"].astype(str).str.lower() == "approved"].copy()
    logs["is_booked"] = False
    matched_driver = {}
    for k, r in logs.iterrows():
        if not str(r.get("vehicle_user_name", "")).startswith(BOOKED_USER_PREFIX):
            continue
        hit = appr[(appr["start_dt"] < r["end_dt"]) & (r["start_dt"] < appr["end_dt"])]
        if len(hit):
            logs.at[k, "is_booked"] = True
            matched_driver.setdefault(hit.index[0], r["driver_name"])
    rows = []
    for r in logs[~logs["is_booked"]].itertuples(index=False):
        rows.append({"driver_name": r.driver_name, "vehicle_plate": r.vehicle_plate, "start_dt": r.start_dt,
                     "end_dt": r.end_dt, "block_type": "walk_in", "ref_id": r.driver_log_id})
    no_driver = 0
    plate = logs["vehicle_plate"].dropna().astype(str).iloc[0] if len(logs) else None
    for idx, b in appr.iterrows():
        drv = matched_driver.get(idx)
        if drv is None:   # ไม่มี log จับคู่ -> ใช้คนขับที่ log ครอบช่วงเวลานั้น
            cover = logs[(logs["start_dt"] <= b["start_dt"])].sort_values("start_dt")
            drv = cover["driver_name"].iloc[-1] if len(cover) else None
        if drv is None:
            no_driver += 1; continue
        rows.append({"driver_name": drv, "vehicle_plate": plate, "start_dt": b["start_dt"],
                     "end_dt": b["end_dt"], "block_type": "booked", "ref_id": b["booking_id"]})
    out = pd.DataFrame(rows)
    if len(out):
        out["trip_hours"] = (out["end_dt"] - out["start_dt"]).dt.total_seconds() / 3600
        out = out.sort_values(["driver_name", "start_dt"]).reset_index(drop=True)
    summary = {"log_total": len(logs), "log_booked": int(logs["is_booked"].sum()), "log_walk_in": int((~logs["is_booked"]).sum()),
               "approved_bookings": len(appr), "approved_without_driver": no_driver}
    return out, summary

all_driver_blocks, now_driver_blocks, block_summary = {}, {}, []
for faculty, logs in driver_logs.items():
    if faculty not in bookings_prepared:
        print(f"⚠️ {faculty}: ไม่มีข้อมูล booking -> ใช้ log ทั้งหมดเป็น walk-in")
        bk = pd.DataFrame(columns=["booking_status_raw", "start_dt", "end_dt", "booking_id"])
    else:
        bk = bookings_prepared[faculty]
    blocks, summ = build_busy_blocks(logs, bk)
    all_driver_blocks[faculty] = blocks
    now_driver_blocks[faculty] = blocks[blocks["driver_name"].apply(lambda x: is_now_driver(x, faculty))].copy()
    block_summary.append({"faculty": faculty, **summ, "blocks_new_driver": len(now_driver_blocks[faculty])})
display(pd.DataFrame(block_summary))
for faculty, b in now_driver_blocks.items():
    display(b.groupby("block_type")["trip_hours"].agg(["count", "mean", "max"]).round(2).rename(columns={"count": "จำนวนบล็อก", "mean": "ชม.เฉลี่ย", "max": "ชม.สูงสุด"}))


  faculty  log_total  log_booked  log_walk_in  approved_bookings  approved_without_driver  blocks_new_driver
0     ICT        598           0          598                217                        0                411
1     SCI       1571           0         1571                135                        0                668
2    SEEN        598         168          430                168                        0                229

            จำนวนบล็อก  ชม.เฉลี่ย  ชม.สูงสุด
block_type                                  
booked             102      10.42       84.5
walk_in            309      26.44     6610.0

            จำนวนบล็อก  ชม.เฉลี่ย  ชม.สูงสุด
block_type                                  
booked               2       7.25        7.5
walk_in            666       3.45      111.0

            จำนวนบล็อก  ชม.เฉลี่ย  ชม.สูงสุด
block_type                                  
booked              70       5.32      36.42
walk_in            159       1.36       5.45

In [20]:
# ฟังก์ชันคำนวณชั่วโมงสะสมก่อนเวลาเริ่มคำขอ (ใช้กติกาวันปฏิบัติงาน 08:00 และการแปลงทริปหลายวันแบบเดียวกับ solver)
def get_driver_state_before_request(logs, driver_name, request_start):
    request_start = pd.Timestamp(request_start)
    history = logs[(logs["driver_name"] == driver_name) & (logs["end_dt"] <= request_start)].copy()
    if history.empty:
        return {"prior_daily_hours": 0.0, "prior_weekly_hours": 0.0,
                "rest_hours": float("inf"), "previous_trip_end": pd.NaT,
                "daily_span_hours": 0.0}

    prof = _hist_profile(history)
    d0 = duty_day_of(request_start)
    prior_daily_hours = prof.get(d0, 0.0)
    prior_weekly_hours = _window_sum(prof, d0)

    day_lo, day_hi = d0 + DUTY_DAY_START, duty_day_end(d0)
    same_day = history[(history["start_dt"] < day_hi) & (history["end_dt"] > day_lo)]
    daily_span_hours = ((same_day["end_dt"].max() - same_day["start_dt"].min()).total_seconds() / 3600) if not same_day.empty else 0.0

    previous_trip_end = history["end_dt"].max()
    rest_hours = (request_start - previous_trip_end).total_seconds() / 3600
    return {"prior_daily_hours": float(prior_daily_hours),
            "prior_weekly_hours": float(prior_weekly_hours),
            "rest_hours": float(rest_hours),
            "previous_trip_end": previous_trip_end,
            "daily_span_hours": float(daily_span_hours)}


In [21]:
# เพิ่มฟังก์ชันใช้ชั่วโมงจำลองใน solver และรายงาน
def get_effective_driver_state(resource, request_start):
    """คืนค่าประวัติจริง หรือแทนด้วยค่าจำลองของ test case"""

    state = get_driver_state_before_request(
        logs=resource["driver_logs"],
        driver_name=resource["driver"],
        request_start=request_start,
    )

    # ค่าจำลองมีไว้สำหรับ test case
    state["prior_daily_hours"] = resource.get(
        "test_daily_hours",
        state["prior_daily_hours"],
    )
    state["prior_weekly_hours"] = resource.get(
        "test_weekly_hours",
        state["prior_weekly_hours"],
    )
    state["rest_hours"] = resource.get(
        "test_rest_hours",
        state["rest_hours"],
    )

    return state

In [22]:
# ตรวจชั่วโมงก่อนจัดสรรคำขอจริง (ใช้ busy blocks ของคนขับใหม่)
sample_request_start = min(df["start_dt"].min() for df in bookings_prepared.values())

precheck_rows = []
for faculty, logs in now_driver_blocks.items():
    for driver_name in logs["driver_name"].unique():
        # ตรวจ ณ เวลาที่คนขับคนนี้เริ่มมีงาน + 30 วัน เพื่อให้มีประวัติให้ตรวจจริง
        probe = logs.loc[logs["driver_name"] == driver_name, "start_dt"].min() + pd.Timedelta(days=30)
        state = get_driver_state_before_request(logs, driver_name, max(sample_request_start, probe))
        precheck_rows.append({
            "faculty": faculty,
            "driver_name": driver_name,
            "request_start": max(sample_request_start, probe),
            **state,
            "daily_hours_ok": state["prior_daily_hours"] <= MAX_DAILY_HOURS,
            "weekly_hours_ok": state["prior_weekly_hours"] <= MAX_WEEKLY_HOURS,
            "rest_ok": state["rest_hours"] >= MIN_REST_HOURS,
        })

precheck = pd.DataFrame(precheck_rows)
display(precheck)


  faculty      driver_name       request_start  prior_daily_hours  prior_weekly_hours  rest_hours   previous_trip_end  daily_span_hours  daily_hours_ok  weekly_hours_ok  rest_ok
0     ICT   ICT_DRIVER_NOW 2025-12-30 17:30:00                4.0           30.666667         5.5 2025-12-30 12:00:00               4.0            True             True    False
1     SCI   SCI_DRIVER_NOW 2024-10-31 10:00:00                0.0           61.416667        19.0 2024-10-30 15:00:00               0.0            True             True     True
2    SEEN  SEEN_DRIVER_NOW 2025-12-15 12:50:00                1.3           29.483333         2.4 2025-12-15 10:26:00               1.3            True             True    False

In [23]:
# สรุปจำนวนคนขับที่ผ่านการตรวจชั่วโมง
if not precheck.empty:
    precheck["passes_all_prechecks"] = (
        precheck["daily_hours_ok"] &
        precheck["weekly_hours_ok"] &
        precheck["rest_ok"]
    )

    display(
        precheck.groupby("faculty")["passes_all_prechecks"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "ผ่าน", "count": "คนขับทั้งหมด"})
    )

         ผ่าน  คนขับทั้งหมด
faculty                    
ICT         0             1
SCI         1             1
SEEN        0             1

In [24]:
# ฟังก์ชันแปลง booking เป็นรูปแบบมาตรฐาน
def normalize_booking_row(row, faculty, request_id=None):
    start = row["start_dt"]
    end = row["end_dt"]

    if end <= start:
        raise ValueError("เวลาสิ้นสุดต้องอยู่หลังเวลาเริ่ม")

    return {
        "request_id": request_id or str(row.get("booking_id", "REQ")),
        "source_faculty": faculty,
        "start": start,
        "end": end,
        "passenger_count": int(row.get("passenger_count", 0)),
        "requested_vehicle_count": int(
            row.get("requested_vehicle_count", 1)
        ),
        "trip_scope": row.get("trip_scope_raw", None),
        "pickup_location": row.get("pickup_location_raw", None),
        "dropoff_location": row.get("dropoff_location_raw", None),
        "destination": row.get("destination_raw", None),
        "purpose": row.get("purpose_raw", None),
        "booking_status": row.get("booking_status_raw", None),
        "duration_hours": (end - start).total_seconds() / 3600,
        "is_multi_day": check_multi_day_trip(start, end)[0],
    }


In [25]:
def make_requests_from_csv(faculty, statuses=("pending",), window=None, n=None):
    """สร้างคำขอจาก booking CSV ตามสถานะ (ค่าเริ่มต้น pending) และช่วงเวลา window=(เริ่ม, สิ้นสุด)"""
    if faculty not in bookings_prepared:
        print(f"⚠️ ไม่พบข้อมูลคำขอจองของคณะ {faculty} ในระบบ (ข้ามการทำงาน)")
        return {}

    df = bookings_prepared[faculty].copy()
    df = df[df["booking_status_raw"].astype(str).str.lower().isin([s.lower() for s in statuses])]
    if window is not None:
        df = df[(df["start_dt"] >= window[0]) & (df["end_dt"] <= window[1])]
    df = df.sort_values("start_dt")
    if n:
        df = df.head(n)
    requests = {}
    for _, row in df.iterrows():
        req = normalize_booking_row(row, faculty)
        req["requested_vehicle_count"] = 1  # CSV ไม่มีคอลัมน์นี้
        requests[req["request_id"]] = req
    return requests

# ช่วงเวลาที่คนขับใหม่ของแต่ละคณะทำงานอยู่ (ใช้เป็นหน้าต่างจำลองการจัดสรร)
active_window = {
    fac: (b["start_dt"].min(), b["end_dt"].max())
    for fac, b in now_driver_blocks.items() if len(b)
}
sample_requests = {fac: make_requests_from_csv(fac, statuses=("pending",), window=active_window.get(fac)) for fac in FACULTIES}
for faculty, reqs in sample_requests.items():
    print(f"{faculty}: {len(reqs)} requests (pending ในช่วงที่คนขับปัจจุบันทำงาน {active_window.get(faculty)})")


ICT: 1 requests (pending ในช่วงที่คนขับปัจจุบันทำงาน (Timestamp('2025-11-30 17:30:00'), Timestamp('2026-10-10 18:00:00')))
SCI: 0 requests (pending ในช่วงที่คนขับปัจจุบันทำงาน (Timestamp('2024-10-01 10:00:00'), Timestamp('2025-09-30 16:30:00')))
SEEN: 9 requests (pending ในช่วงที่คนขับปัจจุบันทำงาน (Timestamp('2025-11-15 12:50:00'), Timestamp('2026-07-31 14:39:00')))


In [26]:
# สร้าง resource คนขับ–รถจากคนขับปัจจุบัน
def build_resources(now_driver_logs):
    resources = {}

    for faculty, logs in now_driver_logs.items():
        for driver_name, group in logs.groupby("driver_name"):
            plate_values = group["vehicle_plate"].dropna().astype(str).unique()
            plate = plate_values[0] if len(plate_values) else "UNKNOWN"

            resource_id = f"{faculty}_{driver_name}"

            resources[resource_id] = {
                "driver": driver_name,
                "vehicle": plate,
                "faculty": faculty,
                "available": True,
                "driver_logs": group.copy(),
            }

    return resources


resources = build_resources(now_driver_blocks)   # driver_logs ของ resource = busy blocks (walk-in + booking ที่อนุมัติ)

print("จำนวนคู่คนขับ–รถ:", len(resources))
for resource_id, resource in resources.items():
    print(resource_id, resource["faculty"], resource["vehicle"])


จำนวนคู่คนขับ–รถ: 3
ICT_ICT_DRIVER_NOW ICT 1 นช 3009 กรุงเทพมหานคร
SCI_SCI_DRIVER_NOW SCI นข 3948 พะเยา
SEEN_SEEN_DRIVER_NOW SEEN SIM-SEEN-01


In [27]:
# กำหนดนโยบายลำดับความสำคัญของคณะ
FACULTIES = ["ICT", "SCI", "SEEN"]
# โหมดมาตรฐาน: รถคณะใดให้คณะนั้นก่อน (1) ถ้าไม่ว่างจึงใช้รถคณะอื่นได้ (2) โดยไม่จัดลำดับผู้ให้ยืม
faculty_priority = {q: {o: (1 if q == o else 2) for o in FACULTIES} for q in FACULTIES}
UNASSIGNED_PENALTY = 1000

# [โหมดเสริม - ปิดไว้] ลำดับผู้ให้ยืม rank 2/3 ดูเซลล์ "โหมดเสริม" ด้านล่าง


In [28]:
# ฟังก์ชันตรวจ overlap และเวลาพัก
def requests_overlap(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]


def has_rest_conflict(logs, driver_name, request):
    history = logs[logs["driver_name"] == driver_name].copy()

    # เที่ยวที่จบก่อนคำขอใหม่
    prior = history[history["end_dt"] <= request["start"]]
    if not prior.empty:
        previous_end = prior["end_dt"].max()
        rest = (request["start"] - previous_end).total_seconds() / 3600
        if rest < MIN_REST_HOURS:
            return True

    # เที่ยวที่เริ่มหลังคำขอใหม่
    future = history[history["start_dt"] >= request["end"]]
    if not future.empty:
        next_start = future["start_dt"].min()
        rest = (next_start - request["end"]).total_seconds() / 3600
        if rest < MIN_REST_HOURS:
            return True

    return False

## เตรียม Test

In [29]:
import pulp
from itertools import combinations
import pandas as pd

def _history_conflicts(resource, start, end, min_rest_hours):
    """ชนกับ busy block คงที่ หรือเวลาพักถึงงานก่อน/หลังน้อยกว่า min_rest_hours"""
    logs = resource["driver_logs"]
    if logs is None or len(logs) == 0:
        return False
    for r in logs.itertuples(index=False):
        hs, he = pd.Timestamp(r.start_dt), pd.Timestamp(r.end_dt)
        if hs < end and start < he:
            return True
        if he <= start and (start - he).total_seconds() / 3600 < min_rest_hours:
            return True
        if end <= hs and (hs - end).total_seconds() / 3600 < min_rest_hours:
            return True
    return False

def _history_conflict_detail(resource, start, end, min_rest_hours):
    """คืนข้อความอธิบายว่าชนกับ busy block ใด หรือพักไม่ครบกับงานใด (None ถ้าไม่มีปัญหา)"""
    logs = resource["driver_logs"]
    if logs is None or len(logs) == 0:
        return None
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    for r in logs.itertuples(index=False):
        hs, he = pd.Timestamp(r.start_dt), pd.Timestamp(r.end_dt)
        kind = getattr(r, "block_type", "งานเดิม")
        if hs < end and start < he:
            return f"เวลาทับกับงานคงที่ ({kind}) {hs:%d/%m/%Y %H:%M}–{he:%d/%m/%Y %H:%M}"
        if he <= start and (start - he).total_seconds() / 3600 < min_rest_hours:
            return f"พักหลังงานคงที่ ({kind}) จบ {he:%d/%m %H:%M} เพียง {(start - he).total_seconds()/3600:.1f} ชม. ต่ำกว่า {min_rest_hours:.0f} ชม."
        if end <= hs and (hs - end).total_seconds() / 3600 < min_rest_hours:
            return f"พักก่อนงานคงที่ ({kind}) เริ่ม {hs:%d/%m %H:%M} เพียง {(hs - end).total_seconds()/3600:.1f} ชม. ต่ำกว่า {min_rest_hours:.0f} ชม."
    return None

def solve_assignment(resources, requests, capacity_by_resource=None, faculty_priority=faculty_priority,
                     max_daily_hours=MAX_DAILY_HOURS, max_weekly_hours=MAX_WEEKLY_HOURS,
                     min_rest_hours=MIN_REST_HOURS, objective_method="lexicographic"):
    if capacity_by_resource is None:
        capacity_by_resource = {i: 11 for i in resources}
    I, J = list(resources), list(requests)
    if not J:
        return {}, pd.DataFrame(), "NO_REQUESTS"

    # หน้าต่างเวลา = ช่วงเวลาจอง (โหมดมาตรฐานไม่บวกเวลาเดินทาง ETA)
    win = {j: (pd.Timestamp(requests[j]["start"]), pd.Timestamp(requests[j]["end"])) for j in J}
    # [OPTIONAL: ETA MODE] win = {j: (start, _busy_end(requests[j])) for j in J}   -> ดูเซลล์ "โหมดเสริม"

    # w[j,d] = ชั่วโมงทำงานของคำขอ j ในวันปฏิบัติงาน d (ทริปหลายวัน = 14 ชม./พัก 10 ชม. ต่อรอบ 24 ชม.)
    multi, seg = {}, {}
    for j in J:
        multi[j], seg[j] = daily_profile(*win[j])
    hist = {i: _hist_profile(resources[i]["driver_logs"]) for i in I}

    model = pulp.LpProblem("Vehicle_Driver_Assignment", pulp.LpMinimize)
    x = {(i, j): pulp.LpVariable(f"Assign_{i}_{j}", cat="Binary") for i in I for j in J}
    missing = {j: pulp.LpVariable(f"Missing_{j}", 0, requests[j]["requested_vehicle_count"], cat="Integer") for j in J}

    # กติกาความจุตามเว็บ: ขอรถ 1 คัน -> ผู้โดยสาร <= ความจุรถคันนั้น (11)
    #                      ขอรถ > 1 คัน -> ผู้โดยสารเกิน 11 ได้ แต่ต้องไม่เกินความจุรวม (ความจุสูงสุดต่อคัน x จำนวนรถที่ขอ)
    def capacity_ok(i, j):
        pax, veh = requests[j]["passenger_count"], requests[j]["requested_vehicle_count"]
        cap_i = capacity_by_resource.get(i, 11)
        if veh <= 1:
            return cap_i >= pax
        return max(capacity_by_resource.get(r, 11) for r in I) * veh >= pax

    # (C2) สิทธิ์ใช้ได้
    eligible = {}
    for i in I:
        for j in J:
            elapsed = (win[j][1] - win[j][0]).total_seconds() / 3600
            single_shift_ok = multi[j] or elapsed <= max_daily_hours + 1e-9   # ทริปวันเดียวต้องไม่เกิน 14 ชม. ต่อเนื่อง
            eligible[i, j] = int(
                resources[i].get("available", True)
                and capacity_ok(i, j)
                and resources[i].get("test_rest_hours", float("inf")) >= min_rest_hours
                and single_shift_ok
                and not _history_conflicts(resources[i], win[j][0], win[j][1], min_rest_hours))
            model += x[i, j] <= eligible[i, j]
    # (C1)
    for j in J:
        model += pulp.lpSum(x[i, j] for i in I) + missing[j] == requests[j]["requested_vehicle_count"]

    # (C3) ไม่ชนเวลา + พักอย่างน้อย min_rest_hours ระหว่างคำขอที่ให้ทรัพยากรเดียวกัน
    for i in I:
        for j, k in combinations(J, 2):
            (s1, e1), (s2, e2) = win[j], win[k]
            if s1 < e2 and s2 < e1:
                model += x[i, j] + x[i, k] <= 1
            elif e1 <= s2 and (s2 - e1).total_seconds() / 3600 < min_rest_hours:
                model += x[i, j] + x[i, k] <= 1
            elif e2 <= s1 and (s1 - e2).total_seconds() / 3600 < min_rest_hours:
                model += x[i, j] + x[i, k] <= 1

    # (C4) รายวัน
    all_days = sorted({d for s in seg.values() for d in s})
    for i in I:
        for d in all_days:
            h_d = resources[i].get("test_daily_hours", hist[i].get(d, 0.0))
            if h_d > max_daily_hours + 1e-9:
                for j in J:
                    if seg[j].get(d, 0) > 0:
                        model += x[i, j] == 0
            else:
                model += pulp.lpSum(seg[j].get(d, 0.0) * x[i, j] for j in J) <= max_daily_hours - h_d

    # (C5) รายสัปดาห์ แบบ rolling 7 วันปฏิบัติงาน (ตรวจทุกหน้าต่างที่แตะคำขอ)
    check_days = sorted({d + pd.Timedelta(days=k) for d in all_days for k in range(7)})
    for i in I:
        for end_day in check_days:
            lo = end_day - pd.Timedelta(days=6)
            h_w = resources[i].get("test_weekly_hours", _window_sum(hist[i], end_day))
            contrib = {j: sum(h for d, h in seg[j].items() if lo <= d <= end_day) for j in J}
            if not any(c > 0 for c in contrib.values()):
                continue
            if h_w > max_weekly_hours + 1e-9:
                for j in J:
                    if contrib[j] > 0:
                        model += x[i, j] == 0
            else:
                model += pulp.lpSum(contrib[j] * x[i, j] for j in J) <= max_weekly_hours - h_w

    # วัตถุประสงค์
    missing_expr = pulp.lpSum(missing.values())
    cross_expr = pulp.lpSum(x[i, j] for i in I for j in J if resources[i]["faculty"] != requests[j]["source_faculty"])
    priority_expr = pulp.lpSum(faculty_priority[requests[j]["source_faculty"]][resources[i]["faculty"]] * x[i, j] for i in I for j in J)
    solver = pulp.PULP_CBC_CMD(msg=False)

    def stage(expr):
        model.setObjective(expr); model.solve(solver)
        return pulp.LpStatus[model.status]

    if objective_method == "weighted":
        stage(UNASSIGNED_PENALTY * missing_expr + 10 * cross_expr + priority_expr)
    else:
        stages = [missing_expr, cross_expr, priority_expr]
        # [OPTIONAL: URGENT MODE] stages = [pulp.lpSum(missing[j] for j in J if requests[j].get("urgent"))] + stages
        for n, expr in enumerate(stages):
            if stage(expr) != "Optimal":
                break
            if n < len(stages) - 1:
                model += expr == int(round(pulp.value(expr) or 0))
    status = pulp.LpStatus[model.status]
    ok = status == "Optimal"
    summary = {"objective_method": objective_method,
               "missing_vehicles": int(round(pulp.value(missing_expr) or 0)) if ok else None,
               "cross_faculty_assignments": int(round(pulp.value(cross_expr) or 0)) if ok else None,
               "priority_score": float(pulp.value(priority_expr) or 0) if ok else None}
    out = {}
    for j in J:
        assigned = [i for i in I if (pulp.value(x[i, j]) or 0) > 0.5]
        out[j] = {"assigned_resources": assigned, "assigned_count": len(assigned),
                  "requested_count": requests[j]["requested_vehicle_count"],
                  "missing_count": int(round(pulp.value(missing[j]) or 0)) if ok else requests[j]["requested_vehicle_count"],
                  "is_multi_day": multi[j], "counted_work_hours": round(sum(seg[j].values()), 2),
                  "status": status, **summary}
    audit = pd.DataFrame([{"request_id": j, "resource_id": i, "vehicle_faculty": resources[i]["faculty"],
                           "request_faculty": requests[j]["source_faculty"], "eligible": bool(eligible[i, j]),
                           "assigned": i in out[j]["assigned_resources"]} for j in J for i in I])
    return out, audit, status


In [30]:
# ---- คำขอด่วนข้ามหน่วยงาน: ไม่ส่งคำขอเข้าระบบ แต่โทรหาผู้ดูแลรถตู้ ----
# ผู้ดูแลรถตู้ตัดสินใจเอง ถ้าอนุมัติ ให้บันทึกเป็น busy block คงที่ของรถนั้น (solver จะเคารพต่อไป)
def add_manager_approved_block(resource, start, end, note="อนุมัติทางโทรศัพท์"):
    row = {"driver_name": resource["driver"], "vehicle_plate": resource.get("vehicle"),
           "start_dt": pd.Timestamp(start), "end_dt": pd.Timestamp(end), "block_type": "manager_approved", "ref_id": note}
    row["trip_hours"] = (row["end_dt"] - row["start_dt"]).total_seconds() / 3600
    logs = resource["driver_logs"]
    resource["driver_logs"] = pd.concat([logs, pd.DataFrame([row])], ignore_index=True) if len(logs) else pd.DataFrame([row])
    return resource

def manager_escalation_notes(requests, results, resources):
    """โหมดมาตรฐานไม่ให้ flag urgent มีอำนาจเหนือคำขอปกติ -> แจ้งให้ผู้ขอติดต่อผู้ดูแลรถตู้โดยตรง"""
    notes = []
    for j, r in requests.items():
        if not r.get("urgent"):
            continue
        res = results.get(j, {})
        cross = [i for i in res.get("assigned_resources", []) if resources[i]["faculty"] != r["source_faculty"]]
        if res.get("missing_count", 0) > 0:
            notes.append(f"{j}: ด่วนแต่ระบบจัดรถไม่ได้ครบ -> โทรหาผู้ดูแลรถตู้คณะอื่นโดยตรง")
        elif cross:
            notes.append(f"{j}: ด่วนและใช้รถข้ามคณะ -> ควรยืนยันกับผู้ดูแลรถตู้เจ้าของรถทางโทรศัพท์")
    return notes


In [31]:
# รันทดลองกับคำขอ pending จาก CSV เทียบกับงานคงที่ (walk-in + booking ที่อนุมัติแล้ว)
csv_requests = {}
for faculty in FACULTIES:
    csv_requests.update(sample_requests[faculty])

# ป้องกัน request_id ซ้ำ โดยเติมคณะนำหน้า
csv_requests_unique = {}
for req_id, req in csv_requests.items():
    new_id = f"{req['source_faculty']}_{req_id}"
    req["request_id"] = new_id
    csv_requests_unique[new_id] = req

csv_results, csv_audit, csv_status = solve_assignment(
    resources=resources,
    requests=csv_requests_unique,
    capacity_by_resource={i: 11 for i in resources},
)

print("Solver status:", csv_status)
csv_table = pd.DataFrame([
    {
        "request_id": j,
        "faculty": csv_requests_unique[j]["source_faculty"],
        "start": csv_requests_unique[j]["start"],
        "end": csv_requests_unique[j]["end"],
        "หลายวัน": result["is_multi_day"],
        "ชม.ทำงานที่นับ": result["counted_work_hours"],
        "requested": result["requested_count"],
        "assigned": result["assigned_count"],
        "missing": result["missing_count"],
        "resources": result["assigned_resources"],
    }
    for j, result in csv_results.items()
])
display(csv_table)
if len(csv_table):
    print(f"จัดได้ {int(csv_table['assigned'].sum())} จาก {int(csv_table['requested'].sum())} คำขอ pending")


Solver status: Optimal
จัดได้ 9 จาก 10 คำขอ pending


          request_id faculty               start                 end  หลายวัน  ชม.ทำงานที่นับ  requested  assigned  missing               resources
0       ICT_BK000447     ICT 2026-06-27 13:00:00 2026-06-27 22:00:00    False            9.00          1         1        0    [SCI_SCI_DRIVER_NOW]
1  SEEN_SEEN_BK_0153    SEEN 2025-12-04 06:00:00 2025-12-04 10:12:00    False            4.20          1         1        0  [SEEN_SEEN_DRIVER_NOW]
2  SEEN_SEEN_BK_0172    SEEN 2026-01-23 09:30:00 2026-01-23 14:17:00    False            4.78          1         1        0    [SCI_SCI_DRIVER_NOW]
3  SEEN_SEEN_BK_0186    SEEN 2026-02-18 09:15:00 2026-02-18 16:53:00    False            7.63          1         1        0    [SCI_SCI_DRIVER_NOW]
4  SEEN_SEEN_BK_0200    SEEN 2026-03-19 11:00:00 2026-03-19 12:41:00    False            1.68          1         1        0    [SCI_SCI_DRIVER_NOW]
5  SEEN_SEEN_BK_0205    SEEN 2026-03-29 10:00:00 2026-03-29 14:42:00    False            4.70          1        

In [32]:
# แสดง audit ผู้สมัครของการจัดสรร
display(
    csv_audit.sort_values(
        ["request_id", "eligible", "vehicle_faculty"],
        ascending=[True, False, True]
    )
)

           request_id           resource_id vehicle_faculty request_faculty  eligible  assigned
1        ICT_BK000447    SCI_SCI_DRIVER_NOW             SCI             ICT      True      True
0        ICT_BK000447    ICT_ICT_DRIVER_NOW             ICT             ICT     False     False
2        ICT_BK000447  SEEN_SEEN_DRIVER_NOW            SEEN             ICT     False     False
4   SEEN_SEEN_BK_0153    SCI_SCI_DRIVER_NOW             SCI            SEEN      True     False
5   SEEN_SEEN_BK_0153  SEEN_SEEN_DRIVER_NOW            SEEN            SEEN      True      True
3   SEEN_SEEN_BK_0153    ICT_ICT_DRIVER_NOW             ICT            SEEN     False     False
7   SEEN_SEEN_BK_0172    SCI_SCI_DRIVER_NOW             SCI            SEEN      True      True
6   SEEN_SEEN_BK_0172    ICT_ICT_DRIVER_NOW             ICT            SEEN     False     False
8   SEEN_SEEN_BK_0172  SEEN_SEEN_DRIVER_NOW            SEEN            SEEN     False     False
10  SEEN_SEEN_BK_0186    SCI_SCI_DRIVER_

## Set คำขอจอง

In [33]:
# สร้างข้อมูลคำขอและ resource สำหรับ test case
TEST_DATE = pd.Timestamp("2026-09-29 08:00:00")

def make_test_request(request_id, faculty="ICT", start=None, duration_hours=2,
                      passengers=5, vehicles=1, **extra):
    start = pd.Timestamp(start) if start is not None else TEST_DATE
    end = start + pd.Timedelta(hours=duration_hours)
    req = {
        "request_id": request_id, "source_faculty": faculty,
        "start": start, "end": end, "duration_hours": float(duration_hours),
        "passenger_count": int(passengers), "requested_vehicle_count": int(vehicles),
        "trip_scope": "IN_PROVINCE", "pickup_location": "มหาวิทยาลัยพะเยา",
        "dropoff_location": "จุดหมายทดสอบ", "destination": "จุดหมายทดสอบ",
        "purpose": "ทดสอบระบบ", "booking_status": "pending",
    }
    req.update(extra)
    return req


In [34]:
def make_test_resources(
    counts=None,
    default_capacity=11,
    available_ids=None,
    current_hours=None,
    rest_hours=None,
    weekly_hours=None,
):
    counts = counts or {"ICT": 1, "SCI": 1, "SEEN": 1}
    current_hours = current_hours or {}
    rest_hours = rest_hours or {}
    weekly_hours = weekly_hours or {}

    out = {}
    idx = 1

    # Define base names for faculties
    driver_name_templates = {
        "ICT": "driver_ict",
        "SCI": "driver_sci",
        "SEEN": "driver_seen"
    }

    # Counter for each faculty to ensure unique driver names if count > 1
    faculty_driver_counts = {faculty: 0 for faculty in counts.keys()}

    for faculty, count in counts.items():
        for _ in range(count):
            rid = f"T{idx:02d}"
            idx += 1
            faculty_driver_counts[faculty] += 1

            base_driver_name = driver_name_templates.get(faculty, f"TEST_DRIVER_{faculty}")
            if count > 1:
                driver_name = f"{base_driver_name}_{faculty_driver_counts[faculty]}"
            else:
                driver_name = base_driver_name

            out[rid] = {
                "driver": driver_name,
                "vehicle": f"TEST-VAN-{rid}",
                "faculty": faculty,
                "available": (
                    True if available_ids is None
                    else rid in available_ids
                ),
                "driver_logs": pd.DataFrame(columns=[
                    "driver_name",
                    "start_dt",
                    "end_dt",
                    "trip_hours",
                ]),
                "test_capacity": default_capacity,
                "test_daily_hours": current_hours.get(rid, 0.0),
                "test_weekly_hours": weekly_hours.get(rid, 0.0),
                "test_rest_hours": rest_hours.get(rid, 70.0),
            }

    return out

In [35]:
# สร้างรายการ test case
TEST_CASES = []

def add_case(group, case_id, title, requests, resources, capacity=None, note=""):
    TEST_CASES.append({
        "group": group,
        "case_id": case_id,
        "title": title,
        "requests": requests,
        "resources": resources,
        "capacity": capacity,
        "note": note,
    })


# 1. จำนวนรถที่พร้อมให้จัดสรร: 3
add_case("1 รถพร้อม", "TC01", "มีรถพร้อม 1 คัน",
         {"R1": make_test_request("R1", vehicles=1)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("1 รถพร้อม", "TC02", "มีรถพร้อม 2 คัน แต่ต้องการ 1 คัน",
         {"R1": make_test_request("R1", vehicles=1)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))
add_case("1 รถพร้อม", "TC03", "ไม่มีรถพร้อมเลย",
         {"R1": make_test_request("R1", vehicles=1)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0},
                             available_ids=[]))


# 2. คำขอพร้อมกันและการเรียงลำดับ: 4
add_case("2 คำขอพร้อมกัน", "TC04", "สองคำขอเวลาเดียวกัน รถมี 2 คัน",
         {
             "R1": make_test_request("R1", faculty="ICT", vehicles=1),
             "R2": make_test_request("R2", faculty="SCI", vehicles=1),
         },
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))
add_case("2 คำขอพร้อมกัน", "TC05", "สองคำขอเวลาเดียวกัน รถมีคันเดียว",
         {
             "R1": make_test_request("R1", faculty="ICT", vehicles=1),
             "R2": make_test_request("R2", faculty="SCI", vehicles=1),
         },
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("2 คำขอพร้อมกัน", "TC06", "คำขอหนึ่งจบก่อนอีกคำขอเริ่ม",
         {
             "R1": make_test_request("R1", start=TEST_DATE, duration_hours=2),
             "R2": make_test_request("R2", start=TEST_DATE + pd.Timedelta(hours=12),
                                     duration_hours=2),
         },
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("2 คำขอพร้อมกัน", "TC07", "คำขอเวลาเดียวกันแต่คนละคณะเจ้าของรถ",
         {
             "R1": make_test_request("R1", faculty="ICT"),
             "R2": make_test_request("R2", faculty="SCI"),
         },
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))


# 3. ความจุรถและจำนวนรถที่ต้องการ: 4
add_case("3 ความจุและจำนวนรถ", "TC08", "ผู้โดยสารไม่เกินความจุ",
         {"R1": make_test_request("R1", passengers=11)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("3 ความจุและจำนวนรถ", "TC09", "[Robustness] ผู้โดยสาร 12 คน/รถ 1 คัน (เว็บบล็อก -> INVALID_INPUT)",
         {"R1": make_test_request("R1", passengers=12)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("3 ความจุและจำนวนรถ", "TC10", "ขอ 2 คันและมี 2 คัน",
         {"R1": make_test_request("R1", vehicles=2)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))
add_case("3 ความจุและจำนวนรถ", "TC11", "ขอ 3 คันแต่มีรถ 2 คัน",
         {"R1": make_test_request("R1", vehicles=3)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))


# 4. เวลางานและการข้ามวัน: 5
add_case("4 เวลาและข้ามวัน", "TC12", "งานช่วงเช้าจบในวันเดียว",
         {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=3)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("4 เวลาและข้ามวัน", "TC13", "งานข้ามเที่ยงคืน",
         {"R1": make_test_request("R1", start="2026-09-29 22:00", duration_hours=4)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("4 เวลาและข้ามวัน", "TC14", "งานเริ่มและจบวันถัดไป",
         {"R1": make_test_request("R1", start="2026-09-30 00:30", duration_hours=3)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("4 เวลาและข้ามวัน", "TC15", "งานระยะเวลา 14 ชั่วโมงพอดี",
         {"R1": make_test_request("R1", duration_hours=14)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("4 เวลาและข้ามวัน", "TC16", "งานยาวเกิน 14 ชั่วโมง",
         {"R1": make_test_request("R1", duration_hours=15)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))


# 5. ภาระงานและเวลาพักคนขับ: 5
r17 = make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0},
                          current_hours={"T01": 12})
add_case("5 ภาระงานและพัก", "TC17", "ชั่วโมงเดิม 12 บวกงาน 2 ชั่วโมง เท่ากับ 14",
         {"R1": make_test_request("R1", duration_hours=2)}, r17)
r18 = make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0},
                          current_hours={"T01": 13})
add_case("5 ภาระงานและพัก", "TC18", "ชั่วโมงเดิม 13 บวกงาน 2 ชั่วโมง เกิน 14",
         {"R1": make_test_request("R1", duration_hours=2)}, r18)
r19 = make_test_resources(
    {"ICT": 1, "SCI": 0, "SEEN": 0},
    weekly_hours={"T01": 70})
add_case("5 ภาระงานและพัก", "TC19", "ชั่วโมงสะสมสัปดาห์ 70 บวกงาน 2 ชั่วโมง",
         {"R1": make_test_request("R1", duration_hours=2)}, r19)
r20 = make_test_resources(
    {"ICT": 1, "SCI": 0, "SEEN": 0},
    weekly_hours={"T01": 71})
add_case("5 ภาระงานและพัก", "TC20", "ชั่วโมงสะสมสัปดาห์ 71 บวกงาน 2 ชั่วโมง",
         {"R1": make_test_request("R1", duration_hours=2)}, r20)
r21 = make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0},
                          rest_hours={"T01": 9})
add_case("5 ภาระงานและพัก", "TC21", "พัก 9 ชั่วโมง ไม่ผ่านเกณฑ์",
         {"R1": make_test_request("R1")}, r21)


# 6. สถานะคำขอและการยกเลิก: 4
# ไม่นำมารันตามที่ผู้ใช้ระบุว่าไม่รวมกลุ่มนี้


# 7. สิทธิ์รถเจ้าของคณะและรถส่วนกลาง: 4
add_case("7 สิทธิ์รถเจ้าของคณะ", "TC22", "ใช้รถเจ้าของคณะก่อน",
         {"R1": make_test_request("R1", faculty="ICT")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))
add_case("7 สิทธิ์รถเจ้าของคณะ", "TC23", "รถเจ้าของคณะไม่ว่าง จึงใช้รถคณะอื่น",
         {"R1": make_test_request("R1", faculty="ICT")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0},
                             available_ids=["T02"]))
add_case("7 สิทธิ์รถเจ้าของคณะ", "TC24", "รถคณะอื่นช่วยเติมคำขอหลายคัน",
         {"R1": make_test_request("R1", faculty="ICT", vehicles=2)},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))
add_case("7 สิทธิ์รถเจ้าของคณะ", "TC25", "คำขอ SEEN ใช้รถ SEEN ก่อน",
         {"R1": make_test_request("R1", faculty="SEEN")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1}))


# 8. งานเร่งด่วนและแจ้งรถไม่ครบ: 4
add_case("8 งานเร่งด่วน", "TC26", "งานเร่งด่วนมีรถเพียงพอ",
         {"R1": make_test_request("R1", vehicles=1, urgent=True)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("8 งานเร่งด่วน", "TC27", "งานเร่งด่วนขอ 2 คัน มีรถ 1 คัน",
         {"R1": make_test_request("R1", vehicles=2, urgent=True)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("8 งานเร่งด่วน", "TC28", "งานเร่งด่วนชนกับอีกคำขอ",
         {
             "R1": make_test_request("R1", faculty="ICT", urgent=True),
             "R2": make_test_request("R2", faculty="SCI"),
         },
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))
add_case("8 งานเร่งด่วน", "TC29", "งานเร่งด่วนไม่มีรถที่ผ่านเงื่อนไข",
         {"R1": make_test_request("R1", urgent=True)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0},
                             available_ids=[]))


# 9. ระยะทางและประมาณเวลาเดินทาง: 4
add_case("9 ระยะทางและเวลาเดินทาง", "TC30", "มีระยะทางสั้นที่ระบุ",
         {"R1": make_test_request("R1", distance_km=5, estimated_travel_hours=0.5)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("9 ระยะทางและเวลาเดินทาง", "TC31", "มีระยะทางไกลและเวลาเดินทางสูง",
         {"R1": make_test_request("R1", distance_km=250, estimated_travel_hours=5)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("9 ระยะทางและเวลาเดินทาง", "TC32", "ไม่มีข้อมูลระยะทาง",
         {"R1": make_test_request("R1", distance_km=None,
                                  estimated_travel_hours=None)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("9 ระยะทางและเวลาเดินทาง", "TC33", "ระยะทางเป็นศูนย์",
         {"R1": make_test_request("R1", distance_km=0,
                                  estimated_travel_hours=0)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))


# 10. ข้อมูลผิดปกติหรือไม่ครบ: 5
add_case("10 ข้อมูลผิดปกติ", "TC34", "จำนวนผู้โดยสารเป็นศูนย์",
         {"R1": make_test_request("R1", passengers=0)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("10 ข้อมูลผิดปกติ", "TC35", "จำนวนผู้โดยสารติดลบ",
         {"R1": make_test_request("R1", passengers=-1)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("10 ข้อมูลผิดปกติ", "TC36", "จำนวนรถที่ขอเป็นศูนย์",
         {"R1": make_test_request("R1", vehicles=0)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("10 ข้อมูลผิดปกติ", "TC37", "ปลายทางว่าง",
         {"R1": make_test_request("R1", destination=None)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("10 ข้อมูลผิดปกติ", "TC38", "เวลาสิ้นสุดก่อนเวลาเริ่ม",
         {"R1": {
             **make_test_request("R1"),
             "end": TEST_DATE - pd.Timedelta(hours=1),
             "duration_hours": -1,
         }},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))


# 11. การทำรายการซ้ำและการเปลี่ยนแปลงคำขอ: 3
add_case("11 รายการซ้ำ/แก้ไข", "TC39", "มี request ID ซ้ำ",
         {
             "R1": make_test_request("R1"),
             "R1_DUP": make_test_request("R1", faculty="SCI"),
         },
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 0}))
add_case("11 รายการซ้ำ/แก้ไข", "TC40", "แก้เวลาแล้วไม่ชนคำขอเดิม",
         {
             "R1": make_test_request("R1", start=TEST_DATE),
             "R2": make_test_request("R2", start=TEST_DATE + pd.Timedelta(hours=12)),
         },
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("11 รายการซ้ำ/แก้ไข", "TC41", "แก้เวลาจนคำขอชนกัน",
         {
             "R1": make_test_request("R1", start=TEST_DATE),
             "R2": make_test_request("R2", start=TEST_DATE + pd.Timedelta(hours=1)),
         },
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))


# 12. ผลลัพธ์และตรวจสอบย้อนหลัง: 3
add_case("12 ผลลัพธ์และ audit", "TC42", "เก็บผลจัดสรรพร้อมจำนวนรถขาด",
         {"R1": make_test_request("R1", vehicles=2)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("12 ผลลัพธ์และ audit", "TC43", "[Robustness] ผู้โดยสาร 12 คน/รถ 1 คัน ถูกปฏิเสธที่ validation",
         {"R1": make_test_request("R1", passengers=12)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
add_case("12 ผลลัพธ์และ audit", "TC44", "เก็บคณะเจ้าของรถและคณะผู้ร้องขอ",
         {"R1": make_test_request("R1", faculty="SEEN")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1}))


print("จำนวน test cases:", len(TEST_CASES))
assert len(TEST_CASES) == 44

จำนวน test cases: 44


In [36]:
# ============================================================
# เพิ่ม Test Cases TC45–TC54
# (TC45-54 คงเดิม; TC55 เป็นต้นไปปรับตามกติกา v5 ด้านล่าง)
# ============================================================

def make_trip_test_request(
    request_id,
    faculty,
    start,
    calendar_days,
    trip_scope,
    passengers=5,
    vehicles=1,
    workload_hours=8,
    destination="สถานที่ทดสอบ",
):
    """
    สร้างคำขอทดสอบทริปหลายวัน
    - calendar_days == 1 : เวลาจอง = start .. start + workload_hours (เวลาจอง = เวลาทำงาน)
    - calendar_days  > 1 : จบเวลา 20:00 น. ของวันสุดท้าย; ชั่วโมงทำงานคำนวณโดย solver ด้วยกติกา 14/10 (workload_hours ไม่ถูกใช้)
    """
    start = pd.Timestamp(start)

    if calendar_days == 1:
        end = start + pd.Timedelta(hours=workload_hours)
    else:
        end = start.normalize() + pd.Timedelta(days=calendar_days - 1, hours=20)

    request = make_test_request(
        request_id=request_id,
        faculty=faculty,
        start=start,
        duration_hours=(end - start).total_seconds() / 3600,
        passengers=passengers,
        vehicles=vehicles,
        trip_scope=trip_scope,
        destination=destination,
    )
    request["end"] = end
    request["calendar_days"] = int(calendar_days)
    return request


# ------------------------------------------------------------
# กลุ่ม 13: รถพร้อม 3 คัน และมี 3 คำขอ
# ------------------------------------------------------------

# TC45: คณะเดียวกันส่ง 3 คำขอเวลาเดียวกัน มีรถคณะเดียวกัน 3 คัน
add_case(
    "13 รถ 3 คัน/คำขอ 3 รายการ",
    "TC45",
    "คณะ ICT มี 3 คำขอพร้อมกันและมีรถพร้อม 3 คัน",
    {
        "R1": make_test_request(
            "R1", faculty="ICT", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
        "R2": make_test_request(
            "R2", faculty="ICT", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
        "R3": make_test_request(
            "R3", faculty="ICT", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
    },
    make_test_resources({"ICT": 3, "SCI": 0, "SEEN": 0})
)

# TC46: สามคณะส่งคำขอพร้อมกัน มีรถพร้อมคณะละ 1 คัน
add_case(
    "13 รถ 3 คัน/คำขอ 3 รายการ",
    "TC46",
    "ICT, SCI และ SEEN มีคำขอเวลาเดียวกันและมีรถพร้อม 3 คัน",
    {
        "R1": make_test_request(
            "R1", faculty="ICT", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
        "R2": make_test_request(
            "R2", faculty="SCI", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
        "R3": make_test_request(
            "R3", faculty="SEEN", vehicles=1,
            start="2026-09-29 08:00", duration_hours=3
        ),
    },
    make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1})
)


# ------------------------------------------------------------
# กลุ่ม 14: คณะเดียวกันมี 2 คำขอช่วงเวลาเดียวกัน/คาบเกี่ยวกัน
# ใช้รถ ICT 2 คัน เพื่อทดสอบว่าระบบไม่ใช้รถคันเดียวซ้ำกัน
# ------------------------------------------------------------

# TC47: เวลาใกล้กัน เดินทางในจังหวัดทั้งคู่
add_case(
    "14 คณะเดียวกัน/ช่วงเวลาใกล้กัน",
    "TC47",
    "ICT สองคำขอเวลาใกล้กัน เดินทางในจังหวัดปลายทางเดียวกัน",
    {
        "R1": make_test_request(
            "R1", faculty="ICT",
            start="2026-09-29 08:00", duration_hours=4,
            trip_scope="IN_PROVINCE",
            destination="อำเภอเมืองพะเยา"
        ),
        "R2": make_test_request(
            "R2", faculty="ICT",
            start="2026-09-29 09:00", duration_hours=4,
            trip_scope="IN_PROVINCE",
            destination="อำเภอเมืองพะเยา"
        ),
    },
    make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0})
)

# TC48: คำขอแรกไปต่างจังหวัด 3 วัน คำขอที่สองไปต่างจังหวัด 1 วัน
add_case(
    "14 คณะเดียวกัน/ช่วงเวลาใกล้กัน",
    "TC48",
    "ICT ขอรถไปต่างจังหวัด 3 วัน และอีกคำขอไปต่างจังหวัด 1 วัน",
    {
        "R1": make_trip_test_request(
            "R1", "ICT", "2026-09-29 08:00", 3,
            "OUT_PROVINCE", workload_hours=8,
            destination="เชียงใหม่"
        ),
        "R2": make_trip_test_request(
            "R2", "ICT", "2026-09-29 09:00", 1,
            "OUT_PROVINCE", workload_hours=4,
            destination="เชียงราย"
        ),
    },
    make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0})
)

# TC49: คำขอแรกไปต่างจังหวัด 3 วัน คำขอที่สองในจังหวัด 1 วัน
add_case(
    "14 คณะเดียวกัน/ช่วงเวลาใกล้กัน",
    "TC49",
    "ICT ขอรถไปต่างจังหวัด 3 วัน และอีกคำขอเดินทางในจังหวัด 1 วัน",
    {
        "R1": make_trip_test_request(
            "R1", "ICT", "2026-09-29 08:00", 3,
            "OUT_PROVINCE", workload_hours=8,
            destination="เชียงใหม่"
        ),
        "R2": make_trip_test_request(
            "R2", "ICT", "2026-09-29 09:00", 1,
            "IN_PROVINCE", workload_hours=4,
            destination="อำเภอเมืองพะเยา"
        ),
    },
    make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0})
)

# TC50: คำขอแรกเดินทางในจังหวัด 2 วัน คำขอที่สองในจังหวัด 1 วัน
add_case(
    "14 คณะเดียวกัน/ช่วงเวลาใกล้กัน",
    "TC50",
    "ICT ขอรถในจังหวัด 2 วัน และอีกคำขอในจังหวัด 1 วัน",
    {
        "R1": make_trip_test_request(
            "R1", "ICT", "2026-09-29 08:00", 2,
            "IN_PROVINCE", workload_hours=8,
            destination="อำเภอเชียงคำ"
        ),
        "R2": make_trip_test_request(
            "R2", "ICT", "2026-09-29 09:00", 1,
            "IN_PROVINCE", workload_hours=4,
            destination="อำเภอเมืองพะเยา"
        ),
    },
    make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0})
)

# TC51: คำขอแรกเดินทางในจังหวัด 2 วัน คำขอที่สองไปต่างจังหวัด 1 วัน
add_case(
    "14 คณะเดียวกัน/ช่วงเวลาใกล้กัน",
    "TC51",
    "ICT ขอรถในจังหวัด 2 วัน และอีกคำขอไปต่างจังหวัด 1 วัน",
    {
        "R1": make_trip_test_request(
            "R1", "ICT", "2026-09-29 08:00", 2,
            "IN_PROVINCE", workload_hours=8,
            destination="อำเภอเชียงคำ"
        ),
        "R2": make_trip_test_request(
            "R2", "ICT", "2026-09-29 09:00", 1,
            "OUT_PROVINCE", workload_hours=4,
            destination="เชียงราย"
        ),
    },
    make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0})
)


# ------------------------------------------------------------
# กลุ่ม 15: คณะเดียวกันมี 2 คำขอ แต่เดินทางคนละช่วงเวลา
# ใช้รถ ICT คันเดียว เพื่อทดสอบว่ารถเดิมกลับมารับงานถัดไปได้
# ------------------------------------------------------------

# TC52: เดินทางวันที่ 5 ส.ค. และ 7 ส.ค. ห่างกัน 2 วัน
add_case(
    "15 คณะเดียวกัน/คนละช่วงเวลา",
    "TC52",
    "ICT จองวันที่ 5 ส.ค. และ 7 ส.ค. ห่างกัน 2 วัน",
    {
        "R1": make_test_request(
            "R1", faculty="ICT",
            start="2026-08-05 08:00", duration_hours=6,
            trip_scope="OUT_PROVINCE"
        ),
        "R2": make_test_request(
            "R2", faculty="ICT",
            start="2026-08-07 08:00", duration_hours=6,
            trip_scope="IN_PROVINCE"
        ),
    },
    make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0})
)

# TC53: เดินทางวันที่ 9 ก.ค. และ 12 ก.ค. ห่างกัน 3 วัน
add_case(
    "15 คณะเดียวกัน/คนละช่วงเวลา",
    "TC53",
    "ICT จองวันที่ 9 ก.ค. และ 12 ก.ค. ห่างกัน 3 วัน",
    {
        "R1": make_test_request(
            "R1", faculty="ICT",
            start="2026-07-09 08:00", duration_hours=6,
            trip_scope="OUT_PROVINCE"
        ),
        "R2": make_test_request(
            "R2", faculty="ICT",
            start="2026-07-12 08:00", duration_hours=6,
            trip_scope="IN_PROVINCE"
        ),
    },
    make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0})
)

# TC54: เดินทางวันที่ 7 พ.ค. และ 14 พ.ค. ห่างกัน 1 สัปดาห์
add_case(
    "15 คณะเดียวกัน/คนละช่วงเวลา",
    "TC54",
    "ICT จองวันที่ 7 พ.ค. และ 14 พ.ค. ห่างกัน 1 สัปดาห์",
    {
        "R1": make_test_request(
            "R1", faculty="ICT",
            start="2026-05-07 08:00", duration_hours=6,
            trip_scope="OUT_PROVINCE"
        ),
        "R2": make_test_request(
            "R2", faculty="ICT",
            start="2026-05-14 08:00", duration_hours=6,
            trip_scope="IN_PROVINCE"
        ),
    },
    make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0})
)


# ตรวจจำนวนกรณีทั้งหมดหลังเพิ่ม
print("จำนวน test cases:", len(TEST_CASES))
assert len(TEST_CASES) == 54

# ============================================================
# TC55-TC65: กติกา v5 (urgent ไม่มีสิทธิ์เหนือคำขอปกติ, busy blocks, ทริปหลายวัน 14/10)
# ============================================================
_one = lambda **k: make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}, **k)

def _with_blocks(resources, rid, blocks):
    """ใส่ busy block คงที่ (walk-in) ให้ resource: blocks = [(start, end), ...]"""
    rows = [{"driver_name": resources[rid]["driver"], "start_dt": pd.Timestamp(s), "end_dt": pd.Timestamp(e),
             "trip_hours": (pd.Timestamp(e) - pd.Timestamp(s)).total_seconds() / 3600, "block_type": "walk_in"} for s, e in blocks]
    resources[rid]["driver_logs"] = pd.DataFrame(rows)
    return resources

add_case("16 คำขอด่วนข้ามหน่วยงาน", "TC55", "คำขอด่วนของ SCI ไม่ preempt คำขอปกติของ ICT (ด่วนข้ามคณะต้องโทรหาผู้ดูแล ไม่ผ่านระบบ)",
         {"R1": make_test_request("R1", faculty="ICT"), "R2": make_test_request("R2", faculty="SCI", urgent=True)}, _one(),
         note="โหมดมาตรฐานไม่ใช้ flag urgent; ถ้าต้องการให้ด่วนมีสิทธิ์เหนือ ดูโหมดเสริม")
add_case("17 ช่วงว่างระหว่างงาน", "TC56", "ควบคุม: คำขอที่สองเริ่มหลังคำขอแรกจบ 13 ชม.",
         {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=4),
          "R2": make_test_request("R2", start="2026-09-30 01:00", duration_hours=2)}, _one())
add_case("17 ช่วงว่างระหว่างงาน", "TC57", "walk-in (ธุระด่วนในคณะ) ทับเวลาคำขอ",
         {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=4)},
         _with_blocks(_one(), "T01", [("2026-09-29 09:00", "2026-09-29 11:00")]))
add_case("17 ช่วงว่างระหว่างงาน", "TC58", "walk-in จบก่อนคำขอเริ่มเพียง 6 ชม. (พักไม่ถึง 10 ชม.)",
         {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=2)},
         _with_blocks(_one(), "T01", [("2026-09-29 00:00", "2026-09-29 02:00")]))
add_case("18 ผู้ให้ยืม", "TC59", "รถ SCI ไม่ว่าง; SEEN และ ICT ว่าง (ไม่จัดลำดับผู้ให้ยืม เลือกคันใดก็ได้)",
         {"R1": make_test_request("R1", faculty="SCI")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1}, available_ids=["T01", "T03"]))
add_case("18 ผู้ให้ยืม", "TC60", "มีเพียงรถ ICT ว่าง; ยืมข้ามคณะดีกว่าจัดไม่ได้",
         {"R1": make_test_request("R1", faculty="SCI")},
         make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1}, available_ids=["T01"]))
add_case("19 ทริปหลายวัน", "TC61", "ทริป 3 วัน กับคำขอ 1 วันที่อยู่ในช่วงทริป (รถคันเดียว)",
         {"R1": make_trip_test_request("R1", "ICT", "2026-09-29 08:00", 3, "OUT_PROVINCE"),
          "R2": make_trip_test_request("R2", "ICT", "2026-09-30 09:00", 1, "IN_PROVINCE", workload_hours=4)}, _one())
add_case("19 ทริปหลายวัน", "TC62", "ทริป 5 วัน: ชม.ทำงาน 14x4+12 = 68 ไม่เกิน 72",
         {"R1": make_trip_test_request("R1", "ICT", "2026-09-29 08:00", 5, "OUT_PROVINCE")}, _one())
add_case("19 ทริปหลายวัน", "TC63", "ทริป 6 วัน: ชม.ทำงาน 14x5+12 = 82 เกิน 72/สัปดาห์",
         {"R1": make_trip_test_request("R1", "ICT", "2026-09-29 08:00", 6, "OUT_PROVINCE")}, _one())
add_case("19 ทริปหลายวัน", "TC64", "หลังทริป 3 วันจบ 20:00 มีคำขอเริ่มหลังจบ 9 ชม. (พักไม่ครบ)",
         {"R1": make_trip_test_request("R1", "ICT", "2026-09-29 08:00", 3, "OUT_PROVINCE"),
          "R2": make_test_request("R2", start="2026-10-02 05:00", duration_hours=2)}, _one())
add_case("19 ทริปหลายวัน", "TC65", "หลังทริป 3 วันจบ 20:00 มีคำขอเริ่มหลังจบ 10 ชม. (พักครบพอดี)",
         {"R1": make_trip_test_request("R1", "ICT", "2026-09-29 08:00", 3, "OUT_PROVINCE"),
          "R2": make_test_request("R2", start="2026-10-02 06:00", duration_hours=2)}, _one())
add_case("20 ความจุตามกติกาเว็บ", "TC66", "ขอ 2 คัน ผู้โดยสาร 20 คน (> 11 แต่ <= 22)",
         {"R1": make_test_request("R1", passengers=20, vehicles=2)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))
add_case("20 ความจุตามกติกาเว็บ", "TC67", "ขอ 2 คัน ผู้โดยสาร 22 คนพอดี (เพดาน 11x2)",
         {"R1": make_test_request("R1", passengers=22, vehicles=2)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))
add_case("20 ความจุตามกติกาเว็บ", "TC68", "[Robustness] ขอ 2 คัน ผู้โดยสาร 23 คน เกินเพดาน (เว็บบล็อก -> INVALID_INPUT)",
         {"R1": make_test_request("R1", passengers=23, vehicles=2)},
         make_test_resources({"ICT": 2, "SCI": 0, "SEEN": 0}))
add_case("20 ความจุตามกติกาเว็บ", "TC69", "ขอ 2 คัน ผู้โดยสาร 15 คน แต่มีรถพร้อมคันเดียว",
         {"R1": make_test_request("R1", passengers=15, vehicles=2)},
         make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}))
assert len(TEST_CASES) == 69


จำนวน test cases: 54


In [37]:
# ตรวจ test case ที่สร้าง
test_case_list = pd.DataFrame([
    {
        "กลุ่ม": tc["group"],
        "รหัส": tc["case_id"],
        "กรณีทดสอบ": tc["title"],
        "จำนวนคำขอ": len(tc["requests"]),
        "หมายเหตุ": tc["note"],
    }
    for tc in TEST_CASES
])

display(test_case_list)
print("จำนวนทั้งหมด:", len(test_case_list))
print("จำนวนกลุ่มที่ใช้:", test_case_list["กลุ่ม"].nunique())

จำนวนทั้งหมด: 69
จำนวนกลุ่มที่ใช้: 19


                             กลุ่ม  รหัส                                                                              กรณีทดสอบ  จำนวนคำขอ                                                                   หมายเหตุ
0                        1 รถพร้อม  TC01                                                                        มีรถพร้อม 1 คัน          1                                                                           
1                        1 รถพร้อม  TC02                                                       มีรถพร้อม 2 คัน แต่ต้องการ 1 คัน          1                                                                           
2                        1 รถพร้อม  TC03                                                                        ไม่มีรถพร้อมเลย          1                                                                           
3                   2 คำขอพร้อมกัน  TC04                                                         สองคำขอเวลาเดียวกัน รถมี 2 คัน          2      

## รัน test cases

In [38]:
MAX_PAX_PER_VAN = 11   # เพดานผู้โดยสารต่อคัน: เว็บอนุญาต > 11 คนได้เฉพาะเมื่อขอรถ > 1 คัน (ไม่เกิน 11 x จำนวนรถ)

def validate_requests(requests):
    errs, seen = [], set()
    for k, r in requests.items():
        if r["end"] <= r["start"]: errs.append(f"{k}: end <= start")
        if r["passenger_count"] <= 0: errs.append(f"{k}: passengers <= 0")
        if r["requested_vehicle_count"] <= 0: errs.append(f"{k}: vehicles <= 0")
        elif r["passenger_count"] > MAX_PAX_PER_VAN * r["requested_vehicle_count"]:
            errs.append(f"{k}: passengers > {MAX_PAX_PER_VAN} x vehicles (เว็บบล็อก)")
        if not r.get("destination"): errs.append(f"{k}: destination missing")
        # [OPTIONAL: ETA MODE] if _eta_hours(r) < 0: errs.append(f"{k}: negative ETA")
        if r.get("request_id") in seen: errs.append(f"{k}: duplicate request_id {r['request_id']}")
        seen.add(r.get("request_id"))
    return errs

def solve_test_case(tc, **kw):
    res, req = tc["resources"], tc["requests"]
    errs = validate_requests(req)
    if errs:
        return {"solver_status": "INVALID_INPUT", "results": {}, "audit": pd.DataFrame(), "validation_errors": errs}
    cap = tc["capacity"] or {i: res[i].get("test_capacity", 11) for i in res}
    r, a, s = solve_assignment(res, req, capacity_by_resource=cap, **kw)
    return {"solver_status": s, "results": r, "audit": a, "validation_errors": []}


In [39]:
# ---- expected outcomes: กำหนดจากกติกาการปฏิบัติงาน ก่อนรันโมเดล ----
OPT, INV = "Optimal", "INVALID_INPUT"
def E(a, m, cross=None, res=None): return {"status": OPT, "assigned": a, "missing": m, "cross": cross, "res": res}
EXPECTED = {
 "TC01":E(1,0),"TC02":E(1,0),"TC03":E(0,1),"TC04":E(2,0),"TC05":E(1,1),"TC06":E(2,0),"TC07":E(2,0),
 "TC08":E(1,0),"TC09":{"status":INV},"TC10":E(2,0),"TC11":E(2,1),
 "TC12":E(1,0),"TC13":E(1,0),"TC14":E(1,0),"TC15":E(1,0),"TC16":E(0,1),
 "TC17":E(1,0),"TC18":E(0,1),"TC19":E(1,0),"TC20":E(0,1),"TC21":E(0,1),
 "TC22":E(1,0,0),"TC23":E(1,0,1),"TC24":E(2,0,1),"TC25":E(1,0,0),
 "TC26":E(1,0),"TC27":E(1,1),"TC28":E(2,0),"TC29":E(0,1),
 "TC30":E(1,0),"TC31":E(1,0),"TC32":E(1,0),"TC33":E(1,0),
 "TC40":E(2,0),"TC41":E(1,1),"TC42":E(1,1),"TC43":{"status":INV},"TC44":E(1,0),
 "TC45":E(3,0,0),"TC46":E(3,0,0),"TC47":E(2,0),
 "TC48":E(2,0),"TC49":E(2,0),"TC50":E(2,0),"TC51":E(2,0),"TC52":E(2,0),"TC53":E(2,0),"TC54":E(2,0),
 # กติกา v5
 "TC55":E(1,1,0,{"R1":["T01"],"R2":[]}),     # ด่วนข้ามคณะไม่ preempt: ICT ได้รถตัวเอง
 "TC56":E(2,0),
 "TC57":E(0,1),"TC58":E(0,1),                # walk-in ทับ / พักไม่ครบ
 "TC59":E(1,0,1),"TC60":E(1,0,1,{"R1":["T01"]}),
 "TC61":E(1,1),
 "TC62":E(1,0),"TC63":E(0,1),                # 68 ชม. ผ่าน / 82 ชม. ไม่ผ่านเพดานสัปดาห์
 "TC64":E(1,1),"TC65":E(2,0),
 # กติกาความจุตามเว็บ (ผู้โดยสาร <= 11 x จำนวนรถ)
 "TC66":E(2,0),"TC67":E(2,0),"TC68":{"status":INV},"TC69":E(1,1),                # พัก 9 ชม. ไม่ผ่าน / 10 ชม. ผ่าน
}
for c in ("TC34","TC35","TC36","TC37","TC38","TC39"):
    EXPECTED[c] = {"status": INV}

rows = []
for tc in TEST_CASES:
    o = solve_test_case(tc); cid = tc["case_id"]; exp = EXPECTED[cid]
    if o["solver_status"] == INV:
        a = m = 0; cross = None; got_res = {}
    else:
        r = o["results"]
        a = sum(v["assigned_count"] for v in r.values()); m = sum(v["missing_count"] for v in r.values())
        cross = next(iter(r.values()))["cross_faculty_assignments"]
        got_res = {k: v["assigned_resources"] for k, v in r.items()}
    ok = o["solver_status"] == exp["status"]
    if exp["status"] == OPT:
        ok &= (a == exp["assigned"] and m == exp["missing"])
        if exp.get("cross") is not None: ok &= (cross == exp["cross"])
        if exp.get("res"): ok &= all(got_res.get(k) == v for k, v in exp["res"].items())
    rows.append({"case": cid, "group": tc["group"], "title": tc["title"], "status": o["solver_status"],
                 "assigned": a, "missing": m, "cross": cross,
                 "exp_status": exp["status"], "exp_assigned": exp.get("assigned"), "exp_missing": exp.get("missing"),
                 "exp_cross": exp.get("cross"), "pass": bool(ok), "note": "; ".join(o["validation_errors"])})
df = pd.DataFrame(rows)
base = df[df.case.str[2:].astype(int) <= 54]
print("ALL cases:", len(df), "pass:", int(df['pass'].sum()))
print("Original 54: Optimal", int((base.status == OPT).sum()), "rejected", int((base.status == INV).sum()),
      "assigned", int(base.assigned.sum()), "missing", int(base.missing.sum()),
      "requested", int(base.assigned.sum() + base.missing.sum()))
print("Failures:", df[~df["pass"]][["case", "status", "assigned", "missing", "cross"]].to_dict("records"))


ALL cases: 69 pass: 69
Original 54: Optimal 46 rejected 8 assigned 60 missing 11 requested 71
Failures: []


In [40]:
# ส่งออกผล test cases และทะเบียนคนขับ
OUTPUT_DIR = BASE_DIR / "raw_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

driver_master.to_csv(
    OUTPUT_DIR / "new_driver_master.csv",
    index=False,
    encoding="utf-8-sig"
)

precheck.to_csv(
    OUTPUT_DIR / "driver_hours_precheck.csv",
    index=False,
    encoding="utf-8-sig"
)

test_case_list.to_csv(
    OUTPUT_DIR / f"test_case_list_{len(test_case_list)}.csv",
    index=False,
    encoding="utf-8-sig"
)

# แก้ไขจาก test_results_df เป็น df ให้ถูกต้องตามที่ประกาศไว้ด้านบน
df.to_csv(
    OUTPUT_DIR / f"test_case_results_{len(df)}.csv",
    index=False,
    encoding="utf-8-sig"
)

csv_audit.to_csv(
    OUTPUT_DIR / "csv_assignment_candidate_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("บันทึกไฟล์ไว้ที่:", OUTPUT_DIR)
print([p.name for p in OUTPUT_DIR.iterdir()])


บันทึกไฟล์ไว้ที่: D:\Pro\raw_output
['baseline_comparison_real.csv', 'baseline_comparison_synthetic.csv', 'baseline_fulfillment.png', 'csv_assignment_candidate_audit.csv', 'driver_hours_precheck.csv', 'empirical_probability_comparison.csv', 'exp1_scalability.csv', 'exp2_fleet_sizing_and_sharing.csv', 'exp3_hos_sensitivity.csv', 'new_driver_master.csv', 'scenario_empirical_vs_synthetic.csv', 'scenario_exhaustive_results.csv', 'sensitivity_and_scalability.png', 'test_case_list_65.csv', 'test_case_list_69.csv', 'test_case_results_296.csv', 'test_case_results_65.csv', 'test_case_results_69.csv']


In [41]:
# รายงานจัดสรรแบบรวม: คำขอจริงจาก CSV และ test case
# แก้ TEST_CASE_FILTER เป็นรหัส เช่น "TC14" เพื่อกรอง; None คือแสดงทั้งหมด
TEST_CASE_FILTER = None
REPORT_CSV_REQUESTS = True
REPORT_TEST_CASES = True


def evaluate_resource_for_request(resource_id, request_id, resources, requests,
                                  capacity_by_resource, min_rest_hours=MIN_REST_HOURS,
                                  max_daily_hours=MAX_DAILY_HOURS,
                                  max_weekly_hours=MAX_WEEKLY_HOURS):
    """ตรวจคุณสมบัติพื้นฐานของรถ/คนขับ และเก็บเหตุผลพร้อมตัวเลขจริง (ใช้กติกาเดียวกับ solver)."""
    resource, request = resources[resource_id], requests[request_id]
    state = get_effective_driver_state(resource, request["start"])
    multi, prof = daily_profile(request["start"], request["end"])
    hist = _hist_profile(resource["driver_logs"])
    elapsed = (request["end"] - request["start"]).total_seconds() / 3600
    # วันที่ใช้ชั่วโมงรวมสูงสุดเมื่อบวกงานเดิม
    h_of = lambda d: resource.get("test_daily_hours", hist.get(d, 0.0))
    worst_d = max(prof, key=lambda d: h_of(d) + prof[d])
    prior_daily, trip_daily = h_of(worst_d), prof[worst_d]
    daily_after = prior_daily + trip_daily
    days = sorted({d + pd.Timedelta(days=k) for d in prof for k in range(7)})
    hw_of = lambda d: resource.get("test_weekly_hours", _window_sum(hist, d))
    def week_total(d):
        lo = d - pd.Timedelta(days=6)
        return hw_of(d) + sum(h for dd, h in prof.items() if lo <= dd <= d)
    worst_w = max(days, key=week_total)
    prior_weekly = hw_of(worst_w)
    weekly_after = week_total(worst_w)
    trip_weekly = weekly_after - prior_weekly
    capacity = capacity_by_resource.get(resource_id, resource.get("test_capacity", 11))
    single_shift_ok = multi or elapsed <= max_daily_hours + 1e-9
    conflict_text = _history_conflict_detail(resource, request["start"], request["end"], min_rest_hours)
    checks = {
        "ไม่ชนงานคงที่/พักครบ": conflict_text is None,
        "รถพร้อม": bool(resource.get("available", True)),
        "ความจุเพียงพอ": request["passenger_count"] <= capacity,
        "ทริปวันเดียวไม่เกิน 14 ชม.": single_shift_ok,
        "ชั่วโมงรายวันผ่าน": daily_after <= max_daily_hours,
        "ชั่วโมงรายสัปดาห์ผ่าน": weekly_after <= max_weekly_hours,
        "เวลาพักผ่าน": state["rest_hours"] >= min_rest_hours,
    }
    reasons = []
    if not checks["รถพร้อม"]:
        reasons.append("รถหรือคนขับไม่พร้อมใช้งาน")
    if conflict_text is not None:
        reasons.append(conflict_text)
    if not checks["ความจุเพียงพอ"]:
        reasons.append(f"ผู้โดยสาร {request['passenger_count']} คน แต่รถรองรับ {capacity} คน")
    if not checks["ทริปวันเดียวไม่เกิน 14 ชม."]:
        reasons.append(f"ทริปวันเดียวยาว {elapsed:.1f} ชม. ต่อเนื่อง เกินเกณฑ์ {max_daily_hours:.1f} ชม.")
    if not checks["ชั่วโมงรายวันผ่าน"]:
        reasons.append(f"ชั่วโมงงานรายวันเดิม {prior_daily:.1f} + ทริปนี้ {trip_daily:.1f} = {daily_after:.1f} ชม. เกินเกณฑ์ {max_daily_hours:.1f} ชม.")
    if not checks["ชั่วโมงรายสัปดาห์ผ่าน"]:
        reasons.append(f"ชั่วโมงงานรายสัปดาห์เดิม {prior_weekly:.1f} + ทริปนี้ {trip_weekly:.1f} = {weekly_after:.1f} ชม. เกินเกณฑ์ {max_weekly_hours:.1f} ชม.")
    if not checks["เวลาพักผ่าน"]:
        reasons.append(f"พักมาเพียง {state['rest_hours']:.1f} ชม. ต่ำกว่าเกณฑ์ {min_rest_hours:.1f} ชม.")
    return {
        "resource_id": resource_id, "driver": resource["driver"],
        "vehicle": resource["vehicle"], "vehicle_faculty": resource["faculty"],
        "eligible": all(checks.values()), "checks": checks, "failed_reasons": reasons,
        "prior_daily_hours": prior_daily, "daily_hours_after": daily_after,
        "prior_weekly_hours": prior_weekly, "weekly_hours_after": weekly_after,
        "rest_hours": state["rest_hours"], "capacity": capacity,
    }


def _schedule_conflicts(resource_id, request_id, request, requests, assignment_result,
                        min_rest_hours=MIN_REST_HOURS):
    """คำขออื่นที่ solver จัดให้ resource นี้และชนเวลา/พักไม่ครบ."""
    conflicts = []
    for other_id, other in requests.items():
        if other_id == request_id:
            continue
        assigned = assignment_result.get(other_id, {}).get("assigned_resources", [])
        if resource_id not in assigned:
            continue
        if requests_overlap(request, other):
            conflicts.append(f"ถูกจัดให้ {other_id} ({other['start']}–{other['end']}) ซึ่งเวลาทับซ้อน")
        else:
            if request["end"] <= other["start"]:
                gap = (other["start"]-request["end"]).total_seconds()/3600
            else:
                gap = (request["start"]-other["end"]).total_seconds()/3600
            if gap < min_rest_hours:
                conflicts.append(f"ถูกจัดให้ {other_id}; เวลาว่างระหว่างงาน {gap:.1f} ชม. ต่ำกว่าเกณฑ์พัก {min_rest_hours:.1f} ชม.")
    return conflicts


def print_assignment_explanation(request_id, resources, requests, assignment_result,
                                 capacity_by_resource=None, faculty_priority=faculty_priority,
                                 test_case_info=None):
    if capacity_by_resource is None:
        capacity_by_resource = {rid: r.get("test_capacity", 11) for rid, r in resources.items()}
    request, result = requests[request_id], assignment_result[request_id]
    requester = request["source_faculty"]
    assigned_ids = result["assigned_resources"]
    requested = result["requested_count"]
    missing = result["missing_count"]
    print("=" * 100)
    if test_case_info is not None:
        print(f"กลุ่มทดสอบ: {test_case_info.get('group', '-')}")
        print(f"รหัสกรณีทดสอบ: {test_case_info.get('case_id', '-')}")
        print(f"กรณีทดสอบเรื่อง: {test_case_info.get('title', '-')}")
    print(f"รหัสคำขอ: {request_id}")
    print(f"คณะผู้ขอ: {requester}")
    print(f"เวลา: {request['start']} ถึง {request['end']}")
    print(f"ผู้โดยสาร: {request['passenger_count']} คน | ต้องการรถ: {requested} คัน | จัดได้: {len(assigned_ids)} คัน | ไม่สามารถจัดได้: {missing} คัน")
    print("-" * 100)

    all_eval = {rid: evaluate_resource_for_request(rid, request_id, resources, requests, capacity_by_resource) for rid in resources}
    same_faculty = [rid for rid in resources if resources[rid]["faculty"] == requester]

    # ชี้แจงรถคณะผู้ขอที่ไม่มีสิทธิ์ใช้ในคำขอนี้ ทั้งที่ไม่ผ่านเกณฑ์และถูกจองไปแล้ว
    local_reasons = []
    for rid in same_faculty:
        ev = all_eval[rid]
        if ev["failed_reasons"]:
            local_reasons.append(f"{ev['vehicle']} ({ev['driver']}): " + "; ".join(ev["failed_reasons"]))
        conflicts = _schedule_conflicts(rid, request_id, request, requests, assignment_result)
        local_reasons.extend(f"{ev['vehicle']} ({ev['driver']}): {reason}" for reason in conflicts)

    print("รถที่จัดให้")
    if not assigned_ids:
        print("  ไม่มีรถที่จัดให้ในคำขอนี้")
    for rid in assigned_ids:
        ev = all_eval[rid]
        owner = ev["vehicle_faculty"]
        if owner == requester:
            reason = "เป็นรถของคณะผู้ขอและผ่านเงื่อนไขทุกข้อ"
        else:
            reason = f"เป็นรถสำรองจากคณะ {owner} เพราะรถคณะ {requester} ใช้ไม่ได้/ไม่พอ"
            if local_reasons:
                reason += ": " + " | ".join(local_reasons)
            else:
                reason += ": รถคณะผู้ขอไม่สามารถรองรับจำนวนรถที่ต้องการได้ภายใต้ผลการจัดสรรรวม"
            reason += f"; รถคณะ {owner} ผ่านเงื่อนไขและมีลำดับนโยบาย {faculty_priority[requester][owner]}"
        rest_text = "ไม่พบเที่ยวก่อนหน้าในประวัติ" if ev["rest_hours"] == float("inf") else f"{ev['rest_hours']:.1f} ชม."
        print(f"  ✓ {ev['vehicle']} | คนขับ {ev['driver']} | คณะเจ้าของรถ {owner}")
        print(f"    เหตุผล: {reason}")
        print(f"    ชั่วโมงงานรายวัน {ev['prior_daily_hours']:.1f} + ทริปนี้ {ev['daily_hours_after']-ev['prior_daily_hours']:.1f} = {ev['daily_hours_after']:.1f}/{MAX_DAILY_HOURS:.0f} ชม.; รายสัปดาห์ {ev['prior_weekly_hours']:.1f} + ทริปนี้ {ev['weekly_hours_after']-ev['prior_weekly_hours']:.1f} = {ev['weekly_hours_after']:.1f}/{MAX_WEEKLY_HOURS:.0f} ชม.; เวลาพัก {rest_text}; ความจุ {ev['capacity']} คน")

    print("\nรถคณะเดียวกันที่ไม่ผ่านหรือถูกใช้ไปแล้ว")
    if not local_reasons:
        unused_pass = [rid for rid in same_faculty if rid not in assigned_ids and all_eval[rid]["eligible"]]
        if unused_pass:
            for rid in unused_pass:
                ev=all_eval[rid]
                print(f"  • {ev['vehicle']} ({ev['driver']}): ผ่านกฎรายคัน แต่ solver จัดให้คำขออื่น/จัดตามข้อจำกัดรวม")
        else:
            print("  ไม่มีรถคณะเดียวกันที่ไม่ผ่าน; รถคณะเดียวกันที่ผ่านถูกใช้เท่าที่จำเป็น")
    else:
        for reason in dict.fromkeys(local_reasons):
            print("  ✗ " + reason)

    if missing > 0:
        print("\nเหตุผลที่จัดรถได้ไม่ครบ")
        candidate_failures = []
        for rid, ev in all_eval.items():
            if rid in assigned_ids:
                continue
            failures = list(ev["failed_reasons"])
            failures += _schedule_conflicts(rid, request_id, request, requests, assignment_result)
            # Resource ที่ผ่านทุกเกณฑ์แต่ไม่ได้รับเลือก เป็นข้อจำกัดจำนวนรถ/การจัดสรรรวม
            if not failures:
                failures.append("ผ่านเกณฑ์รายคัน แต่ไม่มีรถเหลือพอเมื่อจัดคำขอทั้งหมดร่วมกัน")
            candidate_failures.append((ev, list(dict.fromkeys(failures))))
        if candidate_failures:
            for ev, failures in candidate_failures:
                print(f"  ✗ {ev['vehicle']} ({ev['driver']}, {ev['vehicle_faculty']}): " + "; ".join(failures))
        else:
            print("  ไม่มีรถที่เหลือให้จัด เพราะรถทุกคันที่ผ่านถูกใช้ครบแล้ว")
        print(f"  สรุป: ต้องการ {requested} คัน จัดได้ {len(assigned_ids)} คัน จึงขาด {missing} คัน")

    backups = []
    for rid, ev in all_eval.items():
        if rid in assigned_ids or ev["vehicle_faculty"] == requester:
            continue
        if ev["eligible"] and not _schedule_conflicts(rid, request_id, request, requests, assignment_result):
            backups.append((faculty_priority[requester][ev["vehicle_faculty"]], ev))
    backups.sort(key=lambda item: (item[0], item[1]["vehicle"]))
    print("\nรถสำรองจากคณะอื่นที่ผ่านและไม่ได้ติดตาราง")
    if not backups:
        print("  ไม่มีรถสำรองที่ว่างและผ่านเงื่อนไขครบ")
    else:
        for rank, ev in backups:
            print(f"  → {ev['vehicle']} | คนขับ {ev['driver']} | คณะ {ev['vehicle_faculty']} | เสนอเพราะผ่านความจุ ชั่วโมงงาน เวลาพัก และไม่มีงานชน (ลำดับนโยบาย {rank})")
    print("=" * 100)


# ผลคำขอจริงจาก CSV
if REPORT_CSV_REQUESTS:
    for request_id in csv_requests_unique:
        print_assignment_explanation(request_id, resources, csv_requests_unique, csv_results,
                                     {rid: 11 for rid in resources}, faculty_priority)

# ผล test case ใช้รูปแบบรายงานเดียวกัน พร้อมกลุ่ม/รหัส/หัวข้อ
if REPORT_TEST_CASES:
    for test_case in TEST_CASES:
        if TEST_CASE_FILTER and test_case["case_id"] != TEST_CASE_FILTER:
            continue
        output = solve_test_case(test_case)
        if output["solver_status"] == "INVALID_INPUT":
            print("=" * 100)
            print(f"กลุ่มทดสอบ: {test_case['group']}\nรหัสกรณีทดสอบ: {test_case['case_id']}\nกรณีทดสอบเรื่อง: {test_case['title']}")
            print("สถานะ: ข้อมูลคำขอไม่ถูกต้อง — " + "; ".join(output["validation_errors"]))
            print("=" * 100)
            continue
        cap = test_case["capacity"] or {rid: r.get("test_capacity", 11) for rid, r in test_case["resources"].items()}
        for request_id in test_case["requests"]:
            print_assignment_explanation(request_id, test_case["resources"], test_case["requests"],
                                         output["results"], cap, faculty_priority, test_case)


รหัสคำขอ: ICT_BK000447
คณะผู้ขอ: ICT
เวลา: 2026-06-27 13:00:00 ถึง 2026-06-27 22:00:00
ผู้โดยสาร: 4 คน | ต้องการรถ: 1 คัน | จัดได้: 1 คัน | ไม่สามารถจัดได้: 0 คัน
----------------------------------------------------------------------------------------------------
รถที่จัดให้
  ✓ นข 3948 พะเยา | คนขับ SCI_DRIVER_NOW | คณะเจ้าของรถ SCI
    เหตุผล: เป็นรถสำรองจากคณะ SCI เพราะรถคณะ ICT ใช้ไม่ได้/ไม่พอ: 1 นช 3009 กรุงเทพมหานคร (ICT_DRIVER_NOW): เวลาทับกับงานคงที่ (walk_in) 08/01/2026 08:00–10/10/2026 18:00; ชั่วโมงงานรายวันเดิม 23.0 + ทริปนี้ 9.0 = 32.0 ชม. เกินเกณฑ์ 14.0 ชม.; ชั่วโมงงานรายสัปดาห์เดิม 183.0 + ทริปนี้ 9.0 = 192.0 ชม. เกินเกณฑ์ 72.0 ชม.; รถคณะ SCI ผ่านเงื่อนไขและมีลำดับนโยบาย 2
    ชั่วโมงงานรายวัน 0.0 + ทริปนี้ 9.0 = 9.0/14 ชม.; รายสัปดาห์ 0.0 + ทริปนี้ 9.0 = 9.0/72 ชม.; เวลาพัก 6476.5 ชม.; ความจุ 11 คน

รถคณะเดียวกันที่ไม่ผ่านหรือถูกใช้ไปแล้ว
  ✗ 1 นช 3009 กรุงเทพมหานคร (ICT_DRIVER_NOW): เวลาทับกับงานคงที่ (walk_in) 08/01/2026 08:00–10/10/2026 18:00; ชั่วโมงงานรายวันเดิม 23

## โหมดเสริม (ปิดไว้ — โค้ดทั้งหมดถูกคอมเมนต์ ไม่ถูกรัน)
ฟีเจอร์ 3 อย่างด้านล่างเป็นส่วนเสริมที่ **ไม่อยู่ในกติกามาตรฐาน** จึงปิดไว้ก่อน ถ้าต้องการเปิด: ลบ `# ` หน้าบรรทัดในเซลล์ถัดไป แล้วแก้จุด `[OPTIONAL: ...]` ใน `solve_assignment` ตามที่ระบุ

In [42]:
# ==================== [OPTIONAL 1] ลำดับผู้ให้ยืม (rank 2 / 3) ====================
# ใช้แทน faculty_priority แบบ binary ในเซลล์ "กำหนดนโยบายลำดับความสำคัญของคณะ"
# faculty_priority = {
#     "ICT":  {"ICT": 1, "SCI": 2, "SEEN": 3},
#     "SCI":  {"SCI": 1, "SEEN": 2, "ICT": 3},
#     "SEEN": {"SEEN": 1, "ICT": 2, "SCI": 3},
# }

# ==================== [OPTIONAL 2] ETA: บวกเวลาเดินทางกลับเข้าเวลาที่ไม่ว่าง ====================
# def _eta_hours(req):
#     v = req.get("estimated_travel_hours")
#     return 0.0 if v is None else float(v)
#
# def _busy_end(req):
#     return pd.Timestamp(req["end"]) + pd.Timedelta(hours=_eta_hours(req))
#
# ใน solve_assignment แทนบรรทัด win = {...} ด้วย
#     win = {j: (pd.Timestamp(requests[j]["start"]), _busy_end(requests[j])) for j in J}
# และใน validate_requests เปิดบรรทัด "negative ETA" ที่คอมเมนต์ไว้

# ==================== [OPTIONAL 3] URGENT: ให้ด่วนมีสิทธิ์เหนือคำขอปกติ ====================
# ใน solve_assignment (ส่วน stages ของ lexicographic) เปิดบรรทัด
#     stages = [pulp.lpSum(missing[j] for j in J if requests[j].get("urgent"))] + stages
# หมายเหตุ: ขัดกับกติกา "รถคณะใดให้คณะนั้นก่อน" เพราะคำขอด่วนจากคณะอื่นจะแย่งรถเจ้าของได้

# ==================== test case ของโหมดเสริม (ย้ายมาจาก v4) ====================
# _one = lambda **k: make_test_resources({"ICT": 1, "SCI": 0, "SEEN": 0}, **k)
# OPT-U1: urgent SCI แย่งรถ ICT เดิม -> คาดหวัง R2 ได้รถ T01, R1 ไม่ได้ (assigned 1, missing 1, cross 1)
# add_case("OPT urgent", "OPT-U1", "คำขอด่วน SCI แย่งรถ ICT เมื่อเปิดโหมด urgent",
#          {"R1": make_test_request("R1", faculty="ICT"), "R2": make_test_request("R2", faculty="SCI", urgent=True)}, _one())
# OPT-E1: ETA 5 ชม. ทำให้พักเหลือ 8 ชม. -> คาดหวัง assigned 1, missing 1
# add_case("OPT ETA", "OPT-E1", "ETA กลับ 5 ชม. ทำให้ช่วงพักเหลือ 8 ชม.",
#          {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=4, estimated_travel_hours=5),
#           "R2": make_test_request("R2", start="2026-09-30 01:00", duration_hours=2)}, _one())
# OPT-E2: งาน 10 ชม. + ETA 5 ชม. = 15 ชม. เกิน 14 -> คาดหวัง assigned 0, missing 1
# add_case("OPT ETA", "OPT-E2", "งาน 10 ชม. + ETA 5 ชม. เกิน 14 ชม./วัน",
#          {"R1": make_test_request("R1", start="2026-09-29 08:00", duration_hours=10, estimated_travel_hours=5)}, _one())
# OPT-R1: ลำดับผู้ให้ยืม -> SCI ขอ, SCI ไม่ว่าง, SEEN(rank 2) กับ ICT(rank 3) ว่าง -> คาดหวังเลือก SEEN (T03)
# add_case("OPT lender rank", "OPT-R1", "SEEN (rank 2) ต้องถูกเลือกก่อน ICT (rank 3)",
#          {"R1": make_test_request("R1", faculty="SCI")},
#          make_test_resources({"ICT": 1, "SCI": 1, "SEEN": 1}, available_ids=["T01", "T03"]))
print("โหมดเสริมทั้งหมดปิดอยู่ (urgent / ETA / ลำดับผู้ให้ยืม)")


โหมดเสริมทั้งหมดปิดอยู่ (urgent / ETA / ลำดับผู้ให้ยืม)


# Probability case test



In [43]:
from itertools import product

# จำลอง Features (นำมาจาก cell_8bZAEJY0xgzg)
faculties = ['ICT', 'SCI', 'SEEN']
trip_scopes = ['IN_PROVINCE', 'OUT_PROVINCE']
passenger_ranges = ['<=11', '>11']
vehicle_ranges = ['1', '>1']
urgencies = [True, False]
day_ranges = [1, 2, 3]

# สร้างทุกชุดของความเป็นไปได้
all_combinations = list(product(faculties, faculties, trip_scopes, passenger_ranges, vehicle_ranges, urgencies, day_ranges))

# สร้าง DataFrame จากชุดความเป็นไปได้
df_combinations = pd.DataFrame(all_combinations, columns=[
    'Requester_Faculty', # คณะที่จอง
    'Vehicle_Owner_Faculty', # รถของคณะที่ต้องการ
    'Trip_Scope', #ขอบเขตการเดินทาง(ใน/นอกจังหวัด)
    'Passenger_Range', # จำนวนผู้โดยสารที่ต้องการ
    'Vehicle_Count_Range', # จำนวนรถที่ต้องการ
    'Urgent_Request', # คำขอจองที่เร่งด่วน
    'Day_Range' # ช่วงเวลาที่เดินทาง
])

# คำนวณจำนวนความเป็นไปได้ทั้งหมด
total_combinations_count = len(df_combinations)

# คำนวณความน่าจะเป็นของแต่ละชุด (หากสมมติว่าแต่ละฟีเจอร์มีความน่าจะเป็นเท่ากัน)
df_combinations['Probability'] = 1 / total_combinations_count

print(f"📊 ตารางแสดง {total_combinations_count} ชุดของความเป็นไปได้:")
display(df_combinations)

📊 ตารางแสดง 432 ชุดของความเป็นไปได้:


    Requester_Faculty Vehicle_Owner_Faculty    Trip_Scope Passenger_Range Vehicle_Count_Range  Urgent_Request  Day_Range  Probability
0                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          1     0.002315
1                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          2     0.002315
2                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          3     0.002315
3                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          1     0.002315
4                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          2     0.002315
5                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          3     0.002315
6                 ICT                   ICT   IN_PROVINCE     

In [44]:
import pandas as pd

# ตั้งค่าให้แสดงผลทุกแถว
pd.set_option('display.max_rows', None)

# แสดง DataFrame df_combinations
display(df_combinations)

# คืนค่าการตั้งค่าเริ่มต้นของ pandas
pd.reset_option('display.max_rows')

    Requester_Faculty Vehicle_Owner_Faculty    Trip_Scope Passenger_Range Vehicle_Count_Range  Urgent_Request  Day_Range  Probability
0                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          1     0.002315
1                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          2     0.002315
2                 ICT                   ICT   IN_PROVINCE            <=11                   1            True          3     0.002315
3                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          1     0.002315
4                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          2     0.002315
5                 ICT                   ICT   IN_PROVINCE            <=11                   1           False          3     0.002315
6                 ICT                   ICT   IN_PROVINCE     

### กำหนดความน่าจะเป็นแบบไม่เท่ากัน

หากต้องการให้แต่ละชุดความเป็นไปได้มีความน่าจะเป็นไม่เท่ากัน เราสามารถกำหนดน้ำหนักหรือความน่าจะเป็นแยกกันสำหรับแต่ละค่าในแต่ละฟีเจอร์ จากนั้นคำนวณความน่าจะเป็นของแต่ละชุดโดยการคูณความน่าจะเป็นของค่าต่างๆ เข้าด้วยกัน

ตัวอย่างด้านล่างแสดงการกำหนดน้ำหนักให้กับคณะต่างๆ และขอบเขตการเดินทาง จากนั้นคำนวณความน่าจะเป็นของแต่ละชุดความเป็นไปได้ใหม่

In [45]:
# กำหนดน้ำหนัก (ความน่าจะเป็น) ที่ไม่เท่ากันสำหรับแต่ละฟีเจอร์
# ตัวอย่าง: สมมติว่า ICT มีคำขอมากกว่า SCI และ SEEN
# และ IN_PROVINCE มีคำขอมากกว่า OUT_PROVINCE

faculty_weights = {
    'ICT': 0.5,
    'SCI': 0.3,
    'SEEN': 0.2
}

trip_scope_weights = {
    'IN_PROVINCE': 0.7,
    'OUT_PROVINCE': 0.3
}

# สำหรับฟีเจอร์อื่นๆ ที่เหลือ เราจะให้ความน่าจะเป็นเท่ากันไปก่อน
passenger_range_weights = {'<=11': 0.8, '>11': 0.2} # สมมติว่าส่วนใหญ่ผู้โดยสารไม่เกิน 11
vehicle_count_weights = {'1': 0.9, '>1': 0.1} # สมมติว่าส่วนใหญ่ต้องการรถ 1 คัน
urgency_weights = {True: 0.1, False: 0.9} # สมมติว่าคำขอด่วนมีน้อยกว่า
day_range_weights = {1: 0.6, 2: 0.3, 3: 0.1} # สมมติว่าส่วนใหญ่เป็นการเดินทาง 1 วัน

# สร้าง DataFrame จากชุดความเป็นไปได้เดิม
df_combinations_weighted = df_combinations.copy()

# คำนวณความน่าจะเป็นใหม่สำหรับแต่ละแถว
df_combinations_weighted['Weighted_Probability'] = df_combinations_weighted.apply(
    lambda row:
        faculty_weights[row['Requester_Faculty']] *
        faculty_weights[row['Vehicle_Owner_Faculty']] *
        trip_scope_weights[row['Trip_Scope']] *
        passenger_range_weights[row['Passenger_Range']] *
        vehicle_count_weights[row['Vehicle_Count_Range']] *
        urgency_weights[row['Urgent_Request']] *
        day_range_weights[row['Day_Range']],
    axis=1
)

# Normalize the probabilities so they sum to 1
df_combinations_weighted['Weighted_Probability'] = df_combinations_weighted['Weighted_Probability'] / df_combinations_weighted['Weighted_Probability'].sum()

print(f"📊 ตารางแสดง {len(df_combinations_weighted)} ชุดของความเป็นไปได้พร้อมความน่าจะเป็นแบบไม่เท่ากัน:")
display(df_combinations_weighted.head())

print(f"✅ ผลรวมของความน่าจะเป็นทั้งหมด: {df_combinations_weighted['Weighted_Probability'].sum():.6f}")

📊 ตารางแสดง 432 ชุดของความเป็นไปได้พร้อมความน่าจะเป็นแบบไม่เท่ากัน:
✅ ผลรวมของความน่าจะเป็นทั้งหมด: 1.000000


  Requester_Faculty Vehicle_Owner_Faculty   Trip_Scope Passenger_Range Vehicle_Count_Range  Urgent_Request  Day_Range  Probability  Weighted_Probability
0               ICT                   ICT  IN_PROVINCE            <=11                   1            True          1     0.002315               0.00756
1               ICT                   ICT  IN_PROVINCE            <=11                   1            True          2     0.002315               0.00378
2               ICT                   ICT  IN_PROVINCE            <=11                   1            True          3     0.002315               0.00126
3               ICT                   ICT  IN_PROVINCE            <=11                   1           False          1     0.002315               0.06804
4               ICT                   ICT  IN_PROVINCE            <=11                   1           False          2     0.002315               0.03402

In [46]:
# ตัวอย่างการใช้ check_multi_day_trip กับข้อมูลจริง (ฟังก์ชันนิยามไว้ที่ส่วน core ด้านบน และถูกใช้ใน solver แล้ว)
_demo = []
for _fac, (_d, _b, _appr) in _eda.items():
    for _r in _appr.itertuples():
        _m, _ = check_multi_day_trip(_r.s, _r.e)
        _demo.append({"คณะ": _fac, "booking": _r.booking_id, "เริ่ม": _r.s, "จบ": _r.e,
                      "ชม.จอง": round(_r.hours, 1), "หลายวัน": _m,
                      "ชม.ทำงานที่นับ": round(sum(daily_profile(_r.s, _r.e)[1].values()), 1)})
_demo = pd.DataFrame(_demo)
print(f"booking ที่อนุมัติ {len(_demo)} รายการ: หลายวัน {int(_demo['หลายวัน'].sum())} รายการ")
display(_demo[_demo["หลายวัน"]].head(10))


booking ที่อนุมัติ 520 รายการ: หลายวัน 53 รายการ


    คณะ   booking               เริ่ม                  จบ  ชม.จอง  หลายวัน  ชม.ทำงานที่นับ
10  ICT  BK000209 2025-01-29 09:00:00 2025-02-01 23:00:00    86.0     True            56.0
11  ICT  BK000210 2025-02-11 13:00:00 2025-02-12 20:00:00    31.0     True            21.0
13  ICT  BK000212 2025-02-14 17:00:00 2025-02-16 22:00:00    53.0     True            33.0
24  ICT  BK000224 2025-03-21 07:00:00 2025-03-23 20:00:00    61.0     True            41.0
25  ICT  BK000225 2025-03-27 06:00:00 2025-03-29 20:00:00    62.0     True            42.0
28  ICT  BK000228 2025-04-18 06:00:00 2025-04-21 20:00:00    86.0     True            56.0
29  ICT  BK000229 2025-04-23 06:00:00 2025-04-24 20:00:00    38.0     True            28.0
30  ICT  BK000230 2025-04-18 06:00:00 2025-04-22 22:00:00   112.0     True            70.0
31  ICT  BK000231 2025-04-03 09:00:00 2025-04-07 16:00:00   103.0     True            63.0
35  ICT  BK000235 2025-04-25 07:00:00 2025-04-27 18:00:00    59.0     True            39.0

In [47]:
# 2) คำสั่งเสนอรถสองคันที่พร้อม หากผู้โดยสาร > 11 คน และต้องการรถมากกว่า 1 คัน
# ==========================================

def suggest_vans_for_large_group(request, available_resources):
    """
    หากจำนวนผู้โดยสาร > 11 คน และความต้องการรถ > 1 คัน
    จะดึงและเสนอรถ 2 คันที่พร้อมเดินทาง (Available) ของคณะผู้ขอ หรือคณะสำรอง
    """
    passengers = request.get("passenger_count", 0)
    requested_vehicles = request.get("requested_vehicle_count", 1)
    requester_faculty = request.get("source_faculty", "ICT")

    print(f"🔍 ตรวจสอบคำขอ {request['request_id']}: ผู้โดยสาร {passengers} คน, ต้องการรถ {requested_vehicles} คัน")

    if passengers > 11 and requested_vehicles > 1:
        print("⚠️ ตรวจพบจำนวนผู้โดยสารเกินความจุ 1 คัน (>11 คน) และต้องการรถมากกว่า 1 คัน -> เสนอจัดหารถ 2 คัน")

        # กรองหารถที่พร้อมเดินทาง (Available)
        ready_vans = []
        for rid, res in available_resources.items():
            if res.get("available", True):
                ready_vans.append(res)

        # เรียงลำดับรถตามคณะที่เป็นเจ้าของก่อนเพื่อสิทธิประโยชน์สูงสุด
        ready_vans.sort(key=lambda x: 0 if x["faculty"] == requester_faculty else 1)

        if len(ready_vans) >= 2:
            selected_vans = ready_vans[:2]
            print("✅ ข้อเสนอการจัดสรรรถยนต์ 2 คันที่พร้อมเดินทาง:")
            for idx, van in enumerate(selected_vans, 1):
                print(f"   คันที่ {idx}: ทะเบียน {van['vehicle']} | คนขับ {van['driver']} (สังกัดคณะ {van['faculty']})")
            return selected_vans
        else:
            print(f"❌ มีรถพร้อมใช้งานเพียง {len(ready_vans)} คัน ซึ่งไม่เพียงพอสำหรับการเดินทางพร้อมกัน 2 คัน")
            return []
    else:
        print("ℹ️ คำขอนี้ไม่เข้าเงื่อนไขกลุ่มใหญ่ (ผู้โดยสาร <= 11 หรือต้องการรถเพียง 1 คัน)")
        return []

# Exhaustive Scenario Test (ตามคำแนะนำอาจารย์ที่ปรึกษา)

ไล่ทุกความเป็นไปได้ของ **ลักษณะคำขอ** (คณะ x ใน/นอกจังหวัด x ผู้โดยสาร x จำนวนรถ x ด่วน x 1-3 วัน = 144 แบบ) กับ **สถานะรถว่าง/ไม่ว่าง** (2^3 = 8 แบบ) = 1,152 แบบ แล้ว **ตัดกรณีผู้โดยสาร > 11 คน + ขอรถ 1 คัน ออก** (เว็บบล็อกอยู่แล้ว) เหลือ **864 สถานการณ์**
รันโมเดลทุกสถานการณ์ แล้วเทียบกับ *oracle* (คำตอบที่คำนวณจากกติกาโดยตรง) พร้อมถ่วงน้ำหนักด้วยความน่าจะเป็น

> ต้องรันเซลล์ core (solver, test helpers) ด้านบนก่อน. ค่าความน่าจะเป็นเป็นสมมติฐาน ควรแทนด้วยสัดส่วนจากข้อมูลจริง
> สมมติฐานการทดสอบ: ข้อมูลนำเข้าผ่านการตรวจจากเว็บก่อนถึงโมเดล (ผู้โดยสาร <= 11 x จำนวนรถ) และ `validate_requests` ซ้ำเป็นด่านที่สอง


In [48]:
# ==================== Exhaustive Scenario Test (ตามคำแนะนำอาจารย์ที่ปรึกษา) ====================
# แนวคิด: ไล่ "ทุกความเป็นไปได้" ของ (ลักษณะคำขอ x สถานะรถว่าง/ไม่ว่างของ 3 คัน)
# แล้วรันโมเดลทุกสถานการณ์ เทียบกับ "คำตอบที่คาดหวัง" ที่คำนวณจากกติกาโดยตรง (oracle) ไม่ผ่าน solver
from itertools import product

SC_FACULTIES = ["ICT", "SCI", "SEEN"]
SC_SCOPES = ["IN_PROVINCE", "OUT_PROVINCE"]
SC_PAX = {"<=11": 5, ">11": 15}                 # ตัวแทนผู้โดยสารแต่ละช่วง (>11 ใช้ได้เฉพาะเมื่อขอรถ >1 คัน ตามกติกาเว็บ)
SC_VEH = {"1": 1, ">1": 2}                      # ตัวแทนจำนวนรถที่ขอ
SC_DAYS = [1, 2, 3]
SC_URGENT = [False, True]
VAN_OF = {"ICT": "T01", "SCI": "T02", "SEEN": "T03"}   # make_test_resources สร้างรหัสตามลำดับนี้
FLEET_STATES = [frozenset(c) for n in range(4) for c in combinations(VAN_OF.values(), n)]  # 2^3 = 8 แบบ

# ความน่าจะเป็น (สมมติฐาน — ควรแทนด้วยสถิติจากข้อมูลจริง เช่นสัดส่วนใน raw_data)
P_FAC = {"ICT": 0.5, "SCI": 0.3, "SEEN": 0.2}
P_SCOPE = {"IN_PROVINCE": 0.7, "OUT_PROVINCE": 0.3}
P_PAX = {"<=11": 0.8, ">11": 0.2}
P_VEH = {"1": 0.9, ">1": 0.1}
P_URG = {False: 0.9, True: 0.1}
P_DAY = {1: 0.6, 2: 0.3, 3: 0.1}
P_VAN_FREE = 0.7                                  # โอกาสที่รถแต่ละคันว่างในช่วงที่ขอ

def sc_duration(days, scope):
    return {1: 6 if scope == "IN_PROVINCE" else 12, 2: 30, 3: 54}[days]   # ชั่วโมง (ทริปวันเดียว <= 14)

def sc_expected(fac, pax, veh, free_vans):
    """oracle: คำนวณผลที่ควรได้จากกติกาโดยตรง (รถว่าง + ความจุ 11 >= ผู้โดยสาร)"""
    # ขอรถ 1 คัน: ผู้โดยสาร <= 11 ; ขอรถ > 1 คัน: ผู้โดยสาร <= 11 x จำนวนรถที่ขอ
    eligible = set(free_vans) if pax <= 11 * veh else set()
    n = min(veh, len(eligible))
    own = VAN_OF[fac]
    own_used = own in eligible          # ถ้ารถคณะตัวเองใช้ได้ ต้องได้ใช้ก่อน
    return {"assigned": n, "missing": veh - n, "own_used": own_used and n >= 1,
            "cross": n - (1 if (own_used and n >= 1) else 0)}

def sc_probability(fac, scope, pax_k, veh_k, urg, days, free_vans):
    p = P_FAC[fac] * P_SCOPE[scope] * P_PAX[pax_k] * P_VEH[veh_k] * P_URG[urg] * P_DAY[days]
    for v in VAN_OF.values():
        p *= P_VAN_FREE if v in free_vans else (1 - P_VAN_FREE)
    return p

rows = []
for fac, scope, pax_k, veh_k, urg, days, free in product(SC_FACULTIES, SC_SCOPES, SC_PAX, SC_VEH, SC_URGENT, SC_DAYS, FLEET_STATES):
    if pax_k == ">11" and veh_k == "1":
        continue   # เว็บบล็อกอยู่แล้ว (ผู้โดยสาร > 11 ต้องขอรถ > 1 คัน) จึงไม่นับเป็นสถานการณ์ที่เกิดขึ้นได้
    pax, veh = SC_PAX[pax_k], SC_VEH[veh_k]
    req = make_test_request("S1", faculty=fac, start=TEST_DATE, duration_hours=sc_duration(days, scope),
                            passengers=pax, vehicles=veh, trip_scope=scope, urgent=urg)
    res = make_test_resources(available_ids=set(free))
    out, _, status = solve_assignment(res, {"S1": req}, capacity_by_resource={i: 11 for i in res})
    r = out["S1"]
    assigned_fac = [res[i]["faculty"] for i in r["assigned_resources"]]
    got_own = fac in assigned_fac
    exp = sc_expected(fac, pax, veh, free)
    ok = (status == "Optimal" and r["assigned_count"] == exp["assigned"] and r["missing_count"] == exp["missing"]
          and got_own == exp["own_used"] and (r["assigned_count"] - int(got_own)) == exp["cross"])
    if exp["missing"] == 0:
        outcome = "จัดได้ครบ"
    elif exp["assigned"] > 0:
        outcome = "จัดได้บางส่วน"
    else:
        outcome = "จัดไม่ได้"
    reason = ("ผู้โดยสารเกินความจุ" if pax > 11 * veh else "รถว่างไม่พอ" if exp["missing"] else "")
    rows.append({"Requester": fac, "Scope": scope, "Pax": pax_k, "Vehicles": veh_k, "Urgent": urg, "Days": days,
                 "Free_vans": ",".join(sorted(free)) or "-", "Status": status,
                 "Expected_assigned": exp["assigned"], "Model_assigned": r["assigned_count"],
                 "Outcome": outcome, "Reason": reason, "Cross_vans": r["assigned_count"] - int(got_own),
                 "Match_expected": ok,
                 "Probability": sc_probability(fac, scope, pax_k, veh_k, urg, days, free)})

df_scen = pd.DataFrame(rows)
df_scen["Probability"] /= df_scen["Probability"].sum()

print(f"จำนวนสถานการณ์ทั้งหมด: {len(df_scen)}  (ตัดกรณีผู้โดยสาร > 11 กับรถ 1 คัน ที่เว็บบล็อกออกแล้ว)")
print(f"โมเดลตรงกับ oracle: {int(df_scen['Match_expected'].sum())}/{len(df_scen)}  | solver ไม่ Optimal: {(df_scen['Status'] != 'Optimal').sum()}")
display(df_scen[~df_scen["Match_expected"]].head(20))

summary = df_scen.groupby("Outcome").agg(scenarios=("Outcome", "size"), probability=("Probability", "sum"))
summary["share_of_scenarios"] = summary["scenarios"] / len(df_scen)
print("\nผลการจัดสรร: จำนวนสถานการณ์ และความน่าจะเป็นถ่วงน้ำหนัก")
display(summary)
print("\nสาเหตุที่จัดไม่ครบ")
display(df_scen[df_scen["Outcome"] != "จัดได้ครบ"].groupby("Reason").agg(scenarios=("Reason", "size"), probability=("Probability", "sum")))
df_scen.to_csv(OUTPUT_DIR / "scenario_exhaustive_results.csv", index=False, encoding="utf-8-sig")


จำนวนสถานการณ์ทั้งหมด: 864  (ตัดกรณีผู้โดยสาร > 11 กับรถ 1 คัน ที่เว็บบล็อกออกแล้ว)
โมเดลตรงกับ oracle: 864/864  | solver ไม่ Optimal: 0

ผลการจัดสรร: จำนวนสถานการณ์ และความน่าจะเป็นถ่วงน้ำหนัก

สาเหตุที่จัดไม่ครบ


Empty DataFrame
Columns: [Requester, Scope, Pax, Vehicles, Urgent, Days, Free_vans, Status, Expected_assigned, Model_assigned, Outcome, Reason, Cross_vans, Match_expected, Probability]
Index: []

               scenarios  probability  share_of_scenarios
Outcome                                                  
จัดได้ครบ            540     0.949951               0.625
จัดได้บางส่วน        216     0.023049               0.250
จัดไม่ได้            108     0.027000               0.125

             scenarios  probability
Reason                             
รถว่างไม่พอ        324     0.050049

# Baseline: FCFS เทียบกับ MILP

| วิธี | อธิบาย |
|---|---|
| **MILP** | โมเดลของเรา (solve_assignment) มองทุกคำขอพร้อมกัน |
| **FCFS-A (ไม่ตรวจกฎ)** | เลียนแบบการจัดมือ: มาก่อนได้ก่อน ตรวจแค่รถว่าง/ความจุ/เวลาไม่ชน |
| **FCFS-B (ตรวจกฎครบ)** | มาก่อนได้ก่อน ใช้กฎ 14/72/พัก 10 ชม. เหมือน MILP แต่ตัดสินใจทีละคำขอ |

Metrics: fulfillment rate, สัดส่วนคำขอที่ได้ครบ, จำนวนครั้งยืมรถข้ามคณะ, จำนวนการฝ่าฝืนกฎ, runtime
> ต้องรันเซลล์ scenario (ที่นิยาม P_FAC, SC_FACULTIES ฯลฯ) ก่อน

In [49]:
# ==================== Baseline: FCFS (First-Come First-Served) ====================
# เปรียบเทียบกับ MILP (solve_assignment) ด้วยกติกาและข้อมูลชุดเดียวกัน
#   FCFS-A "ไม่ตรวจกฎ"  : เลียนแบบการจัดมือ  -> ตรวจแค่ รถว่าง + ความจุ + เวลาไม่ชนกัน (ไม่ตรวจ 14/72/พัก 10 ชม.)
#   FCFS-B "ตรวจกฎครบ"  : greedy ตามลำดับคำขอ ใช้กฎเดียวกับ MILP ทุกข้อ แต่ตัดสินใจทีละคำขอ (ไม่มองภาพรวม)
# ลำดับ FCFS ใช้ r["submitted_at"] ถ้ามี ไม่เช่นนั้นใช้เวลาเริ่มทริป
import copy, time

def _fcfs_order(requests):
    return sorted(requests, key=lambda j: (pd.Timestamp(requests[j].get("submitted_at", requests[j]["start"])), str(j)))

def _fcfs_result(requests, assigned):
    return {j: {"assigned_resources": assigned.get(j, []), "assigned_count": len(assigned.get(j, [])),
                "requested_count": requests[j]["requested_vehicle_count"],
                "missing_count": requests[j]["requested_vehicle_count"] - len(assigned.get(j, []))} for j in requests}

def _cap_ok_simple(res, req, cap=11):
    pax, veh = req["passenger_count"], req["requested_vehicle_count"]
    return cap >= pax if veh <= 1 else cap * veh >= pax

def solve_fcfs_naive(resources, requests, capacity=11):
    """FCFS-A: เลือกรถคณะตัวเองก่อน ถ้าไม่ว่าง/ชนเวลา ใช้รถคณะอื่น โดยไม่ตรวจชั่วโมงทำงานและเวลาพัก"""
    busy = {i: [] for i in resources}          # (start, end) ที่จัดไปแล้ว
    for i, r in resources.items():
        for b in (r["driver_logs"].itertuples(index=False) if len(r["driver_logs"]) else []):
            busy[i].append((pd.Timestamp(b.start_dt), pd.Timestamp(b.end_dt)))
    assigned = {}
    for j in _fcfs_order(requests):
        rq = requests[j]
        s, e = pd.Timestamp(rq["start"]), pd.Timestamp(rq["end"])
        cands = sorted(resources, key=lambda i: (resources[i]["faculty"] != rq["source_faculty"], str(i)))
        picked = []
        for i in cands:
            if len(picked) == rq["requested_vehicle_count"]:
                break
            if not resources[i].get("available", True) or not _cap_ok_simple(resources[i], rq, capacity):
                continue
            if any(s < be and bs < e for bs, be in busy[i]):
                continue
            picked.append(i)
        for i in picked:
            busy[i].append((s, e))
        assigned[j] = picked
    return _fcfs_result(requests, assigned), None, "Optimal"

def solve_fcfs_rules(resources, requests, capacity=11):
    """FCFS-B: จัดทีละคำขอตามลำดับมาก่อนได้ก่อน ใช้ solve_assignment กับ 1 คำขอ (กฎเหมือน MILP ทุกข้อ)
    แล้วบันทึกงานที่จัดไปเป็น busy block ของคนขับ เพื่อให้คำขอถัดไปเห็น"""
    res = {i: dict(r, driver_logs=r["driver_logs"].copy()) for i, r in resources.items()}
    assigned = {}
    for j in _fcfs_order(requests):
        out, _, st = solve_assignment(res, {j: requests[j]}, capacity_by_resource={i: capacity for i in res})
        picked = out[j]["assigned_resources"] if st == "Optimal" else []
        rq = requests[j]
        for i in picked:
            new = pd.DataFrame([{"driver_name": res[i]["driver"], "start_dt": pd.Timestamp(rq["start"]),
                                 "end_dt": pd.Timestamp(rq["end"]),
                                 "trip_hours": (pd.Timestamp(rq["end"]) - pd.Timestamp(rq["start"])).total_seconds() / 3600}])
            res[i]["driver_logs"] = pd.concat([res[i]["driver_logs"], new], ignore_index=True)
        assigned[j] = picked
    return _fcfs_result(requests, assigned), None, "Optimal"

def count_violations(results, requests, resources, max_daily=None, max_weekly=None, min_rest=None):
    """นับการฝ่าฝืนกฎที่เกิดจากงานที่ "จัดใหม่" (รวมการชน/พักไม่ครบกับ busy block เดิม):
    ชนเวลา / พัก < 10 ชม. / เกิน 14 ชม.ต่อวัน / เกิน 72 ชม. rolling 7 วัน (นับเมื่อมีงานใหม่เกี่ยวข้อง)"""
    max_daily = max_daily or MAX_DAILY_HOURS; max_weekly = max_weekly or MAX_WEEKLY_HOURS; min_rest = min_rest or MIN_REST_HOURS
    per = {i: [] for i in resources}                 # (start, end, is_new)
    for i, r in resources.items():
        for b_ in (r["driver_logs"].itertuples(index=False) if len(r["driver_logs"]) else []):
            per[i].append((pd.Timestamp(b_.start_dt), pd.Timestamp(b_.end_dt), False))
    for j, r in results.items():
        for i in r["assigned_resources"]:
            per[i].append((pd.Timestamp(requests[j]["start"]), pd.Timestamp(requests[j]["end"]), True))
    v = {"overlap": 0, "rest": 0, "daily": 0, "weekly": 0}
    for i, iv in per.items():
        iv.sort(key=lambda t: (t[0], t[1]))
        for (s1, e1, n1), (s2, e2, n2) in zip(iv, iv[1:]):
            if not (n1 or n2):
                continue
            if s2 < e1: v["overlap"] += 1
            elif (s2 - e1).total_seconds() / 3600 < min_rest: v["rest"] += 1
        prof, new_days = {}, set()
        for s, e, is_new in iv:
            for d, h in daily_profile(s, e)[1].items():
                prof[d] = prof.get(d, 0.0) + h
                if is_new: new_days.add(d)
        v["daily"] += sum(prof[d] > max_daily + 1e-9 for d in new_days)
        v["weekly"] += sum(_window_sum(prof, d) > max_weekly + 1e-9 for d in prof
                           if any(d - pd.Timedelta(days=6) <= nd <= d for nd in new_days))
    v["total"] = sum(v.values())
    return v

def evaluate_method(name, fn, resources, requests):
    t0 = time.perf_counter()
    out = fn(resources, requests)[0]
    dt = time.perf_counter() - t0
    req_v = sum(r["requested_count"] for r in out.values()); got = sum(r["assigned_count"] for r in out.values())
    own = sum(1 for j, r in out.items() for i in r["assigned_resources"] if resources[i]["faculty"] == requests[j]["source_faculty"])
    viol = count_violations(out, requests, resources)
    return {"method": name, "fulfillment": got / req_v if req_v else 1.0, "assigned": got, "requested": req_v,
            "full_requests": sum(r["missing_count"] == 0 for r in out.values()) / len(out),
            "cross_loans": got - own, "violations": viol["total"], **{f"v_{k}": viol[k] for k in ("overlap", "rest", "daily", "weekly")},
            "runtime_s": dt}

METHODS = {"MILP": lambda r, q: solve_assignment(r, q, capacity_by_resource={i: 11 for i in r}),
           "FCFS-A (ไม่ตรวจกฎ)": solve_fcfs_naive,
           "FCFS-B (ตรวจกฎครบ)": solve_fcfs_rules}
print("✅ baseline functions พร้อมใช้งาน:", list(METHODS))

def solve_sdf_rules(resources, requests, capacity=11):
    """Heuristic-SDF (Shortest-Duration-First): เรียงคำขอจากทริปสั้นไปยาว (ตามทฤษฎี Interval Scheduling) แล้วจัดภายใต้กฎ C1-C5 ครบถ้วน"""
    res = {i: dict(r, driver_logs=r["driver_logs"].copy()) for i, r in resources.items()}
    assigned = {}
    order = sorted(requests, key=lambda j: ((pd.Timestamp(requests[j]["end"]) - pd.Timestamp(requests[j]["start"])).total_seconds(),
                                            pd.Timestamp(requests[j]["start"]), str(j)))
    for j in order:
        out, _, st = solve_assignment(res, {j: requests[j]}, capacity_by_resource={i: capacity for i in res})
        picked = out[j]["assigned_resources"] if st == "Optimal" else []
        rq = requests[j]
        for i in picked:
            new = pd.DataFrame([{"driver_name": res[i]["driver"], "start_dt": pd.Timestamp(rq["start"]),
                                 "end_dt": pd.Timestamp(rq["end"]),
                                 "trip_hours": (pd.Timestamp(rq["end"]) - pd.Timestamp(rq["start"])).total_seconds() / 3600}])
            res[i]["driver_logs"] = pd.concat([res[i]["driver_logs"], new], ignore_index=True)
        assigned[j] = picked
    return _fcfs_result(requests, assigned), None, "Optimal"

METHODS["Heuristic-SDF (ทริปสั้นก่อน)"] = solve_sdf_rules


✅ baseline functions พร้อมใช้งาน: ['MILP', 'FCFS-A (ไม่ตรวจกฎ)', 'FCFS-B (ตรวจกฎครบ)']


In [50]:
# ==================== Experiment: MILP vs FCFS (กลุ่มคำขอจำลองหลายคำขอแย่งรถ) ====================
# สุ่มชุดคำขอ N คำขอในช่วง 14 วัน (ลักษณะคำขอตามสัดส่วนในเซลล์ scenario) รถ 3 คัน (ICT/SCI/SEEN)
# ลำดับการยื่น (submitted_at) สุ่มต่างจากเวลาเดินทาง เพื่อให้เห็นผลของการตัดสินใจแบบ myopic ของ FCFS
EXP_N_REQUESTS = [5, 10, 20, 30]
EXP_SEEDS = range(10)                       # ปรับเป็น 30+ สำหรับเปเปอร์
EXP_DURATIONS = [2, 4, 6, 8, 12, 30, 54]
EXP_DUR_P = [0.2, 0.2, 0.2, 0.15, 0.1, 0.1, 0.05]

def make_random_batch(n, seed, horizon_days=14):
    rng = np.random.default_rng(seed)
    reqs = {}
    for k in range(n):
        fac = rng.choice(SC_FACULTIES, p=[P_FAC[f] for f in SC_FACULTIES])
        veh = int(rng.choice([1, 2], p=[P_VEH["1"], P_VEH[">1"]]))
        pax = int(rng.integers(1, 11 * veh + 1)) if veh > 1 else int(rng.integers(1, 12))
        start = TEST_DATE + pd.Timedelta(hours=int(rng.integers(0, horizon_days * 24)))
        reqs[f"R{k+1}"] = make_test_request(f"R{k+1}", faculty=str(fac), start=start,
                                           duration_hours=int(rng.choice(EXP_DURATIONS, p=EXP_DUR_P)),
                                           passengers=pax, vehicles=veh,
                                           submitted_at=TEST_DATE - pd.Timedelta(minutes=int(rng.integers(0, 60 * 24 * 14))))
    return reqs

def fresh_resources():
    res = make_test_resources()
    for r in res.values():                  # ให้ solver ใช้ประวัติจริงจาก driver_logs แทนค่าจำลอง
        for k in ("test_daily_hours", "test_weekly_hours", "test_rest_hours"):
            r.pop(k, None)
    return res

exp_rows = []
for n in EXP_N_REQUESTS:
    for seed in EXP_SEEDS:
        reqs = make_random_batch(n, seed)
        for name, fn in METHODS.items():
            m = evaluate_method(name, fn, fresh_resources(), reqs)
            exp_rows.append({"N": n, "seed": seed, **m})
df_exp = pd.DataFrame(exp_rows)

agg = df_exp.groupby(["N", "method"]).agg(fulfillment=("fulfillment", "mean"), full_requests=("full_requests", "mean"),
                                          cross_loans=("cross_loans", "mean"), violations=("violations", "mean"),
                                          runtime_s=("runtime_s", "mean")).round(3)
display(agg)

# MILP ต้องจัดได้ไม่น้อยกว่า FCFS-B เสมอ (เพราะ FCFS-B เป็นคำตอบที่ฟีสิเบิลของปัญหาเดียวกัน)
piv = df_exp.pivot_table(index=["N", "seed"], columns="method", values="assigned")
print("MILP >= FCFS-B ทุกชุด:", bool((piv["MILP"] >= piv["FCFS-B (ตรวจกฎครบ)"]).all()))
print("MILP ฝ่าฝืนกฎรวม:", int(df_exp.loc[df_exp.method == "MILP", "violations"].sum()),
      "| FCFS-B:", int(df_exp.loc[df_exp.method == "FCFS-B (ตรวจกฎครบ)", "violations"].sum()),
      "| FCFS-A:", int(df_exp.loc[df_exp.method == "FCFS-A (ไม่ตรวจกฎ)", "violations"].sum()))
df_exp.to_csv(OUTPUT_DIR / "baseline_comparison_synthetic.csv", index=False, encoding="utf-8-sig")

# กราฟเปรียบเทียบอัตราจัดได้
fig, ax = plt.subplots(figsize=(7, 4))
for name in METHODS:
    s = agg.xs(name, level="method")["fulfillment"]; ax.plot(s.index, s.values, marker="o", label=name.split(" ")[0])
ax.set_xlabel("Requests per 14 days"); ax.set_ylabel("Mean fulfillment rate"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "baseline_fulfillment.png", dpi=300); plt.show()

# ---- ข้อมูลจริง: คำขอ pending จาก CSV (ถ้ารันเซลล์ csv_requests_unique แล้ว) ----
if "csv_requests_unique" in globals() and csv_requests_unique:
    real = pd.DataFrame([evaluate_method(nm, fn, resources, csv_requests_unique) for nm, fn in METHODS.items()]).round(3)
    print("\nข้อมูลจริง (pending จาก CSV เทียบกับงานคงที่):")
    display(real)
    real.to_csv(OUTPUT_DIR / "baseline_comparison_real.csv", index=False, encoding="utf-8-sig")


MILP >= FCFS-B ทุกชุด: True
MILP ฝ่าฝืนกฎรวม: 0 | FCFS-B: 0 | FCFS-A: 175

ข้อมูลจริง (pending จาก CSV เทียบกับงานคงที่):


                                 fulfillment  full_requests  cross_loans  violations  runtime_s
N  method                                                                                      
5  FCFS-A (ไม่ตรวจกฎ)                  1.000          1.000          0.6         0.1      0.000
   FCFS-B (ตรวจกฎครบ)                  1.000          1.000          0.5         0.0      0.336
   Heuristic-SDF (ทริปสั้นก่อน)        1.000          1.000          0.5         0.0      0.322
   MILP                                1.000          1.000          0.5         0.0      0.088
10 FCFS-A (ไม่ตรวจกฎ)                  1.000          1.000          1.4         0.9      0.000
   FCFS-B (ตรวจกฎครบ)                  0.971          0.970          1.8         0.0      0.675
   Heuristic-SDF (ทริปสั้นก่อน)        0.981          0.980          1.8         0.0      0.669
   MILP                                0.981          0.980          1.7         0.0      0.099
20 FCFS-A (ไม่ตรวจกฎ)                  0

                         method  fulfillment  assigned  requested  full_requests  cross_loans  violations  v_overlap  v_rest  v_daily  v_weekly  runtime_s
0                          MILP          0.9         9         10            0.9            8           0          0       0        0         0      0.246
1            FCFS-A (ไม่ตรวจกฎ)          1.0        10         10            1.0            4           7          0       7        0         0      0.010
2            FCFS-B (ตรวจกฎครบ)          0.9         9         10            0.9            8           0          0       0        0         0      1.502
3  Heuristic-SDF (ทริปสั้นก่อน)          0.9         9         10            0.9            8           0          0       0        0         0      1.296

# Scalability & Sensitivity Analysis (สำหรับวารสารวิชาการ)

การทดลองเพื่อพิสูจน์ว่าโมเดล MILP เป็น **Generalized Framework** ที่องค์กรหรือมหาวิทยาลัยอื่นสามารถนำไปปรับใช้ได้จริง โดยแบ่งเป็น 4 ชุดการทดลอง:
1. **Multi-University Scalability**: ขยายขนาดจาก 3 คณะ (3 คัน, 15 คำขอ) ไปจนถึงมหาวิทยาลัยขนาดใหญ่ 10 คณะ (15 คัน, 75 คำขอ)
2. **Fleet Sizing Analysis**: ทดสอบการเพิ่มจำนวนรถต่อคณะ (1, 2, 3 คัน/คณะ) เมื่อความต้องการเพิ่มขึ้น
3. **Sharing Policy Evaluation (Shared Pool vs. Siloed)**: วัดผลประโยชน์ของการแชร์รถข้ามคณะ เทียบกับนโยบายห้ามยืมรถข้ามคณะ
4. **HOS Sensitivity Analysis**: ปรับเปลี่ยนเพดานชั่วโมงทำงานต่อวัน (10–16 ชม.) และเวลาพักขั้นต่ำ (8–12 ชม.)

In [51]:
# ==================== Scalability & Sensitivity Analysis (สำหรับวารสารวิชาการ) ====================
# วัตถุประสงค์: พิสูจน์ว่าโมเดล MILP เป็น Generalized Framework ที่องค์กร/มหาวิทยาลัยอื่นนำไปปรับใช้ได้จริง
# โดยแบ่งการทดลองเป็น 4 ชุดหลัก:
#   Exp 1 (Scalability)      : ขยายจำนวนคณะ (3 -> 5 -> 10 คณะ) จำนวนรถ (3 -> 10 -> 20 คัน) และคำขอ (15 -> 45 -> 90 รายการ)
#   Exp 2 (Fleet Sizing)     : วิเคราะห์จำนวนรถต่อคณะ (1, 2, 3 คัน/คณะ) เมื่อความต้องการเพิ่มขึ้น
#   Exp 3 (HOS Sensitivity)  : ปรับเปลี่ยนกฎความปลอดภัย (ชั่วโมงทำงานสูงสุดต่อวัน 10/12/14/16 ชม., เวลาพักขั้นต่ำ 8/10/12 ชม.)
#   Exp 4 (Sharing Policy)   : เปรียบเทียบ "ห้ามยืมรถข้ามคณะ (Siloed / Isolated)" vs "ยืมรถข้ามคณะได้ (Shared Pool - โมเดลของเรา)"

import time

def make_general_resources(faculties_counts, default_capacity=11):
    """สร้าง resources แบบทั่วไป รองรับ N คณะ และคณะละ M คัน โดยไม่จำกัดแค่ ICT/SCI/SEEN"""
    res = make_test_resources(counts=faculties_counts, default_capacity=default_capacity)
    for r in res.values():
        for k in ("test_daily_hours", "test_weekly_hours", "test_rest_hours"):
            r.pop(k, None)
    return res

def make_general_priority(faculties):
    """สร้างตารางลำดับความสำคัญแบบทั่วไปสำหรับชุดคณะใดๆ (รถตัวเอง = 1, รถคณะอื่น = 2)"""
    return {q: {o: (1 if q == o else 2) for o in faculties} for q in faculties}

def make_general_batch(faculties, n_requests, seed, horizon_days=14, max_veh=2):
    """สุ่มคำขอจองสำหรับชุดคณะใดๆ"""
    rng = np.random.default_rng(seed)
    reqs = {}
    for k in range(n_requests):
        fac = str(rng.choice(faculties))
        veh = int(rng.choice([1, 2], p=[0.9, 0.1])) if max_veh >= 2 else 1
        pax = int(rng.integers(1, 11 * veh + 1)) if veh > 1 else int(rng.integers(1, 12))
        start = TEST_DATE + pd.Timedelta(hours=int(rng.integers(0, horizon_days * 24)))
        dur = int(rng.choice(EXP_DURATIONS, p=EXP_DUR_P))
        reqs[f"R{k+1}"] = make_test_request(f"R{k+1}", faculty=fac, start=start, duration_hours=dur,
                                           passengers=pax, vehicles=veh,
                                           submitted_at=TEST_DATE - pd.Timedelta(minutes=int(rng.integers(0, 60 * 24 * 14))))
    return reqs

# -------------------------------------------------------------------------
# Experiment 1: Multi-University Scalability Test (ขยายขนาดองค์กรและจำนวนคำขอ)
# -------------------------------------------------------------------------
SCALABILITY_CONFIGS = [
    {"label": "Small (3 Fac, 3 Vans, 15 Req)",   "n_fac": 3,  "vans_per_fac": 1, "n_req": 15},
    {"label": "Medium (5 Fac, 5 Vans, 30 Req)",  "n_fac": 5,  "vans_per_fac": 1, "n_req": 30},
    {"label": "Large (5 Fac, 10 Vans, 50 Req)",  "n_fac": 5,  "vans_per_fac": 2, "n_req": 50},
    {"label": "X-Large (10 Fac, 15 Vans, 75 Req)", "n_fac": 10, "vans_per_fac": 1, "n_req": 75, "extra_vans": 5},
]
SCALE_SEEDS = range(5)

scale_rows = []
for cfg in SCALABILITY_CONFIGS:
    facs = [f"FAC_{i+1:02d}" for i in range(cfg["n_fac"])]
    counts = {f: cfg["vans_per_fac"] for f in facs}
    for i in range(cfg.get("extra_vans", 0)):
        counts[facs[i]] += 1
    prio = make_general_priority(facs)
    total_vans = sum(counts.values())
    for seed in SCALE_SEEDS:
        res = make_general_resources(counts)
        reqs = make_general_batch(facs, cfg["n_req"], seed)
        t0 = time.perf_counter()
        out, _, status = solve_assignment(res, reqs, capacity_by_resource={i: 11 for i in res}, faculty_priority=prio)
        dt = time.perf_counter() - t0
        req_v = sum(r["requested_count"] for r in out.values())
        got_v = sum(r["assigned_count"] for r in out.values())
        own_v = sum(1 for j, r in out.items() for i in r["assigned_resources"] if res[i]["faculty"] == reqs[j]["source_faculty"])
        scale_rows.append({
            "Scale": cfg["label"], "Faculties": cfg["n_fac"], "Total_Vans": total_vans,
            "Requests": cfg["n_req"], "seed": seed, "Status": status,
            "Fulfillment": got_v / req_v if req_v else 1.0,
            "Own_Faculty_Rate": own_v / got_v if got_v else 0.0,
            "Cross_Loans": got_v - own_v, "Runtime_s": dt
        })

df_scale = pd.DataFrame(scale_rows)
agg_scale = df_scale.groupby(["Scale", "Faculties", "Total_Vans", "Requests"], sort=False).agg(
    Fulfillment=("Fulfillment", "mean"), Own_Faculty_Rate=("Own_Faculty_Rate", "mean"),
    Cross_Loans=("Cross_Loans", "mean"), Runtime_mean_s=("Runtime_s", "mean"), Runtime_max_s=("Runtime_s", "max")
).round(3)
print("📊 [Exp 1] ผลการทดสอบ Scalability (ขยายจำนวนคณะ จำนวนรถ และคำขอ):")
display(agg_scale)
df_scale.to_csv(OUTPUT_DIR / "exp1_scalability.csv", index=False, encoding="utf-8-sig")

# -------------------------------------------------------------------------
# Experiment 2 & 4: Fleet Sizing (1, 2, 3 คัน/คณะ) & Sharing Policy (Siloed vs Shared Pool)
# -------------------------------------------------------------------------
def solve_siloed(resources, requests, prio):
    """Siloed Policy: ห้ามยืมรถข้ามคณะ (แยกแก้ปัญหาเฉพาะรถและคำขอของแต่ละคณะ)"""
    out_all = {}
    for fac in {r["faculty"] for r in resources.values()}:
        sub_res = {i: r for i, r in resources.items() if r["faculty"] == fac}
        sub_req = {j: q for j, q in requests.items() if q["source_faculty"] == fac}
        if sub_req:
            o, _, _ = solve_assignment(sub_res, sub_req, capacity_by_resource={i: 11 for i in sub_res}, faculty_priority=prio)
            out_all.update(o)
    return out_all

fleet_rows = []
for vans_per_fac in [1, 2, 3]:
    counts = {f: vans_per_fac for f in SC_FACULTIES}
    prio = make_general_priority(SC_FACULTIES)
    for n_req in [15, 30, 45]:
        for seed in SCALE_SEEDS:
            reqs = make_random_batch(n_req, seed)
            # 1) Shared Pool (MILP มาตรฐาน)
            res_shared = make_general_resources(counts)
            out_sh, _, _ = solve_assignment(res_shared, reqs, capacity_by_resource={i: 11 for i in res_shared}, faculty_priority=prio)
            req_v = sum(r["requested_count"] for r in out_sh.values())
            got_sh = sum(r["assigned_count"] for r in out_sh.values())
            own_sh = sum(1 for j, r in out_sh.items() for i in r["assigned_resources"] if res_shared[i]["faculty"] == reqs[j]["source_faculty"])
            # 2) Siloed (ห้ามยืมข้ามคณะ)
            res_silo = make_general_resources(counts)
            out_si = solve_siloed(res_silo, reqs, prio)
            got_si = sum(r["assigned_count"] for r in out_si.values())
            fleet_rows.append({
                "Vans_per_Faculty": vans_per_fac, "Total_Vans": vans_per_fac * 3, "Requests": n_req, "seed": seed,
                "Fulfillment_Shared": got_sh / req_v, "Fulfillment_Siloed": got_si / req_v,
                "Sharing_Gain_pct": (got_sh - got_si) / req_v * 100.0,
                "Cross_Loans": got_sh - own_sh
            })

df_fleet = pd.DataFrame(fleet_rows)
agg_fleet = df_fleet.groupby(["Vans_per_Faculty", "Total_Vans", "Requests"]).agg(
    Fulfillment_Siloed=("Fulfillment_Siloed", "mean"),
    Fulfillment_Shared=("Fulfillment_Shared", "mean"),
    Sharing_Gain_pct=("Sharing_Gain_pct", "mean"),
    Cross_Loans=("Cross_Loans", "mean")
).round(3)
print("\n📊 [Exp 2 & 4] ผลการวิเคราะห์ขนาดกองรถ (Fleet Sizing) และประโยชน์ของการแชร์รถข้ามคณะ (Shared vs Siloed):")
display(agg_fleet)
df_fleet.to_csv(OUTPUT_DIR / "exp2_fleet_sizing_and_sharing.csv", index=False, encoding="utf-8-sig")

# -------------------------------------------------------------------------
# Experiment 3: HOS Parameter Sensitivity Analysis (ปรับเพดานชั่วโมงทำงานและเวลาพัก)
# -------------------------------------------------------------------------
HOS_DAILY_LIST = [10.0, 12.0, 14.0, 16.0]
HOS_REST_LIST = [8.0, 10.0, 12.0]
hos_rows = []
for max_d in HOS_DAILY_LIST:
    for min_r in HOS_REST_LIST:
        for seed in SCALE_SEEDS:
            res = make_general_resources({"ICT": 1, "SCI": 1, "SEEN": 1})
            reqs = make_random_batch(25, seed)
            out, _, _ = solve_assignment(res, reqs, capacity_by_resource={i: 11 for i in res},
                                         max_daily_hours=max_d, min_rest_hours=min_r)
            req_v = sum(r["requested_count"] for r in out.values())
            got_v = sum(r["assigned_count"] for r in out.values())
            hos_rows.append({
                "Max_Daily_Hours": max_d, "Min_Rest_Hours": min_r, "seed": seed,
                "Fulfillment": got_v / req_v, "Assigned": got_v, "Requested": req_v
            })

df_hos = pd.DataFrame(hos_rows)
piv_hos = df_hos.pivot_table(index="Max_Daily_Hours", columns="Min_Rest_Hours", values="Fulfillment", aggfunc="mean").round(3)
print("\n📊 [Exp 3] Sensitivity Analysis — อัตราการจัดสรรสำเร็จ (Fulfillment Rate) เมื่อเปลี่ยนเพดานชั่วโมงทำงานต่อวัน และเวลาพักขั้นต่ำ:")
display(piv_hos)
df_hos.to_csv(OUTPUT_DIR / "exp3_hos_sensitivity.csv", index=False, encoding="utf-8-sig")

# สร้างกราฟรวมสำหรับใส่ในเปเปอร์ (Figure: Fleet Sizing & Sharing Gain + HOS Sensitivity)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for vpf in [1, 2, 3]:
    sub = agg_fleet.xs(vpf, level="Vans_per_Faculty").droplevel(0)
    axes[0].plot(sub.index, sub["Fulfillment_Shared"], marker="o", label=f"{vpf*3} vans (Shared)")
    axes[0].plot(sub.index, sub["Fulfillment_Siloed"], marker="x", linestyle="--", alpha=0.7, label=f"{vpf*3} vans (Siloed)")
axes[0].set_xlabel("Requests per 14 days"); axes[0].set_ylabel("Mean fulfillment rate")
axes[0].set_title("(a) Fleet Sizing & Cross-Faculty Sharing"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for min_r in HOS_REST_LIST:
    axes[1].plot(piv_hos.index, piv_hos[min_r], marker="s", label=f"Min rest = {int(min_r)}h")
axes[1].set_xlabel("Max daily working hours"); axes[1].set_ylabel("Mean fulfillment rate (N=25)")
axes[1].set_title("(b) Sensitivity to HOS Regulations"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(OUTPUT_DIR / "sensitivity_and_scalability.png", dpi=300); plt.show()


📊 [Exp 1] ผลการทดสอบ Scalability (ขยายจำนวนคณะ จำนวนรถ และคำขอ):

📊 [Exp 2 & 4] ผลการวิเคราะห์ขนาดกองรถ (Fleet Sizing) และประโยชน์ของการแชร์รถข้ามคณะ (Shared vs Siloed):

📊 [Exp 3] Sensitivity Analysis — อัตราการจัดสรรสำเร็จ (Fulfillment Rate) เมื่อเปลี่ยนเพดานชั่วโมงทำงานต่อวัน และเวลาพักขั้นต่ำ:


                                                                 Fulfillment  Own_Faculty_Rate  Cross_Loans  Runtime_mean_s  Runtime_max_s
Scale                             Faculties Total_Vans Requests                                                                           
Small (3 Fac, 3 Vans, 15 Req)     3         3          15              0.965             0.743          4.2           0.131          0.149
Medium (5 Fac, 5 Vans, 30 Req)    5         5          30              0.976             0.725          9.0           0.235          0.243
Large (5 Fac, 10 Vans, 50 Req)    5         10         50              0.985             0.894          5.8           0.651          0.732
X-Large (10 Fac, 15 Vans, 75 Req) 10        15         75              0.981             0.791         17.2           1.973          2.074

                                      Fulfillment_Siloed  Fulfillment_Shared  Sharing_Gain_pct  Cross_Loans
Vans_per_Faculty Total_Vans Requests                                                                       
1                3          15                     0.770               0.976            20.650          3.4
                            30                     0.605               0.747            14.125          5.2
                            45                     0.478               0.607            12.907          7.2
2                6          15                     0.965               1.000             3.538          0.6
                            30                     0.854               0.965            11.093          3.8
                            45                     0.751               0.889            13.813          7.0
3                9          15                     1.000               1.000             0.000          0.0
                            

Min_Rest_Hours    8.0    10.0   12.0
Max_Daily_Hours                     
10.0             0.700  0.693  0.693
12.0             0.735  0.728  0.728
14.0             0.850  0.829  0.822
16.0             0.850  0.829  0.822

# การคำนวณสัดส่วนความน่าจะเป็นจริงจากข้อมูล (Empirical Probability Estimation)

วิเคราะห์สัดส่วนทางสถิติจริงจากชุดข้อมูลการจอง (ooking.csv) และบันทึกการเดินทางของคนขับ (drivers.csv) เพื่อนำมาแทนที่ค่าน้ำหนักสมมติ (Synthetic Weights):
- **Trip Scope**: การเดินทางนอกจังหวัดมีสัดส่วน 51.4% และในจังหวัด 48.6%
- **Passenger Count**: 100% มีผู้โดยสาร $\le 11$ คน (เฉลี่ย 5.14 คน)
- **Requested Vehicles**: 100% ขอรถ 1 คัน
- **Trip Duration**: ทริป 1 วันคิดเป็น 89.1%, ทริป 2 วันคิดเป็น 6.1%, และทริป 3+ วันคิดเป็น 4.9%
- **Fleet Free Rate**: รถ ICT มีโอกาสว่าง 33.8% (ชั่วโมงงานหนาแน่น) ขณะที่รถ SCI มีโอกาสว่าง 88.3%

In [52]:
# ==================== Empirical Probability Estimation (จากข้อมูลจริงใน raw_data: ICT + SCI จริง) ====================
# คำนวณสัดส่วนความน่าจะเป็นเชิงประจักษ์จากประวัติการจองจริง (ict_booking.csv + sci_bookingN.csv)
# และบันทึกการใช้รถจริงของคนขับ (ict_drivers.csv + sci_driversN.csv)

import pandas as pd
import numpy as np
from pathlib import Path

# ใช้ข้อมูลที่ผ่านการทำความสะอาดแล้วของทั้ง ICT และ SCI (ข้อมูลจริง)
df_b_real = df_bookings_clean[df_bookings_clean["source_faculty_code"].isin(["ICT", "SCI"])].copy()
df_d_real = df_drivers_clean[df_drivers_clean["source_faculty_code"].isin(["ICT", "SCI"])].copy()

print(f"จำนวนข้อมูลจริงที่ใช้คำนวณ Empirical Weights: คำขอจองสะอาด {len(df_b_real):,} รายการ | บันทึกคนขับสะอาด {len(df_d_real):,} รายการ")

# 1. สัดส่วนคณะผู้ขอ (Requester Faculty) จากคำขอจองจริง
emp_fac = df_b_real["source_faculty_code"].value_counts(normalize=True).to_dict()

# 2. สัดส่วนขอบเขตการเดินทาง (Trip Scope)
scope_mapping = {"ภายในจังหวัด": "IN_PROVINCE", "ภายนอกจังหวัด": "OUT_PROVINCE"}
df_b_real["trip_scope_std"] = df_b_real["trip_scope_raw"].map(scope_mapping)
emp_scope = df_b_real["trip_scope_std"].value_counts(normalize=True).to_dict()

# 3. สัดส่วนจำนวนผู้โดยสาร (Passenger Range)
emp_pax = {"<=11": float((df_b_real["passenger_count"] <= 11).mean()), ">11": float((df_b_real["passenger_count"] > 11).mean())}

# 4. สัดส่วนจำนวนรถที่ขอ (Requested Vehicle Count)
emp_veh = {"1": 1.0, ">1": 0.0}

# 5. สัดส่วนระยะเวลาทริป (Day Range: 1 วัน, 2 วัน, 3+ วัน)
dur_days = df_b_real["planned_duration_hours"] / 24.0
day_cat = dur_days.apply(lambda d: 1 if d <= 1.0 else (2 if d <= 2.0 else 3))
emp_days = day_cat.value_counts(normalize=True).to_dict()

# 6. อัตราความว่างของรถ (Fleet Free Rate) จากบันทึกการเดินทางจริงของคนขับแต่ละคณะ
emp_van_free = {}
emp_busy_hours = {}
emp_span_hours = {}
for fac in ["ICT", "SCI"]:
    sub = df_d_real[df_d_real["source_faculty_code"] == fac]
    span_h = (sub["actual_end_dt"].max() - sub["actual_start_dt"].min()).total_seconds() / 3600
    busy_h = sub["actual_duration_hours"].clip(upper=24*14).sum()
    emp_busy_hours[fac] = busy_h
    emp_span_hours[fac] = span_h
    emp_van_free[fac] = float(max(0.0, 1.0 - (busy_h / span_h)))
emp_van_free["SEEN"] = float(np.mean([emp_van_free["ICT"], emp_van_free["SCI"]]))

comparison_df = pd.DataFrame([
    {"Feature": "Requester Faculty", "Category": "ICT / SCI (Empirical Share)", "Synthetic_Weight": "0.50 / 0.30 / 0.20", "Empirical_Weight": f"ICT: {emp_fac.get('ICT', 0):.3f}, SCI: {emp_fac.get('SCI', 0):.3f}"},
    {"Feature": "Trip Scope", "Category": "In-Province / Out-Province", "Synthetic_Weight": "0.70 / 0.30", "Empirical_Weight": f"{emp_scope.get('IN_PROVINCE', 0):.3f} / {emp_scope.get('OUT_PROVINCE', 0):.3f}"},
    {"Feature": "Passenger Range", "Category": "<=11 pax / >11 pax", "Synthetic_Weight": "0.80 / 0.20", "Empirical_Weight": f"{emp_pax['<=11']:.3f} / {emp_pax['>11']:.3f}"},
    {"Feature": "Vehicle Count", "Category": "1 van / >1 van", "Synthetic_Weight": "0.90 / 0.10", "Empirical_Weight": f"{emp_veh['1']:.3f} / {emp_veh['>1']:.3f}"},
    {"Feature": "Trip Duration", "Category": "1 day / 2 days / 3+ days", "Synthetic_Weight": "0.60 / 0.30 / 0.10", "Empirical_Weight": f"{emp_days.get(1, 0):.3f} / {emp_days.get(2, 0):.3f} / {emp_days.get(3, 0):.3f}"},
    {"Feature": "Fleet Free Rate", "Category": "Van Available Probability", "Synthetic_Weight": "0.700", "Empirical_Weight": f"ICT: {emp_van_free['ICT']:.3f}, SCI: {emp_van_free['SCI']:.3f}"},
])

print("📊 ตารางเปรียบเทียบสัดส่วนความน่าจะเป็น: ค่าสมมติฐานเดิม (Synthetic) vs ค่าจริงจากข้อมูล (Empirical ICT + SCI):")
display(comparison_df)
comparison_df.to_csv(OUTPUT_DIR / "empirical_probability_comparison.csv", index=False, encoding="utf-8-sig")

# คำนวณความน่าจะเป็นถ่วงน้ำหนักใหม่สำหรับ Exhaustive Scenarios (864 รูปแบบ)
# ใช้สัดส่วนจริงของ ICT และ SCI (ปรับเกลี่ยเข้ากับ SEEN 0.10)
w_ict = emp_fac.get("ICT", 0.5) * 0.90
w_sci = emp_fac.get("SCI", 0.5) * 0.90
P_FAC_EMP = {"ICT": w_ict, "SCI": w_sci, "SEEN": 0.10}
P_SCOPE_EMP = {"IN_PROVINCE": emp_scope.get("IN_PROVINCE", 0.5), "OUT_PROVINCE": emp_scope.get("OUT_PROVINCE", 0.5)}
P_PAX_EMP = {"<=11": 0.98, ">11": 0.02}
P_VEH_EMP = {"1": 0.95, ">1": 0.05}
P_URG_EMP = {False: 0.90, True: 0.10}
P_DAY_EMP = {1: emp_days.get(1, 0.85), 2: emp_days.get(2, 0.08), 3: emp_days.get(3, 0.07)}
P_VAN_FREE_EMP = {"T01": emp_van_free["ICT"], "T02": emp_van_free["SCI"], "T03": emp_van_free["SEEN"]}

def sc_emp_probability(fac, scope, pax_k, veh_k, urg, days, free_vans):
    p = P_FAC_EMP[fac] * P_SCOPE_EMP[scope] * P_PAX_EMP[pax_k] * P_VEH_EMP[veh_k] * P_URG_EMP[urg] * P_DAY_EMP[days]
    for vid in ["T01", "T02", "T03"]:
        p_free = P_VAN_FREE_EMP[vid]
        p *= p_free if vid in free_vans else (1.0 - p_free)
    return p

if "df_scen" in globals():
    df_scen_emp = df_scen.copy()
    df_scen_emp["Empirical_Probability"] = [
        sc_emp_probability(r.Requester, r.Scope, r.Pax, str(r.Vehicles), r.Urgent, r.Days,
                           set(r.Free_vans.split(",")) if r.Free_vans != "-" else set())
        for r in df_scen_emp.itertuples()
    ]
    df_scen_emp["Empirical_Probability"] /= df_scen_emp["Empirical_Probability"].sum()
    
    summary_emp = df_scen_emp.groupby("Outcome").agg(
        Scenarios=("Outcome", "size"),
        Synthetic_Prob=("Probability", "sum"),
        Empirical_Prob=("Empirical_Probability", "sum")
    ).round(4)
    print("\n📊 ผลการเปรียบเทียบอัตราความสำเร็จของสถานการณ์ (Synthetic Weights vs Empirical Weights จริง):")
    display(summary_emp)
    summary_emp.to_csv(OUTPUT_DIR / "scenario_empirical_vs_synthetic.csv", encoding="utf-8-sig")


จำนวนข้อมูลจริงที่ใช้คำนวณ Empirical Weights: คำขอจองสะอาด 383 รายการ | บันทึกคนขับสะอาด 2,169 รายการ
📊 ตารางเปรียบเทียบสัดส่วนความน่าจะเป็น: ค่าสมมติฐานเดิม (Synthetic) vs ค่าจริงจากข้อมูล (Empirical ICT + SCI):

📊 ผลการเปรียบเทียบอัตราความสำเร็จของสถานการณ์ (Synthetic Weights vs Empirical Weights จริง):


             Feature                     Category    Synthetic_Weight        Empirical_Weight
0  Requester Faculty  ICT / SCI (Empirical Share)  0.50 / 0.30 / 0.20  ICT: 0.642, SCI: 0.358
1         Trip Scope   In-Province / Out-Province         0.70 / 0.30           0.543 / 0.457
2    Passenger Range           <=11 pax / >11 pax         0.80 / 0.20           1.000 / 0.000
3      Vehicle Count               1 van / >1 van         0.90 / 0.10           1.000 / 0.000
4      Trip Duration     1 day / 2 days / 3+ days  0.60 / 0.30 / 0.10   0.867 / 0.047 / 0.086
5    Fleet Free Rate    Van Available Probability               0.700  ICT: 0.691, SCI: 0.758

               Scenarios  Synthetic_Prob  Empirical_Prob
Outcome                                                 
จัดได้ครบ            540           0.950          0.9709
จัดได้บางส่วน        216           0.023          0.0084
จัดไม่ได้            108           0.027          0.0207